In [1]:
# Escondendo Warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# EDA – Base das Irregularidades (1 célula por irregularidade) – otimizado para não crashar

Este notebook foi ajustado para **não estourar memória** e para gerar **todas as irregularidades** de forma padronizada:

- **Cálculo das irregularidades**: carrega e processa **somente as colunas necessárias** (base mínima).
- **Saída (DF + Parquet + PDF)**: para cada irregularidade, materializa um **dataset completo** com **todas as colunas do parquet original** + **a coluna da irregularidade** (e a coluna `dia_operacional` para análise).
- **Unidade de contagem**: sempre **pessoa-dia** (evita multiplicidade por batidas repetidas no mesmo dia).
- **Relatórios**: layout padronizado com capa, metodologia, indicadores e rankings para triagem rápida.

Estrutura de saída:
- `outputs_irregularidades/dfs`: **parquets** completos por irregularidade.
- `outputs_irregularidades/pdfs`: **relatórios PDF** por irregularidade.

Observações importantes:
- Não acusar **borda de contexto** (dia incompleto no início/fim da janela).
- Para irregularidades com portaria, considerar **sequência irregular** (E/S repetida) além de ausência de E ou S.
- Para `ponto sem portaria`, só acusar quando houver **terceiro elegível** no entorno do ponto presencial.

---


### Solução (visão geral)

1. Ler base mínima (parquet) com colunas críticas.
2. Normalizar data/hora e construir `dia_operacional`.
3. Construir tabela `day` (features por cadastro + dia).
4. Aplicar regras por irregularidade (1 célula por regra).
5. Materializar datasets completos e gerar PDF/Parquet em `outputs_irregularidades`.

### Irregularidades Monitoradas

1. Irregularidade - 999 como Trajeto
2. Irregularidade - Ponto sem Portaria
3. Irregularidade - Registro sem Portaria
4. Irregularidade - Executivos com Histórico de 999 como Trajeto
5. Irregularidade - Ausência de Descanso entre Jornadas de Trabalho
6. Irregularidade - Pessoas Trabalhando Além da Jornada de Trabalho
7. Irregularidade - Pessoas Trabalhando sem Ganhar
8. Irregularidade - Registros de Pessoas Desligadas
9. Irregularidade - Pessoas sem cumprir os 3 Dias Presenciais em Semana de Feriado Prolongado
10. Irregularidade - Não entrou no refeitório, mas passou no grill
11. Irregularidade - Ausência das horas Presenciais Mínimas

---


#### Objetivo geral (triagem e priorização)

- Identificar **padrões consistentes** de irregularidade por pessoa-dia.
- Facilitar a **priorização por recorrência** e **concentração de risco**.
- Garantir **rastreabilidade** do dado até o registro bruto.
- Gerar relatórios **claros e objetivos** para auditoria e ação.

---


## 1) Bibliotecas e diretórios de saída


In [ ]:
# Importando bibliotecas
from __future__ import annotations

from pathlib import Path
import re
import unicodedata
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


In [ ]:
# Diretórios de saída
OUT_ROOT = Path("./outputs_irregularidades")
PDF_DIR = OUT_ROOT / "pdfs"
DFS_DIR = OUT_ROOT / "dfs"

PDF_DIR.mkdir(parents=True, exist_ok=True)
DFS_DIR.mkdir(parents=True, exist_ok=True)

# Entrada (parquet da Ronda)
PARQUET_PATH = Path(r".\df_ronda.parquet")  # ajuste se necessário

# Se o seu notebook ainda estiver pesado, troque para False
MANTER_DF_EM_MEMORIA = False

print("OUT_ROOT:", OUT_ROOT)
print("PDF_DIR :", PDF_DIR)
print("DFS_DIR :", DFS_DIR)
print("PARQUET :", PARQUET_PATH)


## 2) Parâmetros, regex e semanas de feriado prolongado


In [ ]:
# Parâmetros (mantendo a lógica)
janela_portaria_min = 120
tolerancia_entrada_min = 30
tolerancia_saida_min = 30
janela_proxy_minutos = 3
janela_999_horas = 6

hora_corte_operacional = 11

descanso_min_horas = 11
jornada_min_horas = 10
jornada_min_minutos = 50

usar_jornada_por_portaria = True
max_jornada_portaria_horas = 20

# Regex
regex_portaria = (
    r"\bPORTARIA\b|"
    r"\bPORT\.\b|"
    r"\bPORT(?=[\.\-/ ])|"
    r"\bCANCELA\b|"
    r"\bGUARITA\b|"
    r"\bPEDESTRES?\b|"
    r"\bVISITANTE\b|"
    r"\bFUNCIONARIO\b|"
    r"\bVEICULOS?\b"
)
regex_refeitorio = r"^(?=.*REF)(?!.*GRILL).*$"
regex_grill = r"(?i)\bGRILL\b"
regex_ponto = r"\bMARCA\b|\bPONTO\b|\bPONT\b"  # Ponto = MARCA / PONTO / PONT

# Semanas "feriado prolongado"
semanas_feriado_prolongado = [
    (pd.Timestamp("2025-04-28"), pd.Timestamp("2025-05-04")),
    (pd.Timestamp("2025-06-16"), pd.Timestamp("2025-06-22")),
    (pd.Timestamp("2025-11-17"), pd.Timestamp("2025-11-23")),
    (pd.Timestamp("2025-12-22"), pd.Timestamp("2025-12-28")),
]

# Timedeltas
t_jan_port = pd.Timedelta(minutes=janela_portaria_min)
t_tol_ent = pd.Timedelta(minutes=tolerancia_entrada_min)
t_tol_sai = pd.Timedelta(minutes=tolerancia_saida_min)
t_jan_999 = pd.Timedelta(hours=janela_999_horas)

t_desc_min = pd.Timedelta(hours=descanso_min_horas)
t_jornada_min = pd.Timedelta(hours=jornada_min_horas, minutes=jornada_min_minutos)

t_gap_work = pd.Timedelta(hours=2)
t_max_jornada_port = pd.Timedelta(hours=max_jornada_portaria_horas)



## 3) Leitura mínima (somente colunas necessárias para calcular as irregularidades)


In [ ]:
# Colunas mínimas para cálculo (não é para relatório nem export final)
MIN_COLS = [
    "cadastro",
    "nome",
    "data",
    "hora",
    "dir",
    "planta",
    "descricao coletor",
    "cargo",
    "área",
    "heranca",
    "desligado",
    "Data_Dem",
]

# Validação rápida
# (Se faltar alguma dessas, o parquet não atende a lógica atual)
try:
    try:
        # usa backend Arrow para reduzir pico de memória em colunas texto
        df_min_raw = pd.read_parquet(PARQUET_PATH, columns=MIN_COLS, dtype_backend="pyarrow")
    except TypeError:
        df_min_raw = pd.read_parquet(PARQUET_PATH, columns=MIN_COLS)
except Exception as e:
    raise RuntimeError(
        "Falha ao ler parquet com colunas mínimas. "
        "Verifique se o arquivo existe e se o engine de parquet está instalado (pyarrow/fastparquet)."
    ) from e

missing = [c for c in MIN_COLS if c not in df_min_raw.columns]
if missing:
    raise ValueError(f"Parquet não tem colunas mínimas necessárias: {missing}")

# Remove duplicatas
df_min_raw = df_min_raw.drop_duplicates().reset_index(drop=True)

# ID posicional para materialização com iloc no parquet completo
df_min_raw["_row_id"] = np.arange(len(df_min_raw), dtype=np.int64)

print("Linhas (parquet):", len(df_min_raw))
print("Colunas carregadas (mínimo):", len(df_min_raw.columns))
df_min_raw.head(3)


## 4) Pré-processamento mínimo (flags e dia operacional)


In [ ]:
# Copiando base
df = df_min_raw.copy()

# cadastro como string
df["cadastro"] = df["cadastro"].astype(str).str.strip()

# descrição
desc = df["descricao coletor"].fillna("").astype(str)

df["Portaria"] = desc.str.contains(regex_portaria, case=False, na=False, regex=True)
df["Ponto"] = desc.str.contains(regex_ponto, case=False, na=False, regex=True)

# Regra solicitada:
# - se tiver "GRILL" na descrição => GRILL
# - se tiver "REF" e NÃO tiver "GRILL" => REFEITÓRIO normal
desc_upper = desc.str.upper()
df["_GRILL"] = desc_upper.str.contains("GRILL", regex=False, na=False)
df["_REFEITORIO"] = desc_upper.str.contains("REF", regex=False, na=False) & (~df["_GRILL"])

# planta == 999 (robusto)
planta_num = pd.to_numeric(df["planta"], errors="coerce")
df["999"] = planta_num.eq(999)

# Presencial: planta válida e não 999
df["Presencial"] = planta_num.notna() & (~df["999"])

# Executivo: cargo == 'E'
cargo_norm = df["cargo"].fillna("").astype(str).str.strip().str.upper()
df["_is_executivo"] = cargo_norm.eq("E")

# dir normalizado
dir_norm = df["dir"].fillna("").astype(str).str.strip().str.upper()
dir_norm = dir_norm.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
dir_norm = dir_norm.where(~dir_norm.str.startswith("E"), "E")
dir_norm = dir_norm.where(~dir_norm.str.startswith("S"), "S")
df["_dir_norm"] = dir_norm

# dt_ronda robusto (data + hora)
data_dt = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
hora_s = df["hora"].fillna("00:00").astype(str).str.strip()
hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
hora_td = pd.to_timedelta(hora_s, errors="coerce")

df["_dia_key"] = data_dt.dt.normalize()
df["_dt_ronda"] = df["_dia_key"] + hora_td

# Dia operacional base (corte às 11h)
df["_dia_operacional"] = (df["_dt_ronda"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

# Mantém df sem ordenar globalmente para não “embaralhar” o _row_id,
# mas cria uma visão ordenada quando necessário.
print("OK: pré-processamento mínimo aplicado.")
df.head(3)


## 5) Ajuste de dia operacional por jornada Portaria (E->S) – mantém a lógica


In [ ]:
# Ajuste de dia operacional por jornada (Portaria E->S) - versão otimizada
if usar_jornada_por_portaria:
    # Proteção para ambiente com memória limitada.
    if len(df) > 12_000_000:
        print("WARN: ajuste de jornada por portaria pulado por volume muito alto para memória local.")
    else:
        try:
            cols_min = ["cadastro", "_dt_ronda", "Portaria", "_dir_norm", "999", "_dia_operacional"]
            df_min = df[cols_min]

            # Ordena sem reset_index(drop=False) para reduzir pico de memória.
            df_sorted = df_min.sort_values(["cadastro", "_dt_ronda"], kind="mergesort")

            orig_idx = df_sorted.index.to_numpy(dtype=np.int64, copy=False)
            times    = df_sorted["_dt_ronda"].to_numpy()  # datetime64[ns]
            port     = df_sorted["Portaria"].fillna(False).to_numpy(dtype=bool)
            dirn     = df_sorted["_dir_norm"].to_numpy()
            is999    = df_sorted["999"].fillna(False).to_numpy(dtype=bool)

            # array alvo (escrita rápida)
            dia_op = df_sorted["_dia_operacional"].to_numpy(copy=True)

            # garante que é gravável
            if not dia_op.flags.writeable:
                dia_op = dia_op.copy()
                dia_op.setflags(write=True)

            # converte janelas para numpy timedelta64[ns]
            tmax = np.timedelta64(int(pd.Timedelta(t_max_jornada_port).value), "ns")
            tjan = np.timedelta64(int(pd.Timedelta(t_jan_999).value), "ns")

            # iterar grupos por posições (mais rápido que fatiar DF)
            grp_pos = df_sorted.groupby("cadastro", sort=False).indices

            for cad, pos in grp_pos.items():
                t = times[pos]
                valid = ~np.isnat(t)
                if not valid.any():
                    continue

                posv  = pos[valid]
                tv    = t[valid]
                portv = port[posv]
                dirv  = dirn[posv]
                is9v  = is999[posv]

                m_ent = portv & (dirv == "E")
                m_sai = portv & (dirv == "S")

                ent = tv[m_ent]
                sai = tv[m_sai]

                if ent.size == 0 or sai.size == 0:
                    continue

                i = j = 0
                while i < ent.size and j < sai.size:
                    dt_ent = ent[i]
                    dt_sai = sai[j]

                    if dt_sai < dt_ent:
                        j += 1
                        continue

                    if (dt_sai - dt_ent) > tmax:
                        i += 1
                        continue

                    # dia operacional = dia da entrada
                    dia = dt_ent.astype("datetime64[D]").astype("datetime64[ns]")

                    # intervalo core [ent, sai]
                    l = np.searchsorted(tv, dt_ent, side="left")
                    r = np.searchsorted(tv, dt_sai, side="right")
                    if r > l:
                        dia_op[posv[l:r]] = dia

                    # 999 antes (ent - janela, ent)
                    l2 = np.searchsorted(tv, dt_ent - tjan, side="left")
                    r2 = np.searchsorted(tv, dt_ent, side="left")
                    if r2 > l2:
                        sl = slice(l2, r2)
                        m9 = is9v[sl]
                        if m9.any():
                            dia_op[posv[sl][m9]] = dia

                    # 999 depois (sai, sai + janela]
                    l3 = np.searchsorted(tv, dt_sai, side="right")
                    r3 = np.searchsorted(tv, dt_sai + tjan, side="right")
                    if r3 > l3:
                        sl = slice(l3, r3)
                        m9 = is9v[sl]
                        if m9.any():
                            dia_op[posv[sl][m9]] = dia

                    i += 1
                    j += 1

            # grava de volta no df original pelo índice original
            df.loc[orig_idx, "_dia_operacional"] = dia_op

            print("OK: _dia_operacional ajustado por jornada (Portaria E->S) [otimizado].")

        except MemoryError:
            print("WARN: memória insuficiente para ajuste de jornada por portaria; mantendo _dia_operacional atual.")

# limpa temporários pesados desta etapa
for _v in ("df_min", "df_sorted", "orig_idx", "times", "port", "dirn", "is999", "dia_op", "grp_pos"):
    if _v in globals():
        del globals()[_v]
gc.collect()



## 6) Tabela day (features por cadastro + dia_operacional) – mínima


In [ ]:
# =========================
# Considera apenas registros com dt válido (sem cópia profunda)
# =========================

# (Se _dt_ronda já for datetime64, ótimo. Se for object, converta antes uma vez:)
# df["_dt_ronda"] = pd.to_datetime(df["_dt_ronda"], errors="coerce")

mask_dt = df["_dt_ronda"].notna()

COLS_KEEP = [
    "cadastro", "_dia_operacional", "_dt_ronda",
    "Presencial", "Ponto", "Portaria",
    "_REFEITORIO", "_GRILL", "_is_executivo",
    "_dir_norm",
    "999",
]

# SEM .copy() aqui (era isso que explodia a RAM)
# Fallback: em bases muito grandes, o filtro booleano pode estourar memória.
if bool(mask_dt.all()):
    df_valid = df.loc[:, COLS_KEEP]
else:
    try:
        df_valid = df.loc[mask_dt, COLS_KEEP]
    except MemoryError:
        print(
            "WARN: fallback de memória em df_valid; usando base sem filtro por _dt_ronda "
            "(linhas com dt nulo serão naturalmente descartadas pelos próximos filtros)."
        )
        df_valid = df.loc[:, COLS_KEEP]

# Min/Max por cadastro (anti-truncamento)
cad_minmax = (
    df_valid.groupby("cadastro", sort=False)["_dt_ronda"]
      .agg(min_dt_cad="min", max_dt_cad="max")
      .reset_index()
)

# day_any (qualquer presencial)
pres_any = df_valid.loc[df_valid["Presencial"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
day_any = (
    pres_any.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
            .agg(
                first_any_pres=("_dt_ronda", "min"),
                last_any_pres=("_dt_ronda", "max"),
                n_any_pres=("_dt_ronda", "size"),
            )
)
day_any["dur_any_pres"] = day_any["last_any_pres"] - day_any["first_any_pres"]

# day_ponto (ponto presencial)
pres_ponto = df_valid.loc[
    df_valid["Ponto"].eq(True) & df_valid["Presencial"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda"]
]
day_ponto = (
    pres_ponto.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
              .agg(first_ponto_pres=("_dt_ronda", "min"), last_ponto_pres=("_dt_ronda", "max"))
)

# presença de ponto (qualquer) no dia
day_has_ponto_any = (
    df_valid.loc[df_valid["Ponto"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PONTO_ANY=True)
)

# presença de portaria no dia (qualquer)
day_has_port = (
    df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PORTARIA_ANY=True)
)

# refeitório/grill no dia
day_has_ref = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_ANY=True)
)

day_has_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_GRILL_ANY=True)
)

# executivo no dia
day_is_exec = (
    df_valid.loc[df_valid["_is_executivo"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(IS_EXECUTIVO_DAY=True)
)

# Monta day base
day = day_any.merge(day_ponto, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ponto_any, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_port, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ref, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_grill, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_is_exec, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(cad_minmax, on="cadastro", how="left")

for c in ["HAS_PONTO_ANY", "HAS_PORTARIA_ANY", "HAS_REFEITORIO_ANY", "HAS_GRILL_ANY", "IS_EXECUTIVO_DAY"]:
    day[c] = day[c].fillna(False).astype(bool)

# SEM_COBERTURA_CONTEXTO
day["w_start_needed"] = day["first_any_pres"] - t_jan_port
day["w_end_needed"] = day["last_any_pres"] + t_jan_port
day["SEM_COBERTURA_CONTEXTO"] = False
mask_mm = day["min_dt_cad"].notna() & day["max_dt_cad"].notna()
day.loc[mask_mm, "SEM_COBERTURA_CONTEXTO"] = (day["w_start_needed"] < day["min_dt_cad"]) | (day["w_end_needed"] > day["max_dt_cad"])

# ok_entry / ok_exit (portaria coerente)
port = df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]].copy(deep=False)
if port.empty:
    day["ok_entry"] = False
    day["ok_exit"] = False
else:
    port = port.merge(
        day[["cadastro", "_dia_operacional", "w_start_needed", "w_end_needed", "first_any_pres", "last_any_pres"]],
        on=["cadastro", "_dia_operacional"],
        how="inner",
    )

    in_window = (port["_dt_ronda"] >= port["w_start_needed"]) & (port["_dt_ronda"] <= port["w_end_needed"])

    port["ok_entry"] = (
        in_window
        & port["_dir_norm"].eq("E")
        & (port["_dt_ronda"] <= (port["first_any_pres"] + t_tol_ent))
    )

    port["ok_exit"] = (
        in_window
        & port["_dir_norm"].eq("S")
        & (port["_dt_ronda"] >= (port["last_any_pres"] - t_tol_sai))
    )

    ok = port.groupby(["cadastro", "_dia_operacional"], sort=False)[["ok_entry", "ok_exit"]].any().reset_index()
    day = day.merge(ok, on=["cadastro", "_dia_operacional"], how="left")
    day[["ok_entry", "ok_exit"]] = day[["ok_entry", "ok_exit"]].fillna(False)

# SEM_PORTARIA_COERENTE
day["SEM_PORTARIA_COERENTE"] = np.where(
    day["IS_EXECUTIVO_DAY"],
    ~day["HAS_PORTARIA_ANY"],
    ~(day["ok_entry"] | day["ok_exit"]),
)

# SEM_COBERTURA_CONTEXTO desliga a regra
day.loc[day["SEM_COBERTURA_CONTEXTO"], "SEM_PORTARIA_COERENTE"] = False

# =========================
# Regra grill/refeitório — REFEITÓRIO NORMAL NO MESMO DIA
# =========================
day["HAS_REFEITORIO_NORMAL_NO_DIA"] = day["HAS_REFEITORIO_ANY"].fillna(False).astype(bool)

# Compatibilidade com blocos antigos que ainda leem a flag antiga.
day["HAS_REFEITORIO_BEFORE_GRILL"] = day["HAS_REFEITORIO_NORMAL_NO_DIA"]

day["Irregularidade - Não entrou no refeitório, mas passou no grill"] = (
    day["HAS_GRILL_ANY"].fillna(False).astype(bool)
    & (~day["HAS_REFEITORIO_NORMAL_NO_DIA"])
)

print("OK: day construído.")
day.head(3)


## 7) Helpers: slugify, relatório PDF e export por row_id


In [ ]:
# Limpa df pesado para liberar memória (se necessário)
BATCH_SIZE = 250_000  # usado em loops de chunk/leitura/escrita nesta célula

def _ensure_numpy_datetime_series(s: pd.Series, name: str | None = None) -> pd.Series:
    s2 = pd.to_datetime(s, errors="coerce")
    return pd.Series(
        s2.to_numpy(dtype="datetime64[ns]"),
        index=s2.index,
        name=name or getattr(s, "name", None),
    )

def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text.lower()

IRREGULARIDADES_SEM_PLANTA_999 = {
    "irregularidade_ausencia_das_horas_presenciais_minimas",
    "irregularidade_pessoas_trabalhando_sem_ganhar",
    "irregularidade_ponto_sem_portaria",
    "irregularidade_registro_sem_portaria",
}

def _deve_excluir_planta_999(
    irregularidade_col: str | None = None,
    titulo: str | None = None,
    pdf_path: Path | None = None,
) -> bool:
    candidatos = []

    if irregularidade_col:
        candidatos.append(slugify(irregularidade_col))

    if titulo:
        candidatos.append(slugify(titulo))

    if pdf_path is not None:
        try:
            candidatos.append(slugify(Path(pdf_path).stem))
        except Exception:
            pass
        candidatos.append(slugify(str(pdf_path)))

    return any(
        alvo in cand
        for cand in candidatos
        for alvo in IRREGULARIDADES_SEM_PLANTA_999
    )

def _mask_planta_not_999(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    s_num = pd.to_numeric(s, errors="coerce")
    mask_999 = s_num.eq(999) | s.str.match(r"^999(\D|$)", na=False)
    return ~mask_999

def _mask_planta_eq_1(series: pd.Series) -> pd.Series:
    s = series.fillna("").astype(str).str.strip()
    s_num = pd.to_numeric(s.str.replace(",", ".", regex=False), errors="coerce")
    mask_num = s_num.eq(1)
    mask_txt = s.str.match(r"^0*1(?:[.,]0+)?(?:\D.*)?$", na=False)
    return mask_num | mask_txt

MAPA_PLANTA_NOME = {
    1: "Planta Anchieta",
    8: "Planta Curitiba",
    10: "Planta São Carlos",
    13: "Planta Taubaté",
    17: "Planta Resende",
    20: "Planta Jabaquara",
    27: "Regional Recife",
    30: "Planta Vinhedo",
    40: "Planta Academia VW",
    59: "Curitiba VW FIS",
    888: "Mobilidade - VWCO",
    999: "Marcações de Captura de Ponto",
}

def _fmt_planta_nome(v) -> str:
    if v is None:
        return ""
    s = str(v).strip()
    if s.lower() in {"", "nan", "nat", "none"}:
        return ""

    n = pd.to_numeric(s.replace(",", "."), errors="coerce")
    if pd.notna(n):
        n_int = int(round(float(n)))
        if abs(float(n) - float(n_int)) < 1e-9 and (n_int in MAPA_PLANTA_NOME):
            return MAPA_PLANTA_NOME[n_int]

    m = re.search(r"\b(\d{1,3})\b", s)
    if m:
        n_int = int(m.group(1))
        if n_int in MAPA_PLANTA_NOME:
            return MAPA_PLANTA_NOME[n_int]

    return s

def _filtrar_df_sem_planta_999(df_: pd.DataFrame | None) -> pd.DataFrame | None:
    if not isinstance(df_, pd.DataFrame):
        return df_
    if df_.empty:
        return df_
    if "planta" not in df_.columns:
        return df_
    return df_.loc[_mask_planta_not_999(df_["planta"])].copy()

def _filtrar_row_ids_sem_planta_999(row_ids: np.ndarray) -> np.ndarray:
    if row_ids is None:
        return np.array([], dtype=np.int64)

    row_ids = np.asarray(row_ids, dtype=np.int64)
    if row_ids.size == 0:
        return row_ids

    if "planta" not in df_min_raw.columns:
        return row_ids

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]
    if row_ids.size == 0:
        return row_ids

    cache_name = "_PLANTA_NUM_CACHE"
    cache = globals().get(cache_name)

    if not isinstance(cache, np.ndarray) or len(cache) != len(df_min_raw):
        cache = pd.to_numeric(df_min_raw["planta"], errors="coerce").to_numpy()
        globals()[cache_name] = cache

    plantas = cache[row_ids]
    keep = ~pd.Series(plantas).eq(999).to_numpy()
    return row_ids[keep]

def _pick_col(df_: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_.columns:
            return c
    return None


def _render_capa_padrao_pdf(
    pdf,
    titulo: str,
    gen_date: str,
    metricas: list[tuple[str, str]],
    periodo: str | None = None,
    concentracoes: list[tuple[str, str]] | None = None,
    top3_piores: list[str] | None = None,
    top3_atuais: list[str] | None = None,
    janela_info: str | None = None,
    extras: list[str] | None = None,
) -> None:
    fig = plt.figure(figsize=(11.69, 8.27))
    ax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
    ax.axis("off")

    bottom_margin = 0.035
    truncated = {"value": False}

    def _clean_text(txt: str, max_chars: int | None = None) -> str:
        s = "" if txt is None else str(txt)
        s = " ".join(s.replace("\r", " ").replace("\n", " ").split())
        if max_chars and len(s) > max_chars:
            s = s[: max_chars - 3].rstrip() + "..."
        return s

    def _wrap_lines(txt: str, width: int, max_lines: int | None = None) -> list[str]:
        txt = "" if txt is None else str(txt)
        out: list[str] = []
        for blk in txt.replace("\r", "").split("\n"):
            blk = blk.strip()
            if blk == "":
                out.append("")
                continue
            words = blk.split()
            line = ""
            for w in words:
                candidate = (line + " " + w).strip()
                if len(candidate) <= width:
                    line = candidate
                else:
                    if line:
                        out.append(line)
                    line = w
            if line:
                out.append(line)

        if not out:
            out = [""]

        if max_lines is not None and len(out) > max_lines:
            out = out[:max_lines]
            if not out[-1].endswith("..."):
                out[-1] = _clean_text(out[-1], max_chars=max(10, width - 2))
        return out

    def _draw_wrapped(
        x: float,
        y: float,
        text: str,
        fontsize: float,
        width: int,
        step: float,
        fontweight: str | None = None,
        prefix_first: str | None = None,
        max_lines: int | None = None,
    ) -> float:
        lines = _wrap_lines(text, width=width, max_lines=max_lines)
        for k, ln in enumerate(lines):
            if y < bottom_margin:
                if not truncated["value"]:
                    ax.text(
                        0.02,
                        bottom_margin,
                        "(Resumo da capa truncado para manter legibilidade nesta página)",
                        fontsize=8.8,
                        va="bottom",
                        color="#555",
                    )
                    truncated["value"] = True
                return y
            ln_txt = f"{prefix_first}{ln}" if (k == 0 and prefix_first) else ln
            ax.text(x, y, ln_txt, fontsize=fontsize, fontweight=fontweight, va="top")
            y -= step
        return y

    y = 0.965
    y = _draw_wrapped(0.02, y, _clean_text(titulo), fontsize=16.8, width=92, step=0.040, fontweight="bold", max_lines=3)
    y -= 0.003
    y = _draw_wrapped(0.02, y, f"Gerado em: {gen_date}", fontsize=11.0, width=100, step=0.030)

    y -= 0.006
    for rotulo, valor in metricas:
        y = _draw_wrapped(0.02, y, f"{rotulo}: {valor}", fontsize=11.6, width=104, step=0.031, prefix_first="- ", max_lines=2)

    if periodo:
        y = _draw_wrapped(0.02, y, f"Período: {periodo}", fontsize=10.9, width=104, step=0.030, prefix_first="- ", max_lines=2)

    if concentracoes:
        for rotulo, valor in concentracoes:
            y = _draw_wrapped(0.02, y, f"{rotulo}: {valor}", fontsize=10.7, width=104, step=0.029, prefix_first="- ", max_lines=2)

    y = _draw_wrapped(
        0.02,
        y,
        "Top 3 piores (quantidade e % do total):",
        fontsize=11.2,
        width=102,
        step=0.030,
        fontweight="bold",
        prefix_first="- ",
        max_lines=2,
    )

    if top3_piores:
        for i, linha in enumerate(top3_piores[:3], start=1):
            linha_limpa = _clean_text(linha, max_chars=220)
            y = _draw_wrapped(0.035, y, f"{i}. {linha_limpa}", fontsize=9.4, width=120, step=0.024, max_lines=3)
    else:
        y = _draw_wrapped(0.035, y, "Sem dados suficientes.", fontsize=9.7, width=120, step=0.026)

    janela_txt = janela_info if janela_info else "base indisponível"
    y = _draw_wrapped(
        0.02,
        y,
        f"Top 3 atuais (últimos 30 dias) - base da janela: {janela_txt}",
        fontsize=10.9,
        width=102,
        step=0.030,
        fontweight="bold",
        prefix_first="- ",
        max_lines=2,
    )

    if top3_atuais:
        for i, linha in enumerate(top3_atuais[:3], start=1):
            linha_limpa = _clean_text(linha, max_chars=220)
            y = _draw_wrapped(0.035, y, f"{i}. {linha_limpa}", fontsize=9.2, width=120, step=0.024, max_lines=3)
    else:
        y = _draw_wrapped(0.035, y, "Sem dados suficientes na janela.", fontsize=9.6, width=120, step=0.026)

    if extras:
        for extra in extras:
            y = _draw_wrapped(0.02, y, _clean_text(str(extra), max_chars=220), fontsize=9.2, width=106, step=0.024, prefix_first="- ", max_lines=2)

    pdf.savefig(fig)
    plt.close(fig)


def gerar_relatorio_pdf_semanal_feriado(df_semanal: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    import datetime as _dt
    import numpy as np
    from matplotlib.backends.backend_pdf import PdfPages
    from matplotlib.colors import LinearSegmentedColormap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    gen_date = _dt.date.today().strftime("%Y-%m-%d")

    if df_semanal is None or df_semanal.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Pessoa-semanas irregulares", "0"),
                    ("Pessoas únicas", "0"),
                    ("Semanas com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem linhas para gerar o relatório semanal."],
            )
        return

    dfw = df_semanal.copy()

    # -----------------------------
    # Normalização mínima
    # -----------------------------
    if "_semana_feriado_ini" in dfw.columns and "semana_inicio" not in dfw.columns:
        dfw = dfw.rename(columns={"_semana_feriado_ini": "semana_inicio"})

    if "área" in dfw.columns and "area" not in dfw.columns:
        dfw = dfw.rename(columns={"área": "area"})

    if "semana_inicio" not in dfw.columns:
        raise ValueError("df_semanal precisa ter a coluna 'semana_inicio'.")

    dfw["semana_inicio"] = pd.to_datetime(dfw["semana_inicio"], errors="coerce").dt.normalize()
    dfw = dfw[dfw["semana_inicio"].notna()].copy()

    if "semana_fim" not in dfw.columns:
        dfw["semana_fim"] = dfw["semana_inicio"] + pd.Timedelta(days=6)
    else:
        dfw["semana_fim"] = pd.to_datetime(dfw["semana_fim"], errors="coerce").dt.normalize()

    if "nome" not in dfw.columns:
        dfw["nome"] = "Sem Nome"
    dfw["nome"] = dfw["nome"].fillna("Sem Nome").astype(str).str.strip()

    if "area" not in dfw.columns:
        dfw["area"] = "Sem Área"
    dfw["area"] = dfw["area"].fillna("Sem Área").astype(str).str.strip()

    if "dias_com_registro" not in dfw.columns:
        dfw["dias_com_registro"] = np.nan

    if "dias_presenciais" not in dfw.columns:
        dfw["dias_presenciais"] = 0
    if "horas_presenciais" not in dfw.columns:
        dfw["horas_presenciais"] = 0.0

    dfw["dias_presenciais"] = pd.to_numeric(dfw["dias_presenciais"], errors="coerce").fillna(0).astype(int)
    dfw["horas_presenciais"] = pd.to_numeric(dfw["horas_presenciais"], errors="coerce").fillna(0.0)

    if "cadastro" in dfw.columns:
        dfw["cadastro"] = dfw["cadastro"].astype(str).str.strip()
    else:
        raise ValueError("df_semanal precisa ter a coluna 'cadastro'.")

    if "nao_cumpriu_3" not in dfw.columns:
        dfw["nao_cumpriu_3"] = (dfw["dias_presenciais"] < 3)
    else:
        dfw["nao_cumpriu_3"] = dfw["nao_cumpriu_3"].fillna(False).astype(bool)

    # -----------------------------
    # Base analítica (pessoa-semana)
    # -----------------------------
    pessoa_semanas_total = int(len(dfw))
    pessoas_unicas = int(dfw["cadastro"].nunique())
    areas_unicas = int(dfw["area"].nunique())
    semanas_unicas = int(dfw["semana_inicio"].nunique())

    dt_min = dfw["semana_inicio"].min()
    dt_max = dfw["semana_fim"].max()

    area_agg = (
        dfw.groupby("area", as_index=False)
           .agg(
               pessoa_semanas=("cadastro", "size"),
               pessoas_unicas=("cadastro", "nunique"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
    )
    area_agg["score_risco"] = area_agg["pessoa_semanas"] * (3 - area_agg["media_dias_presenciais"].clip(lower=0, upper=3))

    if "total_colaboradores_area" in dfw.columns:
        total_area = (
            dfw.groupby("area", as_index=False)["total_colaboradores_area"]
               .max()
        )
        area_agg = area_agg.merge(total_area, on="area", how="left")
    else:
        area_agg["total_colaboradores_area"] = np.nan

    area_agg["total_colaboradores_area"] = (
        pd.to_numeric(area_agg["total_colaboradores_area"], errors="coerce")
          .fillna(area_agg["pessoas_unicas"])
          .clip(lower=1)
    )
    area_agg["pct_colaboradores_irregulares"] = (
        (area_agg["pessoas_unicas"] / area_agg["total_colaboradores_area"]) * 100.0
    ).clip(lower=0.0, upper=100.0)
    area_agg["score_risco_pct"] = area_agg["pct_colaboradores_irregulares"] * (
        3 - area_agg["media_dias_presenciais"].clip(lower=0, upper=3)
    )

    semana_agg = (
        dfw.groupby("semana_inicio", as_index=False)
           .agg(
               pessoa_semanas=("cadastro", "size"),
               pessoas_unicas=("cadastro", "nunique"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
           .sort_values("semana_inicio")
    )

    pessoa_stats = (
        dfw.groupby(["cadastro", "nome", "area"], as_index=False)
           .agg(
               semanas_irregulares=("semana_inicio", "nunique"),
               dias_presenciais_total=("dias_presenciais", "sum"),
               horas_presenciais_total=("horas_presenciais", "sum"),
               media_dias_presenciais=("dias_presenciais", "mean"),
               media_horas_presenciais=("horas_presenciais", "mean"),
           )
    )

    def make_first_name_labels(series_names: pd.Series) -> pd.Series:
        first = (
            series_names.astype(str)
            .str.strip()
            .str.split()
            .str[0]
            .replace({"": "Semnome"})
        )
        counts = {}
        labels = []
        for fn in first.tolist():
            counts[fn] = counts.get(fn, 0) + 1
            labels.append(fn if counts[fn] == 1 else f"{fn} ({counts[fn]})")
        return pd.Series(labels, index=series_names.index)

    # -----------------------------
    # Estilo
    # -----------------------------
    VW_NAVY   = "#001E50"
    VW_BLUE   = "#0B5FA5"
    VW_BLUE2  = "#7FB0D6"
    VW_GREY   = "#C9D1D9"
    VW_DARK   = "#1B2A41"
    ACCENT    = "#A8D8F0"
    BG_LIGHT  = "#F6F8FB"

    plt.rcParams.update({
        "figure.facecolor": BG_LIGHT,
        "axes.facecolor": "white",
        "axes.edgecolor": VW_GREY,
        "axes.labelcolor": VW_DARK,
        "xtick.color": VW_DARK,
        "ytick.color": VW_DARK,
        "text.color": VW_DARK,
        "font.size": 10,
        # Evita laranja automático do ciclo padrão.
        "axes.prop_cycle": plt.cycler(color=[VW_BLUE, ACCENT, VW_NAVY, VW_BLUE2]),
    })

    VW_CMAP = LinearSegmentedColormap.from_list(
        "vw_blues",
        [BG_LIGHT, "#CFE2F3", VW_BLUE, VW_NAVY]
    )

    def style_card(ax):
        for s in ax.spines.values():
            s.set_visible(True)
            s.set_color(VW_GREY)
            s.set_linewidth(1.2)
        ax.set_facecolor("white")

    def add_desc(ax, text):
        ax.text(
            0.01, -0.34, text,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=8.8, color="#334155",
            wrap=True,
            bbox=dict(
                boxstyle="round,pad=0.40",
                facecolor=BG_LIGHT,
                edgecolor=VW_GREY,
                linewidth=0.9
            )
        )

    DESC = {
        "A": "Combina volume de pessoa-semana com severidade média (menos dias presenciais = maior severidade).",
        "B": "Prioriza as áreas com maior percentual de colaboradores irregulares, em vez de volume absoluto.",
        "C": "Ajuda a ver se o problema foi pontual ou se repetiu entre as semanas/áreas analisadas.",
        "D": "Compara o volume de semanas irregulares com a média de dias presenciais da semana.",
        "E": "Mostra quão perto do limite mínimo de 3 dias estão os casos. Quanto mais perto de 0 dia, mais severo.",
        "F": "Evidencia quem reaparece em mais de uma semana irregular.",
        "G": "Compara o percentual de colaboradores irregulares por área (pessoas únicas / base total da área).",
        "H": "Destaca os recorrentes com menor média de dias presenciais nas semanas irregulares.",
    }

    with PdfPages(pdf_path) as pdf:
        # =========================================
        # CAPA
        # =========================================
        cad_nome_map = (
            pessoa_stats.drop_duplicates(subset=["cadastro"], keep="first")
                       .set_index("cadastro")["nome"]
            if (not pessoa_stats.empty) and ("cadastro" in pessoa_stats.columns)
            else pd.Series(dtype=str)
        )
        labels_pessoa = dfw["cadastro"].astype(str).map(
            lambda c: f"{c} - {cad_nome_map.get(c, '')}".strip(" -")
        )
        top_pessoas_semana = labels_pessoa.value_counts()
        top1_n = int(top_pessoas_semana.head(1).sum()) if not top_pessoas_semana.empty else 0
        top3_n = int(top_pessoas_semana.head(3).sum()) if not top_pessoas_semana.empty else 0
        top10_n = int(top_pessoas_semana.head(10).sum()) if not top_pessoas_semana.empty else 0

        top1_pct = (top1_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0
        top3_pct = (top3_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0
        top10_pct = (top10_n / pessoa_semanas_total) if pessoa_semanas_total else 0.0

        top3_piores_capa = []
        for k, v in top_pessoas_semana.head(3).items():
            pct = (int(v) / pessoa_semanas_total) if pessoa_semanas_total else 0.0
            top3_piores_capa.append(f"{str(k)} — {int(v)} ({pct:.1%})")

        dt_ref = pd.to_datetime(dfw["semana_inicio"], errors="coerce").dropna()
        dt_ref_max = dt_ref.max().normalize() if not dt_ref.empty else pd.NaT
        if pd.notna(dt_ref_max):
            dt_cut = dt_ref_max - pd.Timedelta(days=29)
            recent = dfw.loc[pd.to_datetime(dfw["semana_inicio"], errors="coerce") >= dt_cut]
        else:
            recent = dfw.head(0)

        total_recent = int(len(recent))
        if total_recent > 0:
            labels_recent = recent["cadastro"].astype(str).map(
                lambda c: f"{c} - {cad_nome_map.get(c, '')}".strip(" -")
            )
            top_recent = labels_recent.value_counts().head(3)
        else:
            top_recent = pd.Series(dtype=int)

        top3_atuais_capa = []
        for k, v in top_recent.items():
            pct = (int(v) / total_recent) if total_recent else 0.0
            top3_atuais_capa.append(f"{str(k)} — {int(v)} ({pct:.1%} da janela)")

        janela_info = (
            f"{total_recent:,} ({(total_recent / pessoa_semanas_total):.1%} do total)".replace(",", ".")
            if pessoa_semanas_total > 0 else "0 (0.0% do total)"
        )

        metricas_capa = [
            ("Pessoa-semanas irregulares", f"{pessoa_semanas_total:,}".replace(",", ".")),
            ("Pessoas únicas", f"{pessoas_unicas:,}".replace(",", ".")),
            ("Semanas com ocorrência", f"{semanas_unicas:,}".replace(",", ".")),
        ]

        periodo_capa = None
        if pd.notna(dt_min) and pd.notna(dt_max):
            periodo_capa = f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"

        concentracoes_capa = [
            ("Concentração Top 1", f"{top1_n:,} ({top1_pct:.1%})".replace(",", ".")),
            ("Concentração Top 3", f"{top3_n:,} ({top3_pct:.1%})".replace(",", ".")),
            ("Concentração Top 10", f"{top10_n:,} ({top10_pct:.1%})".replace(",", ".")),
        ]

        extras_capa = [
            f"Áreas com ocorrência: {areas_unicas:,}".replace(",", "."),
            "Critério: menos de 3 dias presenciais na semana de feriado prolongado",
        ]

        _render_capa_padrao_pdf(
            pdf=pdf,
            titulo=titulo,
            gen_date=gen_date,
            metricas=metricas_capa,
            periodo=periodo_capa,
            concentracoes=concentracoes_capa,
            top3_piores=top3_piores_capa,
            top3_atuais=top3_atuais_capa,
            janela_info=janela_info,
            extras=extras_capa,
        )

        # =========================================
        # DASHBOARD 4x2
        # =========================================
        fig = plt.figure(figsize=(18, 24))
        gs = fig.add_gridspec(4, 2, hspace=0.92, wspace=0.28)

        # (A) Prioridade — Volume x Severidade por área
        axA = fig.add_subplot(gs[0, 0]); style_card(axA)

        dfA = area_agg.copy()
        topA = dfA.sort_values(["score_risco_pct", "score_risco"], ascending=[False, False]).head(8).copy()

        axA.scatter(
            dfA["pessoa_semanas"],
            dfA["media_dias_presenciais"],
            s=(dfA["pessoas_unicas"] * 20).clip(50, 1000),
            alpha=0.65, color=VW_BLUE2, edgecolors=VW_NAVY, linewidths=0.7
        )
        axA.scatter(
            topA["pessoa_semanas"],
            topA["media_dias_presenciais"],
            s=(topA["pessoas_unicas"] * 24).clip(70, 1100),
            alpha=0.95, color=ACCENT, edgecolors=VW_NAVY, linewidths=0.9
        )

        for _, r in topA.iterrows():
            axA.annotate(
                str(r["area"]),
                xy=(r["pessoa_semanas"], r["media_dias_presenciais"]),
                xytext=(12, 10),
                textcoords="offset points",
                ha="left", va="bottom", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=VW_GREY, linewidth=0.6)
            )

        axA.set_title("A) Prioridade — Volume × severidade por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axA.set_xlabel("Volume (pessoa-semanas)")
        axA.set_ylabel("Média de dias presenciais")
        axA.grid(True, linestyle="--", alpha=0.18)
        axA.legend(handles=[
            Line2D([0], [0], marker='o', color='w', label='Áreas (geral)',
                   markerfacecolor=VW_BLUE2, markeredgecolor=VW_NAVY, markersize=9),
            Line2D([0], [0], marker='o', color='w', label='Top 8 prioridade',
                   markerfacecolor=ACCENT, markeredgecolor=VW_NAVY, markersize=9),
        ], loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)
        add_desc(axA, DESC["A"])

        # (B) Pareto — percentual de colaboradores irregulares por área
        axB = fig.add_subplot(gs[0, 1]); style_card(axB)
        dfB = area_agg.sort_values(["pct_colaboradores_irregulares", "pessoas_unicas"], ascending=[False, False]).head(12).copy()

        axB.bar(dfB["area"].astype(str), dfB["pct_colaboradores_irregulares"], color=VW_NAVY, alpha=0.92)
        axB.set_title("B) Pareto — % de colaboradores irregulares por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axB.set_xlabel("Área")
        axB.set_ylabel("% de colaboradores da área")
        axB.grid(axis="y", linestyle="--", alpha=0.18)
        axB.tick_params(axis="x", rotation=35)

        max_bar = float(dfB["pct_colaboradores_irregulares"].max()) if not dfB.empty else 1.0
        axB.set_ylim(0, max_bar * 1.30 if max_bar > 0 else 1)
        for i, r in enumerate(dfB.itertuples(index=False)):
            axB.text(
                i,
                r.pct_colaboradores_irregulares + max_bar * 0.03,
                f"{float(r.pct_colaboradores_irregulares):.1f}%\n({int(r.pessoas_unicas)}/{int(r.total_colaboradores_area)} colab.)",
                ha="center", va="bottom", fontsize=8
            )
        add_desc(axB, DESC["B"])

        # (C) Heatmap — área x semana
        axC = fig.add_subplot(gs[1, 0]); style_card(axC)
        top_areas_heat = area_agg.sort_values(["pct_colaboradores_irregulares", "pessoas_unicas"], ascending=[False, False]).head(10)["area"].astype(str).tolist()

        aw = (
            dfw[dfw["area"].astype(str).isin(top_areas_heat)]
            .groupby(["area", "semana_inicio"], as_index=False)
            .size()
            .rename(columns={"size": "qtd"})
        )

        if aw.empty:
            axC.axis("off")
            axC.text(0.02, 0.95, "C) Estabilidade — Área × semana", fontsize=14, fontweight="bold", va="top")
            axC.text(0.02, 0.82, "Sem dados suficientes.", fontsize=11, va="top")
        else:
            pivot = aw.pivot_table(index="area", columns="semana_inicio", values="qtd", aggfunc="sum").fillna(0)
            pivot = pivot.reindex(top_areas_heat)
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)
            mat = pivot.values

            im = axC.imshow(mat, aspect="auto", cmap=VW_CMAP)
            axC.set_title("C) Estabilidade — Pessoa-semanas por Área × semana", pad=10, color=VW_NAVY, fontweight="bold")
            axC.set_xlabel("Semana (início)")
            axC.set_ylabel("Área")

            xlabels = [pd.to_datetime(c).strftime("%Y-%m-%d") for c in pivot.columns]
            axC.set_xticks(np.arange(len(xlabels)))
            axC.set_xticklabels(xlabels, rotation=45, ha="right")
            axC.set_yticks(np.arange(pivot.shape[0]))
            axC.set_yticklabels(pivot.index.astype(str))

            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    v = mat[i, j]
                    axC.text(j, i, f"{int(v)}", ha="center", va="center",
                             fontsize=8, color=("white" if v >= np.nanmax(mat) * 0.60 and np.nanmax(mat) > 0 else VW_DARK))

            cbar = fig.colorbar(im, ax=axC, fraction=0.046, pad=0.04)
            cbar.set_label("Qtd pessoa-semana")

        add_desc(axC, DESC["C"])

        # (D) Tendência semanal
        axD = fig.add_subplot(gs[1, 1]); style_card(axD)
        dfD = semana_agg.copy()
        dfD["semana"] = dfD["semana_inicio"].dt.strftime("%Y-%m-%d")

        axD.plot(dfD["semana"], dfD["pessoa_semanas"], marker="o", color=VW_NAVY, linewidth=2.4, label="Pessoa-semanas")
        axD.set_title("D) Evolução semanal — Volume vs média de dias", pad=10, color=VW_NAVY, fontweight="bold")
        axD.set_xlabel("Semana (início)")
        axD.set_ylabel("Qtd pessoa-semana")
        axD.grid(True, linestyle="--", alpha=0.18)
        axD.tick_params(axis="x", rotation=45)

        axD2 = axD.twinx()
        axD2.plot(dfD["semana"], dfD["media_dias_presenciais"], marker="o", color=VW_BLUE, linewidth=2.2, label="Média dias")
        axD2.set_ylabel("Média dias presenciais")

        lines1, labels1 = axD.get_legend_handles_labels()
        lines2, labels2 = axD2.get_legend_handles_labels()
        axD.legend(lines1 + lines2, labels1 + labels2, loc="upper right", frameon=True, facecolor="white", edgecolor=VW_GREY)

        for x, y in zip(dfD["semana"], dfD["pessoa_semanas"]):
            axD.text(x, y + max(dfD["pessoa_semanas"].max() * 0.03, 0.2), f"{int(y)}", ha="center", fontsize=8)

        add_desc(axD, DESC["D"])

        # (E) Severidade — faixas de horas
        axE = fig.add_subplot(gs[2, 0]); style_card(axE)

        bins = [-0.001, 1.0, 2.0, 3.0001]
        labels = ["0 dia", "1 dia", "2 dias"]
        dfw["faixa_dias"] = pd.cut(dfw["dias_presenciais"], bins=bins, labels=labels, include_lowest=True, right=False)

        dist = (
            dfw.groupby("faixa_dias", dropna=False)
               .size()
               .rename("qtd")
               .reset_index()
        )
        dist["faixa_dias"] = dist["faixa_dias"].astype(str)
        dist = dist[dist["faixa_dias"].isin(labels)].copy()
        dist = dist.set_index("faixa_dias").reindex(labels).fillna(0).reset_index()
        dist["pct"] = dist["qtd"] / dist["qtd"].sum() * 100 if dist["qtd"].sum() > 0 else 0.0

        axE.bar(dist["faixa_dias"], dist["pct"], color=VW_BLUE, alpha=0.92)
        axE.set_title("E) Severidade — Distribuição por faixa de dias", pad=10, color=VW_NAVY, fontweight="bold")
        axE.set_xlabel("Dias presenciais na semana")
        axE.set_ylabel("% de pessoa-semanas")
        axE.grid(axis="y", linestyle="--", alpha=0.18)

        ymaxE = float(dist["pct"].max()) * 1.25 if not dist.empty else 1.0
        axE.set_ylim(0, ymaxE if ymaxE > 0 else 1)
        for i, r in dist.iterrows():
            axE.text(i, float(r["pct"]) + ymaxE * 0.02 if ymaxE > 0 else 0.1, f"{float(r['pct']):.1f}%", ha="center", fontsize=8)

        add_desc(axE, DESC["E"])

        # (F) Recorrência por pessoa
        axF = fig.add_subplot(gs[2, 1]); style_card(axF)

        topF = (
            pessoa_stats.sort_values(["semanas_irregulares", "horas_presenciais_total"], ascending=[False, True])
                       .head(12)
                       .copy()
        )
        topF["primeiro_nome"] = make_first_name_labels(topF["nome"])

        valsF = topF["semanas_irregulares"].to_numpy()
        axF.bar(topF["primeiro_nome"], valsF, color=VW_NAVY, alpha=0.92)
        axF.set_title("F) Investigação — Recorrência por pessoa", pad=10, color=VW_NAVY, fontweight="bold")
        axF.set_xlabel("Primeiro nome")
        axF.set_ylabel("Qtd de semanas irregulares")
        axF.grid(axis="y", linestyle="--", alpha=0.18)
        axF.tick_params(axis="x", rotation=45)
        for lab in axF.get_xticklabels():
            lab.set_ha("right")

        if len(valsF) > 0:
            axF.set_ylim(0, max(valsF.max() * 1.30, valsF.max() + 1))
            for i, r in enumerate(topF.itertuples(index=False)):
                axF.text(i, r.semanas_irregulares + 0.05, f"{str(r.area)}\n{float(r.media_dias_presenciais):.2f} dias",
                         ha="center", va="bottom", fontsize=8)

        add_desc(axF, DESC["F"])

        # (G) Influência da base — percentual por área
        axG = fig.add_subplot(gs[3, 0]); style_card(axG)

        dfG = area_agg.sort_values(["pct_colaboradores_irregulares", "pessoas_unicas"], ascending=[False, False]).head(12).copy()
        valsG = dfG["pct_colaboradores_irregulares"].to_numpy()

        axG.bar(dfG["area"].astype(str), valsG, color=VW_BLUE, alpha=0.92)
        axG.set_title("G) Influência da base — % de colaboradores por Área", pad=10, color=VW_NAVY, fontweight="bold")
        axG.set_xlabel("Área")
        axG.set_ylabel("% de colaboradores da área")
        axG.grid(axis="y", linestyle="--", alpha=0.18)
        axG.tick_params(axis="x", rotation=35)

        if len(valsG) > 0:
            axG.set_ylim(0, max(valsG.max() * 1.25, valsG.max() + 1))
            for i, r in enumerate(dfG.itertuples(index=False)):
                axG.text(i, r.pct_colaboradores_irregulares + 0.05,
                         f"{int(r.pessoas_unicas)}/{int(r.total_colaboradores_area)} colab.",
                         ha="center", va="bottom", fontsize=8)

        add_desc(axG, DESC["G"])

        # (H) Menor média de horas — recorrentes
        axH = fig.add_subplot(gs[3, 1]); style_card(axH)

        baseH = pessoa_stats[pessoa_stats["semanas_irregulares"] >= 2].copy()
        topH = (
            baseH.sort_values(["media_dias_presenciais", "semanas_irregulares"], ascending=[True, False])
                 .head(12)
                 .copy()
        )
        topH["primeiro_nome"] = make_first_name_labels(topH["nome"])

        valsH = topH["media_dias_presenciais"].to_numpy()
        axH.bar(topH["primeiro_nome"], valsH, color=VW_NAVY, alpha=0.85)
        axH.set_title("H) Recorrentes — Menor média de dias presenciais", pad=10, color=VW_NAVY, fontweight="bold")
        axH.set_xlabel("Primeiro nome")
        axH.set_ylabel("Média de dias presenciais")
        axH.grid(axis="y", linestyle="--", alpha=0.18)
        axH.tick_params(axis="x", rotation=45)
        for lab in axH.get_xticklabels():
            lab.set_ha("right")

        if len(valsH) > 0:
            axH.set_ylim(0, max(valsH.max() * 1.35, 1))
            for i, r in enumerate(topH.itertuples(index=False)):
                axH.text(i, r.media_dias_presenciais + 0.03, f"{int(r.semanas_irregulares)} sem.\n{str(r.area)}",
                         ha="center", va="bottom", fontsize=8)

        add_desc(axH, DESC["H"])

        fig.suptitle(
            "Painel semanal — Semana de Feriado Prolongado (pessoa-semana)",
            fontsize=20, fontweight="bold", color=VW_NAVY, y=0.99
        )

        pdf.savefig(fig)
        plt.close(fig)

        # =========================================
        # LISTAS FINAIS
        # =========================================
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        ax.axis("off")

        top_pessoas = (
            pessoa_stats.sort_values(["semanas_irregulares", "media_dias_presenciais"], ascending=[False, True])
                       .head(15)
                       .loc[:, ["cadastro", "nome", "area", "semanas_irregulares", "media_dias_presenciais"]]
                       .copy()
        )
        top_pessoas["media_dias_presenciais"] = top_pessoas["media_dias_presenciais"].round(2)

        ax.text(0.02, 0.97, "Top 15 pessoas — semanas irregulares", fontsize=15, fontweight="bold", va="top")

        if top_pessoas.empty:
            ax.text(0.02, 0.90, "Sem dados suficientes.", fontsize=11, va="top")
        else:
            tabela = ax.table(
                cellText=top_pessoas.values,
                colLabels=["Cadastro", "Nome", "Área", "Semanas", "Média dias"],
                cellLoc="left",
                colLoc="left",
                bbox=[0.02, 0.08, 0.96, 0.82]
            )
            tabela.auto_set_font_size(False)
            tabela.set_fontsize(9)
            tabela.scale(1, 1.15)

        pdf.savefig(fig)
        plt.close(fig)

def gerar_relatorio_pdf(df_irregularidade: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    if (
        isinstance(df_irregularidade, pd.DataFrame)
        and not df_irregularidade.empty
        and {"cadastro", "semana_inicio", "dias_presenciais", "horas_presenciais"}.issubset(df_irregularidade.columns)
    ):
        gerar_relatorio_pdf_semanal_feriado(df_irregularidade, titulo=titulo, pdf_path=pdf_path)
        return

    import datetime as _dt
    import numpy as np

    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 150,
        "font.size": 11,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    })

    def _apply_visual_standard(ax):
        try:
            ax.set_facecolor("white")
            if hasattr(ax, "spines"):
                for s in ax.spines.values():
                    s.set_visible(True)
                    s.set_color("#D7DEE8")
                    s.set_linewidth(1.0)

            if ax.get_title():
                ax.title.set_fontsize(max(float(ax.title.get_fontsize()), 11.0))
                ax.title.set_fontweight("bold")
            if ax.get_xlabel():
                ax.xaxis.label.set_fontsize(10)
            if ax.get_ylabel():
                ax.yaxis.label.set_fontsize(10)

            ax.tick_params(labelsize=9.5, colors="#1B2A41")

            # Grade discreta para eixos cartesianas (evita sobreposição em heatmaps/imagens).
            has_lines_or_bars = (len(getattr(ax, "lines", [])) + len(getattr(ax, "patches", []))) > 0
            has_images = len(getattr(ax, "images", [])) > 0
            if has_lines_or_bars and not has_images:
                ax.grid(alpha=0.20, color="#C9D1D9", linewidth=0.8)
                ax.set_axisbelow(True)
        except Exception:
            pass

    def _reflow_off_axis_texts(ax):
        """
        Reorganiza textos em eixos "off" para evitar sobreposição/corte.
        Atua apenas em textos posicionados em coordenadas do eixo (ax.transAxes).
        """
        try:
            if bool(getattr(ax, "axison", True)):
                return

            cand: list[tuple[float, object]] = []
            for t in list(getattr(ax, "texts", [])):
                if not t.get_visible():
                    continue
                try:
                    tr = t.get_transform()
                    in_axes = tr.contains_branch(ax.transAxes) if hasattr(tr, "contains_branch") else (tr == ax.transAxes)
                except Exception:
                    in_axes = False
                if not in_axes:
                    continue

                x, y = t.get_position()
                if not (isinstance(x, (int, float)) and isinstance(y, (int, float))):
                    continue
                if (x < -0.1) or (x > 1.1) or (y < -0.2) or (y > 1.2):
                    continue
                cand.append((float(y), t))

            if not cand:
                return

            cand.sort(key=lambda it: -it[0])
            y_cursor = min(0.97, max(it[0] for it in cand))
            y_bottom = 0.035

            for _y0, t in cand:
                txt_raw = str(t.get_text() or "")
                lines = txt_raw.splitlines() if txt_raw else [""]
                long_line = max((len(ln) for ln in lines), default=0)
                if long_line > 100:
                    extra = int(np.ceil((long_line - 100) / 70.0))
                    lines = lines + ([""] * max(0, extra))

                fs = float(t.get_fontsize() or 10.0)
                step = max(0.026, min(0.092, 0.014 + fs * 0.0032))
                gap = max(0.004, step * 0.18)
                need = max(1, len(lines)) * step + gap

                while (y_cursor - need < y_bottom) and (fs > 7.0):
                    fs = max(7.0, fs - 0.4)
                    t.set_fontsize(fs)
                    step = max(0.024, min(0.090, 0.014 + fs * 0.0032))
                    gap = max(0.004, step * 0.18)
                    need = max(1, len(lines)) * step + gap

                if y_cursor - need < y_bottom:
                    avail_lines = max(1, int((y_cursor - y_bottom - gap) / max(step, 1e-6)))
                    raw_lines = txt_raw.splitlines() if txt_raw else [""]
                    if len(raw_lines) > avail_lines:
                        raw_lines = raw_lines[:avail_lines]
                        if raw_lines:
                            last = raw_lines[-1]
                            if not last.endswith("..."):
                                raw_lines[-1] = (last[: max(1, len(last) - 1)] + "...")
                        t.set_text("\n".join(raw_lines))
                    lines = raw_lines
                    need = max(1, len(lines)) * step + gap

                x, _ = t.get_position()
                t.set_position((x, y_cursor))
                t.set_verticalalignment("top")
                t.set_wrap(True)
                y_cursor -= need
        except Exception:
            pass

    def _save_report_page(pdf_obj, fig_obj):
        try:
            fig_obj.patch.set_facecolor("#F6F8FB")
            try:
                all_axes_off = (len(fig_obj.axes) > 0) and all((not bool(getattr(_ax, "axison", True))) for _ax in fig_obj.axes)
                if all_axes_off:
                    fig_obj.subplots_adjust(left=0.02, right=0.985, top=0.985, bottom=0.035)
            except Exception:
                pass
            for _ax in fig_obj.axes:
                _apply_visual_standard(_ax)
            for _ax in fig_obj.axes:
                _reflow_off_axis_texts(_ax)
            fig_obj.text(
                0.995,
                0.006,
                "Relatório padronizado",
                ha="right",
                va="bottom",
                fontsize=8.0,
                color="#64748B",
            )
        except Exception:
            pass
        pdf_obj.savefig(fig_obj)
        plt.close(fig_obj)

    def _safe_tight_layout(fig_obj, rect=(0.02, 0.03, 0.98, 0.95), pad=1.15):
        try:
            import warnings as _warnings
            with _warnings.catch_warnings():
                _warnings.simplefilter("ignore", category=UserWarning)
                fig_obj.tight_layout(rect=rect, pad=pad)
        except Exception:
            pass

    def _wrap(s: str, width: int = 44) -> str:
        s = "" if s is None else str(s)
        s = s.strip()
        if len(s) <= width:
            return s
        words = s.split()
        out, line = [], ""
        for w in words:
            if len(line) + len(w) + 1 <= width:
                line = (line + " " + w).strip()
            else:
                if line:
                    out.append(line)
                line = w
        if line:
            out.append(line)
        return "\n".join(out)

    def _draw_wrapped_text(
        ax,
        x: float,
        y: float,
        text: str,
        *,
        width: int = 44,
        fontsize: float = 10.0,
        fontweight: str | None = None,
        line_step: float = 0.040,
        max_lines: int = 5,
    ) -> float:
        wrapped = _wrap(text, width=max(8, int(width)))
        lines = [ln for ln in str(wrapped).split("\n") if ln is not None]

        if not lines:
            return y

        step_eff = min(float(line_step), min(0.090, 0.015 + float(fontsize) * 0.0032))
        step_eff = max(0.022, step_eff)
        y_min = 0.008

        for ln in lines:
            if y <= y_min:
                break
            ax.text(x, y, ln, fontsize=fontsize, fontweight=fontweight, va="top")
            y -= step_eff

        y -= max(0.006, step_eff * 0.18)
        return y

    def _as_series(df_: pd.DataFrame, col: str) -> pd.Series:
        x = df_.loc[:, col]
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        return x

    def _pick_col_local(df_: pd.DataFrame, candidates: list[str]) -> str | None:
        for c in candidates:
            if c in df_.columns:
                return c
        return None

    def _norm_text(s: str) -> str:
        s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
        return re.sub(r"\s+", " ", s).strip().lower()

    def _match_any(text_norm: str, patterns_norm: list[str]) -> bool:
        return any(p in text_norm for p in patterns_norm)

    def _annotate_barh(ax, total_base: int | None = None, fs: int = 9):
        xmax = ax.get_xlim()[1] if ax.get_xlim()[1] > 0 else 1.0
        pad = xmax * 0.012
        for p in ax.patches:
            v = float(p.get_width())
            if v <= 0:
                continue
            y = p.get_y() + p.get_height() / 2
            label = f"{int(v)} ({(v / total_base):.1%})" if (total_base and total_base > 0) else f"{int(v)}"
            x = min(v + pad, xmax * 0.985)
            ha = "left" if x < xmax * 0.96 else "right"
            ax.text(x, y, label, va="center", ha=ha, fontsize=fs)

    def _annotate_bar(ax, total_base: int | None = None, fs: int = 9):
        ymax = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1.0
        pad = ymax * 0.015
        for p in ax.patches:
            v = float(p.get_height())
            if v <= 0:
                continue
            x = p.get_x() + p.get_width() / 2
            label = f"{int(v)}\n({(v / total_base):.1%})" if (total_base and total_base > 0) else f"{int(v)}"
            ax.text(x, v + pad, label, ha="center", va="bottom", fontsize=fs)

    def _pie_with_legend(ax, values, legend_labels, title, base_text):
        values = [int(v) for v in values]
        total_vals = sum(values)
        ax.set_title(title, pad=16, fontsize=12)
        if total_vals <= 0:
            ax.axis("off")
            ax.text(0.02, 0.90, "Sem dados suficientes.", fontsize=10.5, va="top")
            return

        # Paleta fixa em tons de azul para evitar laranja automático do matplotlib.
        # Inversão solicitada no pizza: o que era azul claro vira azul escuro e vice-versa.
        pie_palette = ["#A8D8F0", "#0B5FA5", "#7FB0D6", "#001E50"]
        pie_colors = [pie_palette[i % len(pie_palette)] for i in range(len(values))]

        wedges, _, autotexts = ax.pie(
            values,
            labels=None,
            colors=pie_colors,
            autopct=lambda p: f"{p:.0f}%" if p > 0 else "",
            startangle=90,
            pctdistance=0.72,
            radius=0.88,
            wedgeprops={"linewidth": 0.8, "edgecolor": "white"},
            textprops={"fontsize": 9},
        )
        for t in autotexts:
            t.set_fontsize(9)

        ax.legend(
            wedges,
            legend_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.98),
            ncol=min(2, len(legend_labels)),
            fontsize=8.5,
            frameon=False,
            borderaxespad=0.0,
        )
        ax.text(0.5, 0.04, base_text, ha="center", va="bottom", transform=ax.transAxes, fontsize=8.8)
        ax.set_aspect("equal")

    def _find_dim_colab_source(required_cols: list[str]) -> pd.DataFrame | None:
        candidates = [
            "dim_colaborador", "df_dim_colaborador", "df_colaboradores", "df_colaborador",
            "vw_colaboradores", "df_colaborador_hr", "vw_colaborador_hr", "df_colaboradorRH"
        ]
        for name in candidates:
            obj = globals().get(name)
            if isinstance(obj, pd.DataFrame) and all(c in obj.columns for c in required_cols):
                return obj
        return None

    def _get_dim_map(area_col_use: str | None, col_heranca: str | None, col_cargo: str | None = None) -> pd.DataFrame | None:
        if (area_col_use is None) or (col_heranca is None):
            return None

        cache_key = ("_DIM_MAP_CACHE", area_col_use, col_heranca, col_cargo or "")
        if cache_key in globals():
            cached = globals().get(cache_key)
            if isinstance(cached, pd.DataFrame) and not cached.empty and "cadastro" in cached.columns:
                return cached

        required = ["cadastro", area_col_use, col_heranca]
        required_cargo = required + ([col_cargo] if col_cargo else [])

        src = _find_dim_colab_source(required_cargo)
        if src is not None:
            dim = src.loc[:, required_cargo].copy(deep=False).drop_duplicates(subset=["cadastro"], keep="last")
            dim["cadastro"] = dim["cadastro"].astype(str).str.strip()
            dim["_area_norm"] = dim[area_col_use].fillna("").astype(str).str.strip()
            dim["_her_norm"] = dim[col_heranca].fillna("").astype(str).str.strip()
            if col_cargo and (col_cargo in dim.columns):
                dim["_cargo_norm"] = dim[col_cargo].fillna("").astype(str).str.strip().str.upper()
            globals()[cache_key] = dim[["cadastro", "_area_norm", "_her_norm"] + (["_cargo_norm"] if "_cargo_norm" in dim.columns else [])]
            return globals()[cache_key]

        base_all = globals().get("df_min_raw")
        if not isinstance(base_all, pd.DataFrame):
            return None
        if not all(c in base_all.columns for c in required_cargo):
            return None

        seen: set[str] = set()
        parts = []
        n = len(base_all)
        cols = required_cargo

        for i in range(0, n, BATCH_SIZE):
            chunk = base_all.iloc[i:i + BATCH_SIZE][cols].copy(deep=False)
            if chunk.empty:
                continue
            chunk = chunk.drop_duplicates(subset=["cadastro"], keep="last")
            cad_str = chunk["cadastro"].astype(str).str.strip()
            new_mask = ~cad_str.isin(seen)
            if not new_mask.any():
                continue
            chunk = chunk.loc[new_mask]
            cad_new = cad_str.loc[new_mask]
            seen.update(cad_new.tolist())
            chunk = chunk.assign(cadastro=cad_new.to_numpy(copy=False))
            parts.append(chunk)
            if len(seen) >= 400_000:
                break

        if not parts:
            return None

        dim = pd.concat(parts, ignore_index=True).drop_duplicates(subset=["cadastro"], keep="last")
        dim["_area_norm"] = dim[area_col_use].fillna("").astype(str).str.strip()
        dim["_her_norm"] = dim[col_heranca].fillna("").astype(str).str.strip()
        if col_cargo and (col_cargo in dim.columns):
            dim["_cargo_norm"] = dim[col_cargo].fillna("").astype(str).str.strip().str.upper()

        globals()[cache_key] = dim[["cadastro", "_area_norm", "_her_norm"] + (["_cargo_norm"] if "_cargo_norm" in dim.columns else [])]
        return globals()[cache_key]

    def _build_dt_from_data_hora(df_: pd.DataFrame) -> pd.Series:
        if "_dt_ronda" in df_.columns:
            return pd.to_datetime(df_["_dt_ronda"], errors="coerce")
        if ("data" in df_.columns) and ("hora" in df_.columns):
            data_dt = pd.to_datetime(df_["data"], dayfirst=True, errors="coerce")
            hora_s = df_["hora"].fillna("00:00").astype(str).str.strip()
            hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            return data_dt.dt.normalize() + hora_td
        return pd.Series(pd.NaT, index=df_.index)

    def _montar_metodologia_relatorio(titulo_local: str, pdf_path_local: Path | str, janela_proxy_minutos: int) -> tuple[str, list[str], list[str]]:
        txt_norm = _norm_text(f"{titulo_local} {pdf_path_local}")

        eh_ponto = _match_any(
            txt_norm,
            ["irregularidade_ponto_sem_portaria", "ponto sem portaria"],
        )
        eh_grill = _match_any(
            txt_norm,
            [
                "irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill",
                "nao entrou no refeitorio mas passou no grill",
            ],
        )
        eh_999 = _match_any(
            txt_norm,
            ["irregularidade_999_como_trajeto", "999 como trajeto"],
        )

        if eh_ponto:
            titulo_met = "Metodologia - Ponto sem Portaria"
            passos = [
                "Unidade de contagem: a ocorrência é contabilizada em nível de pessoa-dia (deduplicação por RM e dia operacional).",
                "Base elegível: são considerados os dias em que a pessoa possui ponto presencial e atende aos filtros definidos para esta irregularidade.",
                "Regra principal: o caso é classificado como irregular quando a pessoa possui registro de ponto presencial, mas não possui nenhum registro de portaria no mesmo dia.",
                "Dias descartados: são desconsiderados os casos em que a própria pessoa possui registros consistentes de portaria no dia, pois isso reduz a evidência de que o ponto tenha sido batido por terceiros.",
                f"Análise de possível terceiro: para cada caso irregular, é verificado se existe outra pessoa com ao menos 1 registro de portaria no dia, com ocorrência em até {int(janela_proxy_minutos)} minutos do ponto do suspeito, na mesma planta e no mesmo coletor.",
                "Interpretação do terceiro identificado: quando esse pareamento ocorre, a pessoa encontrada é tratada como possível terceiro que pode ter registrado o ponto para o suspeito, pois ela esteve fisicamente associada ao evento, enquanto o suspeito não apresentou passagem na portaria no mesmo dia.",
            ]
            observacoes = [
                "Leitura dos nomes no relatório: RM | Nome abreviado | Área.",
                "Os gráficos de distribuição priorizam a representatividade percentual por pessoa-dia.",
            ]
            return titulo_met, passos, observacoes

        if eh_grill:
            titulo_met = "Metodologia - Refeitorio x Grill"
            passos = [
                "Unidade de contagem: ocorrencia em pessoa-dia (nao por volume de linhas brutas).",
                "Classificacao de local: REF em qualquer ponto da descricao e sem GRILL => refeitorio normal.",
                "Regra principal: houve passagem em GRILL no dia, mas sem passagem em refeitorio normal no mesmo dia.",
                "Consolidacao por pessoa-dia para evitar duplicidade de batidas no mesmo dia.",
                "Comparativos de concentracao em duas janelas: periodo total e ultimos 30 dias.",
                "Heatmaps destacam periodo, horario e dia da semana com maior concentracao.",
            ]
            observacoes = [
                "Top pessoas sempre exibido como RM | Nome abreviado | Area.",
                "Coletores destacados no contexto do caso (refeitorios mais presentes).",
            ]
            return titulo_met, passos, observacoes

        if eh_999:
            titulo_met = "Metodologia - 999 como Trajeto"
            passos = [
                "Unidade de contagem: ocorrencia em pessoa-dia (deduplicacao por RM e dia operacional).",
                "Base da irregularidade: eventos classificados como 999 como trajeto conforme regra da base tratada.",
                "Analise de concentracao por pessoa no periodo total e na janela dos ultimos 30 dias.",
                "Consolidacao de Top 3 e Top 10 por volume para ranking de maior recorrencia.",
                "Heatmaps mostram concentracao por periodo, horario e dia da semana.",
                "Contexto operacional destaca plantas e portarias (descricao de coletor) mais presentes.",
            ]
            observacoes = [
                "Top pessoas sempre exibido como RM | Nome abreviado | Area.",
                "Graficos de distribuicao mostram percentual para facilitar comparacao entre grupos.",
            ]
            return titulo_met, passos, observacoes

        return (
            "Metodologia da Analise",
            [
                "Unidade de contagem: ocorrencia em pessoa-dia.",
                "Padronizacao de data/hora, RM, area, planta e coletor antes das agregacoes.",
                "Concentracao analisada no periodo total e, quando aplicavel, nos ultimos 30 dias.",
                "Ranking e distribuicoes apresentados em percentual para leitura comparativa.",
            ],
            ["Top pessoas no formato RM | Nome abreviado | Area."],
        )

    def _render_metodologia_resumo_pdf(
        pdf,
        titulo_metodologia: str,
        passos: list[str],
        observacoes: list[str] | None = None,
    ) -> None:
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_axes([0.0, 0.0, 1.0, 1.0])
        ax.axis("off")

        bottom_margin = 0.04

        def _wrap_local(txt: str, width: int) -> list[str]:
            txt = "" if txt is None else str(txt)
            out: list[str] = []
            for blk in txt.replace("\r", "").split("\n"):
                blk = blk.strip()
                if blk == "":
                    out.append("")
                    continue
                words = blk.split()
                line = ""
                for w in words:
                    cand = (line + " " + w).strip()
                    if len(cand) <= width:
                        line = cand
                    else:
                        if line:
                            out.append(line)
                        line = w
                if line:
                    out.append(line)
            return out if out else [""]

        def _draw(x: float, y: float, text: str, fs: float, width: int, step: float, weight: str | None = None) -> float:
            for ln in _wrap_local(text, width):
                if y < bottom_margin:
                    return y
                ax.text(x, y, ln, fontsize=fs, fontweight=weight, va="top")
                y -= step
            return y

        y = 0.95
        y = _draw(0.03, y, titulo_metodologia, fs=18, width=90, step=0.045, weight="bold")
        y = _draw(0.03, y, "Resumo objetivo da logica aplicada neste relatorio.", fs=11.2, width=108, step=0.032)

        y -= 0.006
        y = _draw(0.03, y, "Etapas da analise", fs=13.0, width=90, step=0.034, weight="bold")
        for i, p in enumerate(passos, start=1):
            y = _draw(0.04, y, f"{i}. {p}", fs=10.6, width=116, step=0.028)

        if observacoes:
            y -= 0.004
            y = _draw(0.03, y, "Leitura do relatorio", fs=12.2, width=90, step=0.033, weight="bold")
            for obs in observacoes:
                y = _draw(0.04, y, f"- {obs}", fs=10.4, width=116, step=0.027)

        _save_report_page(pdf, fig)

    gen_date = _dt.date.today().strftime("%Y-%m-%d")
    try:
        janela_proxy_minutos = int(globals().get("janela_proxy_minutos", 3))
    except Exception:
        janela_proxy_minutos = 3
    janela_proxy_minutos = max(int(janela_proxy_minutos), 1)

    metodologia_titulo, metodologia_passos, metodologia_observacoes = _montar_metodologia_relatorio(
        titulo_local=titulo,
        pdf_path_local=pdf_path,
        janela_proxy_minutos=janela_proxy_minutos,
    )

    if df_irregularidade is None or df_irregularidade.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Ocorrências (pessoa-dia)", "0"),
                    ("Pessoas (cadastro)", "0"),
                    ("Dias com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem linhas para gerar o relatório."],
            )
            _render_metodologia_resumo_pdf(
                pdf=pdf,
                titulo_metodologia=metodologia_titulo,
                passos=metodologia_passos,
                observacoes=metodologia_observacoes,
            )
        return

    if getattr(df_irregularidade, "columns", None) is not None:
        df_irregularidade = df_irregularidade.loc[:, ~df_irregularidade.columns.duplicated()]

    titulo_norm = _norm_text(titulo)
    path_norm = _norm_text(str(pdf_path))

    excluir_planta_999_nesta_irregularidade = _deve_excluir_planta_999(
        titulo=titulo,
        pdf_path=pdf_path,
    )

    sem_corte_30 = _match_any(
        titulo_norm + " " + path_norm,
        [
            "semana de feriado prolongado",
            "feriado prolongado",
            "3 dias presenciais na semana",
            "irregularidade_pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado",
            "pessoas_sem_cumprir_os_3_dias_presenciais_em_semana_de_feriado_prolongado",
            "executivos com historico de 999 como trajeto",
            "irregularidade_ausencia_das_horas_presenciais_minimas",
            "ausencia das horas presenciais minimas",
            "irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill",
            "nao entrou no refeitorio mas passou no grill",
            "irregularidade_ponto_sem_portaria",
            "ponto sem portaria",
        ],
    )
    somente_executivos_atuais = ("executivos com historico de 999 como trajeto" in titulo_norm)

    excluir_exec_nessas = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_registro_sem_portaria",
            "registro sem portaria",
            "irregularidade_ponto_sem_portaria",
            "ponto sem portaria",
            "irregularidade_pessoas_trabalhando_sem_ganhar",
            "pessoas trabalhando sem ganhar",
            "irregularidade_pessoas_trabalhando_alem_da_jornada_de_trabalho",
            "pessoas trabalhando alem da jornada de trabalho",
        ],
    )
    analisar_proxy_ponto = _match_any(
        titulo_norm + " " + path_norm,
        ["irregularidade_ponto_sem_portaria", "ponto sem portaria"],
    )
    proxy_modo_reduzido = False
    if analisar_proxy_ponto and len(df_irregularidade) > 1_500_000:
        proxy_modo_reduzido = True
        print(
            "[INFO] Proxy de terceiros em modo reduzido nesta execução por volume alto "
            f"(linhas={len(df_irregularidade):,})."
        )

    analisar_heatmap_grill = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_nao_entrou_no_refeitorio_mas_passou_no_grill",
            "nao entrou no refeitorio mas passou no grill",
        ],
    )
    analisar_999_trajeto = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_999_como_trajeto",
            "999 como trajeto",
        ],
    )
    mostrar_periodo_desligado = _match_any(
        titulo_norm + " " + path_norm,
        [
            "irregularidade_registros_de_pessoas_desligadas",
            "registros de pessoas desligadas",
        ],
    )

    col_cad = _pick_col_local(df_irregularidade, ["cadastro"])
    col_nome = _pick_col_local(df_irregularidade, ["nome"])
    col_areas = "área" if "área" in df_irregularidade.columns else None
    col_area_fallback = _pick_col_local(df_irregularidade, ["area"])
    col_planta = "planta" if "planta" in df_irregularidade.columns else None
    col_heranca = "herança" if "herança" in df_irregularidade.columns else ("heranca" if "heranca" in df_irregularidade.columns else None)
    col_cargo = _pick_col_local(df_irregularidade, ["cargo", "cargo_atual"])
    col_data_dem = _pick_col_local(df_irregularidade, ["Data_Dem", "data_desligado", "DataDemissao", "data_demissao"])

    if "_dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "_dia_operacional"), errors="coerce")
    elif "dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "dia_operacional"), errors="coerce")
    else:
        data_dt = (
            pd.to_datetime(_as_series(df_irregularidade, "data"), dayfirst=True, errors="coerce")
            if "data" in df_irregularidade.columns
            else pd.Series(pd.NaT, index=df_irregularidade.index)
        )
        if "hora" not in df_irregularidade.columns:
            dia_op = data_dt.dt.normalize()
        else:
            hora_s = _as_series(df_irregularidade, "hora").fillna("00:00").astype(str).str.strip()
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            dia_op = (data_dt.dt.normalize() + hora_td - pd.Timedelta(hours=11)).dt.normalize()

    cols_keep = ["_dia_operacional"]
    if col_cad: cols_keep.append(col_cad)
    if col_nome: cols_keep.append(col_nome)
    if col_areas: cols_keep.append(col_areas)
    elif col_area_fallback: cols_keep.append(col_area_fallback)
    if col_planta: cols_keep.append(col_planta)
    if col_heranca: cols_keep.append(col_heranca)
    if col_cargo: cols_keep.append(col_cargo)
    if col_data_dem: cols_keep.append(col_data_dem)

    dfv = df_irregularidade.loc[:, [c for c in cols_keep if c != "_dia_operacional"]].copy(deep=False)
    dfv["_dia_operacional"] = _ensure_numpy_datetime_series(dia_op, name="_dia_operacional")

    if analisar_heatmap_grill and col_planta and (col_planta in dfv.columns):
        dfv = dfv.loc[_mask_planta_eq_1(dfv[col_planta])].copy()

    if excluir_planta_999_nesta_irregularidade and col_planta and (col_planta in dfv.columns):
        dfv = dfv.loc[_mask_planta_not_999(dfv[col_planta])].copy()

    if col_cad and (col_cad in dfv.columns):
        dfv[col_cad] = dfv[col_cad].astype(str).str.strip()
        dfv = dfv[dfv[col_cad].ne("")]

    if somente_executivos_atuais and col_cargo and (col_cargo in dfv.columns):
        cargo_norm = dfv[col_cargo].fillna("").astype(str).str.strip().str.upper()
        dfv = dfv.loc[cargo_norm.eq("E")]

    if excluir_exec_nessas and col_cargo and (col_cargo in dfv.columns):
        cargo_norm2 = dfv[col_cargo].fillna("").astype(str).str.strip().str.upper()
        dfv = dfv.loc[~cargo_norm2.eq("E")]

    key_cols = [col_cad, "_dia_operacional"] if (col_cad and col_cad in dfv.columns) else ["_dia_operacional"]
    df_occ = (
        dfv.dropna(subset=["_dia_operacional"])
        .drop_duplicates(subset=key_cols, keep="first")
    ).copy()

    df_occ["_dia_operacional"] = _ensure_numpy_datetime_series(
        df_occ["_dia_operacional"],
        name="_dia_operacional",
    )

    if df_occ.empty:
        with PdfPages(pdf_path) as pdf:
            _render_capa_padrao_pdf(
                pdf=pdf,
                titulo=titulo,
                gen_date=gen_date,
                metricas=[
                    ("Ocorrências (pessoa-dia)", "0"),
                    ("Pessoas (cadastro)", "0"),
                    ("Dias com ocorrência", "0"),
                ],
                periodo=None,
                concentracoes=[
                    ("Concentração Top 1", "0 (0.0%)"),
                    ("Concentração Top 3", "0 (0.0%)"),
                    ("Concentração Top 10", "0 (0.0%)"),
                ],
                top3_piores=[],
                top3_atuais=[],
                janela_info="0 (0.0% do total)",
                extras=["Sem ocorrências após normalização de dia operacional."],
            )
            _render_metodologia_resumo_pdf(
                pdf=pdf,
                titulo_metodologia=metodologia_titulo,
                passos=metodologia_passos,
                observacoes=metodologia_observacoes,
            )
        return

    MIN_OCORRENCIAS_PESSOA_DIA = 30
    if (not sem_corte_30) and col_cad and (col_cad in df_occ.columns):
        _cnt_pd = df_occ[col_cad].astype(str).str.strip().value_counts()
        _elig = _cnt_pd[_cnt_pd >= MIN_OCORRENCIAS_PESSOA_DIA].index
        if _elig.empty:
            with PdfPages(pdf_path) as pdf:
                _render_capa_padrao_pdf(
                    pdf=pdf,
                    titulo=titulo,
                    gen_date=gen_date,
                    metricas=[
                        ("Ocorrências (pessoa-dia)", "0"),
                        ("Pessoas (cadastro)", "0"),
                        ("Dias com ocorrência", "0"),
                    ],
                    periodo=None,
                    concentracoes=[
                        ("Concentração Top 1", "0 (0.0%)"),
                        ("Concentração Top 3", "0 (0.0%)"),
                        ("Concentração Top 10", "0 (0.0%)"),
                    ],
                    top3_piores=[],
                    top3_atuais=[],
                    janela_info="0 (0.0% do total)",
                    extras=["Sem dados suficientes (mínimo 30 ocorrências por pessoa-dia)."],
                )
                _render_metodologia_resumo_pdf(
                    pdf=pdf,
                    titulo_metodologia=metodologia_titulo,
                    passos=metodologia_passos,
                    observacoes=metodologia_observacoes,
                )
            return
        df_occ = df_occ.loc[df_occ[col_cad].astype(str).str.strip().isin(_elig)]
        if (col_cad in dfv.columns) and (not dfv.empty):
            dfv = dfv.loc[dfv[col_cad].astype(str).str.strip().isin(_elig)]

    total = int(len(df_occ))
    n_pessoas = int(df_occ[col_cad].nunique()) if (col_cad and col_cad in df_occ.columns) else 0
    n_dias = int(df_occ["_dia_operacional"].nunique())
    dt_min = df_occ["_dia_operacional"].min()
    dt_max = df_occ["_dia_operacional"].max()

    nome_map = {}
    if col_cad and col_nome and (col_cad in dfv.columns) and (col_nome in dfv.columns):
        tmp = dfv[[col_cad, col_nome]].copy()
        tmp[col_nome] = tmp[col_nome].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[col_nome].ne("")]
        if not tmp.empty:
            cnt = (
                tmp.groupby([col_cad, col_nome], sort=False)
                   .size()
                   .reset_index(name="n")
                   .sort_values([col_cad, "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map = cnt.drop_duplicates(subset=[col_cad], keep="first").set_index(col_cad)[col_nome].to_dict()

    periodo_desligado_map: dict[str, str] = {}
    if mostrar_periodo_desligado and col_data_dem and col_cad and (col_data_dem in dfv.columns) and (col_cad in dfv.columns):
        dem_aux = dfv[[col_cad, "_dia_operacional", col_data_dem]].copy()
        dem_aux["_dia_operacional"] = pd.to_datetime(dem_aux["_dia_operacional"], errors="coerce").dt.normalize().astype("datetime64[ns]")
        dem_aux[col_data_dem] = pd.to_datetime(dem_aux[col_data_dem], dayfirst=True, errors="coerce").dt.normalize().astype("datetime64[ns]")
        dem_aux = dem_aux.dropna(subset=[col_data_dem, "_dia_operacional"])
        if not dem_aux.empty:
            dem_aux[col_cad] = dem_aux[col_cad].astype(str).str.strip()
            dem_aux = dem_aux.loc[dem_aux[col_cad].ne("")]
            dem_aux = dem_aux.loc[dem_aux["_dia_operacional"] >= dem_aux[col_data_dem]]
            if not dem_aux.empty:
                agg_dem = (
                    dem_aux.groupby(col_cad)
                           .agg(dt_ini=(col_data_dem, "min"), dt_fim=("_dia_operacional", "max"))
                           .reset_index()
                )
                for r in agg_dem.itertuples(index=False):
                    cadk = str(getattr(r, col_cad)).strip()
                    if cadk:
                        dt_ini = pd.to_datetime(r.dt_ini, errors="coerce")
                        dt_fim = pd.to_datetime(r.dt_fim, errors="coerce")
                        if pd.notna(dt_ini) and pd.notna(dt_fim):
                            periodo_desligado_map[cadk] = f"{dt_ini.date()} → {dt_fim.date()}"
    if col_cad and (col_cad in df_occ.columns):
        cad_series = df_occ[col_cad].astype(str).str.strip()
        cad_counts = cad_series.value_counts()
        eligible_cads = cad_counts.index if sem_corte_30 else cad_counts[cad_counts >= MIN_OCORRENCIAS_PESSOA_DIA].index

        if len(eligible_cads) == 0:
            top_pessoas = pd.Series(dtype=int)
            top_pessoas_cad_counts = pd.Series(dtype=int)
        else:
            cad_series_elig = cad_series[cad_series.isin(eligible_cads)]
            top_pessoas = cad_series_elig.value_counts()
            top_pessoas_cad_counts = cad_counts.loc[eligible_cads]
    else:
        top_pessoas = pd.Series(dtype=int)
        top_pessoas_cad_counts = pd.Series(dtype=int)

    top1_n = int(top_pessoas.head(1).sum()) if (not top_pessoas.empty) else 0
    top3_n = int(top_pessoas.head(3).sum()) if (not top_pessoas.empty) else 0
    top10_n = int(top_pessoas.head(10).sum()) if (not top_pessoas.empty) else 0
    top1_pct = (top1_n / total) if total else 0.0
    top3_pct = (top3_n / total) if total else 0.0
    top10_pct = (top10_n / total) if total else 0.0

    dia_op_np = pd.Series(
        pd.to_datetime(df_occ["_dia_operacional"], errors="coerce").to_numpy(dtype="datetime64[ns]"),
        index=df_occ.index,
        name="_dia_operacional",
    )

    mes = dia_op_np.dt.to_period("M").dt.to_timestamp()
    mensal = mes.value_counts().sort_index()

    dow_map = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
    dow = (
        df_occ["_dia_operacional"].dt.dayofweek.map(dow_map)
            .value_counts()
            .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
            .fillna(0).astype(int)
    )

    dt_max_norm = pd.to_datetime(dt_max).normalize()
    window_days = 30
    if pd.notna(dt_min) and pd.notna(dt_max):
        span = (pd.to_datetime(dt_max) - pd.to_datetime(dt_min)).days
        if span < 45:
            window_days = 14
        if span < 21:
            window_days = 7

    dt_cut = dt_max_norm - pd.Timedelta(days=window_days)
    recent = df_occ.loc[df_occ["_dia_operacional"] >= dt_cut]
    total_recent = int(len(recent))
    if col_cad and (col_cad in df_occ.columns) and total_recent > 0:
        cad_recent = recent[col_cad].astype(str).str.strip()
        if not sem_corte_30 and (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
            cad_recent = cad_recent[cad_recent.isin(top_pessoas_cad_counts.index.astype(str))]
        top_recent = cad_recent.value_counts()
    else:
        top_recent = pd.Series(dtype=int)

    def _rep_value_per_occ(df_in: pd.DataFrame, key_cols2: list[str], val_col: str) -> pd.Series:
        if (val_col is None) or (val_col not in df_in.columns):
            return pd.Series(dtype=str)
        tmp2 = df_in[key_cols2 + [val_col]].copy()
        tmp2[val_col] = tmp2[val_col].fillna("").astype(str).str.strip()
        tmp2 = tmp2[tmp2[val_col].ne("")]
        if tmp2.empty:
            return pd.Series(dtype=str)
        cnt = (
            tmp2.groupby(key_cols2 + [val_col], sort=False)
                .size()
                .reset_index(name="n")
                .sort_values(key_cols2 + ["n"], ascending=[True] * len(key_cols2) + [False], kind="mergesort")
        )
        rep = cnt.drop_duplicates(subset=key_cols2, keep="first").set_index(key_cols2)[val_col]
        return rep

    area_col_use = col_areas if col_areas else col_area_fallback
    dim_map = _get_dim_map(area_col_use, col_heranca, col_cargo)

    area_map: dict[str, str] = {}
    if isinstance(dim_map, pd.DataFrame) and (not dim_map.empty) and ("cadastro" in dim_map.columns) and ("_area_norm" in dim_map.columns):
        _area_tmp = dim_map[["cadastro", "_area_norm"]].copy(deep=False)
        _area_tmp["cadastro"] = _area_tmp["cadastro"].astype(str).str.strip()
        _area_tmp["_area_norm"] = _area_tmp["_area_norm"].fillna("").astype(str).str.strip()
        _area_tmp = _area_tmp.loc[_area_tmp["cadastro"].ne("")]
        if not _area_tmp.empty:
            area_map = _area_tmp.drop_duplicates(subset=["cadastro"], keep="last").set_index("cadastro")["_area_norm"].to_dict()

    def _collect_data_adm_map() -> dict[str, pd.Timestamp]:
        adm_map: dict[str, pd.Timestamp] = {}

        def _append_from_df(df_src: pd.DataFrame | None) -> None:
            if not isinstance(df_src, pd.DataFrame) or df_src.empty:
                return

            cad_col = _pick_col_local(df_src, ["cadastro", "Cadastro"])
            adm_col = _pick_col_local(
                df_src,
                [
                    "Data_Adm",
                    "data_adm",
                    "data_admissao",
                    "data_admissão",
                    "DataAdmissao",
                    "DataAdmissão",
                ],
            )
            if (cad_col is None) or (adm_col is None):
                return

            tmp = df_src.loc[:, [cad_col, adm_col]].copy(deep=False)
            tmp[cad_col] = tmp[cad_col].astype(str).str.strip()
            tmp[adm_col] = pd.to_datetime(tmp[adm_col], errors="coerce", dayfirst=True)
            tmp = tmp.loc[tmp[cad_col].ne("") & tmp[adm_col].notna()]
            if tmp.empty:
                return

            tmp = tmp.sort_values(adm_col, kind="mergesort").drop_duplicates(subset=[cad_col], keep="first")
            for cad_v, adm_v in tmp.loc[:, [cad_col, adm_col]].itertuples(index=False, name=None):
                cad = str(cad_v).strip()
                dt_adm = pd.to_datetime(adm_v, errors="coerce")
                if cad and pd.notna(dt_adm) and (cad not in adm_map):
                    adm_map[cad] = dt_adm.normalize()

        _append_from_df(df_irregularidade)
        for nm in ("df_desligados", "df_dim_colaborador", "dim_colaborador", "df_colaboradores", "df_colaborador"):
            _append_from_df(globals().get(nm))

        if adm_map:
            return adm_map

        for p in (
            Path("./df_desligados.parquet"),
            Path("./Códigos Extras/Documentação/Analisando Irregularidades dos Registros/df_desligados.parquet"),
        ):
            try:
                if not p.exists():
                    continue
                df_adm = pd.read_parquet(p)
                _append_from_df(df_adm)
                if adm_map:
                    break
            except Exception:
                continue

        return adm_map

    data_adm_map = _collect_data_adm_map()
    ref_tempo_casa = pd.to_datetime(dt_max, errors="coerce")
    if pd.isna(ref_tempo_casa):
        ref_tempo_casa = pd.Timestamp.today().normalize()
    else:
        ref_tempo_casa = ref_tempo_casa.normalize()

    def _format_tempo_casa(cad: str) -> str:
        cad = str(cad).strip()
        if not cad:
            return "-"

        dt_adm = data_adm_map.get(cad)
        if dt_adm is None or pd.isna(dt_adm):
            return "-"

        dt_adm = pd.to_datetime(dt_adm, errors="coerce")
        if pd.isna(dt_adm):
            return "-"

        months = (ref_tempo_casa.year - dt_adm.year) * 12 + (ref_tempo_casa.month - dt_adm.month)
        if ref_tempo_casa.day < dt_adm.day:
            months -= 1
        if months < 0:
            return "-"

        anos, meses = divmod(int(months), 12)
        if anos <= 0:
            return f"{meses}m"
        if meses == 0:
            return f"{anos}a"
        return f"{anos}a {meses}m"

    def _top_val_por_cadastro(df_src: pd.DataFrame, cad_col: str | None, val_col: str | None) -> dict[str, str]:
        if (cad_col is None) or (val_col is None):
            return {}
        if (cad_col not in df_src.columns) or (val_col not in df_src.columns):
            return {}

        tmp = df_src[[cad_col, val_col]].copy(deep=False)
        tmp[cad_col] = tmp[cad_col].astype(str).str.strip()
        tmp[val_col] = tmp[val_col].fillna("").astype(str).str.strip()
        tmp = tmp.loc[tmp[cad_col].ne("") & tmp[val_col].ne("")]
        if tmp.empty:
            return {}

        cnt = (
            tmp.groupby([cad_col, val_col], sort=False)
               .size()
               .reset_index(name="n")
               .sort_values([cad_col, "n", val_col], ascending=[True, False, True], kind="mergesort")
        )
        top = cnt.drop_duplicates(subset=[cad_col], keep="first")
        return top.set_index(cad_col)[val_col].astype(str).to_dict()

    col_desc_coletor = _pick_col_local(
        df_irregularidade,
        ["descricao coletor", "descricao_coletor", "desc_coletor", "coletor", "descricao"],
    )

    df_caso_top_local = df_irregularidade
    coletor_top_rotulo = "Coletor top"

    if (col_desc_coletor is not None) and (col_desc_coletor in df_irregularidade.columns):
        desc_case = df_irregularidade[col_desc_coletor].fillna("").astype(str)

        if analisar_999_trajeto:
            re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
            mask_case = desc_case.str.contains(re_port, case=False, na=False, regex=True)
            coletor_top_rotulo = "Portaria top"
        elif analisar_proxy_ponto:
            re_ponto = globals().get("regex_ponto", r"PONTO")
            re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
            mask_case = (
                desc_case.str.contains(re_ponto, case=False, na=False, regex=True)
                & (~desc_case.str.contains(re_port, case=False, na=False, regex=True))
            )
            coletor_top_rotulo = "Ponto top"
        elif analisar_heatmap_grill:
            re_ref = globals().get("regex_refeitorio", r"REF")
            re_grill = globals().get("regex_grill", r"(?i)\bGRILL\b")
            mask_case = (
                desc_case.str.contains(re_ref, case=False, na=False, regex=True)
                & (~desc_case.str.contains(re_grill, case=False, na=False, regex=True))
            )
            coletor_top_rotulo = "Refeitório top"
        else:
            mask_case = pd.Series(True, index=df_irregularidade.index)

        df_caso_top_local = df_irregularidade.loc[mask_case].copy(deep=False)
        if analisar_heatmap_grill and col_planta and (col_planta in df_caso_top_local.columns):
            df_caso_top_local = df_caso_top_local.loc[_mask_planta_eq_1(df_caso_top_local[col_planta])].copy(deep=False)
        if df_caso_top_local.empty:
            if analisar_heatmap_grill and col_planta and (col_planta in df_irregularidade.columns):
                df_caso_top_local = df_irregularidade.loc[_mask_planta_eq_1(df_irregularidade[col_planta])].copy(deep=False)
            else:
                df_caso_top_local = df_irregularidade

    top_planta_cad_map = _top_val_por_cadastro(df_caso_top_local, col_cad, col_planta)
    top_coletor_cad_map = _top_val_por_cadastro(df_caso_top_local, col_cad, col_desc_coletor)

    area_rep = pd.Series(dtype=str)
    if area_col_use and col_cad and (col_cad in dfv.columns):
        area_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], area_col_use)

    planta_rep = pd.Series(dtype=str)
    if col_planta and col_cad and (col_cad in dfv.columns):
        planta_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], col_planta)
        if not planta_rep.empty:
            _pl_str = planta_rep.astype(str).str.strip()
            _pl_num = pd.to_numeric(_pl_str, errors="coerce")
            _mask_999 = _pl_num.eq(999) | _pl_str.str.match(r"^999(\D|$)", na=False)
            planta_rep = planta_rep.loc[~_mask_999]
            planta_rep = planta_rep.map(_fmt_planta_nome)

    combo_rep = pd.Series(dtype=str)
    if (not area_rep.empty) and (not planta_rep.empty):
        idx = area_rep.index.intersection(planta_rep.index)
        if len(idx) > 0:
            combo_rep = (area_rep.loc[idx].astype(str) + " | " + planta_rep.loc[idx].astype(str))

    top_areas = area_rep.value_counts() if not area_rep.empty else pd.Series(dtype=int)
    top_plantas = planta_rep.value_counts() if not planta_rep.empty else pd.Series(dtype=int)
    top_combo = combo_rep.value_counts() if not combo_rep.empty else pd.Series(dtype=int)

    # =========================
    # PROXY (Ponto sem Portaria):
    # - mesmo dia operacional
    # - janela de até 3 minutos
    # - exige mesma planta e mesmo número do coletor
    # =========================
    proxy_map: dict[str, dict] = {}
    proxy_ranked_cads: list[str] = []
    proxy_top3_cads: list[str] = []
    valid_proxy_cads: set[str] = set()
    proxy_diag = {
        "candidatos_avaliados": 0,
        "candidatos_com_eventos_elegiveis": 0,
        "eventos_elegiveis_candidatos": 0,
        "pares_proximos_tempo": 0,
        "pares_com_reforco_planta_ou_familia": 0,
        "pares_aceitos_suspeitos": 0,
        "pessoas_com_indicio": 0,
        "pessoas_com_recorrencia": 0,
    }

    if analisar_proxy_ponto and col_cad and (col_cad in df_occ.columns):
        base_all = globals().get("df_valid")
        if not isinstance(base_all, pd.DataFrame):
            base_all = globals().get("df")
        if not isinstance(base_all, pd.DataFrame):
            base_all = df_irregularidade

        desc_col = None
        if "descricao coletor" in base_all.columns:
            desc_col = "descricao coletor"
        else:
            for c in ["descricao_coletor", "desc_coletor", "coletor", "descricao"]:
                if c in base_all.columns:
                    desc_col = c
                    break

        # Se df_valid não tiver descrição do coletor, tenta a base da própria irregularidade.
        if desc_col is None and isinstance(df_irregularidade, pd.DataFrame):
            base_alt = df_irregularidade
            if "descricao coletor" in base_alt.columns:
                desc_col = "descricao coletor"
            else:
                for c in ["descricao_coletor", "desc_coletor", "coletor", "descricao"]:
                    if c in base_alt.columns:
                        desc_col = c
                        break
            if desc_col is not None and ("cadastro" in base_alt.columns):
                base_all = base_alt

        if desc_col is not None and ("cadastro" in base_all.columns):
            MAX_PROXY_CANDIDATOS = int(
                globals().get(
                    "max_proxy_candidatos_reduzido" if proxy_modo_reduzido else "max_proxy_candidatos_padrao",
                    120 if proxy_modo_reduzido else 60,
                )
            )
            JANELA_SEGUNDOS = int(janela_proxy_minutos) * 60
            MAX_EVENTOS_POR_CANDIDATO = 3000

            if (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
                proxy_candidate_cads = list(top_pessoas_cad_counts.head(MAX_PROXY_CANDIDATOS).index.astype(str))
            else:
                proxy_candidate_cads = list(df_occ[col_cad].astype(str).str.strip().dropna().unique()[:MAX_PROXY_CANDIDATOS])

            proxy_diag["candidatos_avaliados"] = int(len(proxy_candidate_cads))

            base = base_all.copy(deep=False)
            if proxy_candidate_cads:
                cads_set = set(str(c).strip() for c in proxy_candidate_cads if str(c).strip())
                if cads_set:
                    cad_base_filter = base["cadastro"].astype(str).str.strip()
                    base = base.loc[cad_base_filter.isin(cads_set)].copy(deep=False)

            # Terceiros podem ser de qualquer cargo (H/M/E).
            cad_base = base["cadastro"].astype(str).str.strip()

            if "_dt_ronda" in base.columns:
                dt_base = pd.to_datetime(base["_dt_ronda"], errors="coerce")
            else:
                dt_base = pd.to_datetime(_build_dt_from_data_hora(base), errors="coerce")

            if "_dia_operacional" in base.columns:
                dia_op_base = pd.to_datetime(base["_dia_operacional"], errors="coerce").dt.normalize()
            else:
                dia_op_base = (dt_base - pd.Timedelta(hours=11)).dt.normalize()

            if "planta" in base.columns:
                pl_base = pd.to_numeric(base["planta"], errors="coerce")
            else:
                pl_base = pd.Series(np.nan, index=base.index)

            desc_base = base[desc_col].fillna("").astype(str).str.strip()

            def _proxy_num_coletor_series(s: pd.Series) -> pd.Series:
                txt = s.fillna("").astype(str).str.strip().str.upper()
                txt = txt.apply(lambda v: unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii"))
                num = txt.str.extract(r"(?:COLETOR|CATRACA|RELOGIO(?:\s*PONTO)?|PONTO|MARCA)\D*([0-9]{1,6})", expand=False)
                num_fallback = txt.str.extract(r"\b([0-9]{1,6})\b", expand=False)
                num = num.fillna(num_fallback)
                return num.fillna("").astype(str).str.strip()

            coletor_base = _proxy_num_coletor_series(desc_base)

            point_mask = pd.Series(False, index=base.index)
            for bc in ["Ponto", "PONTO", "_PONTO", "_PONTO_PRESENCIAL", "PONTO_PRES", "HAS_PONTO_PRES", "PontoPresencial"]:
                if bc in base.columns and pd.api.types.is_bool_dtype(base[bc]):
                    point_mask = point_mask | base[bc].fillna(False)
            if not point_mask.any():
                point_mask = desc_base.str.contains(r"\bponto\b|rel[oó]gio|marca", case=False, na=False, regex=True)

            port_mask = pd.Series(False, index=base.index)
            for bc in ["Portaria", "PORTARIA", "_PORTARIA", "HAS_PORTARIA_ANY"]:
                if bc in base.columns and pd.api.types.is_bool_dtype(base[bc]):
                    port_mask = port_mask | base[bc].fillna(False)
            if not port_mask.any():
                _re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
                port_mask = desc_base.str.contains(_re_port, case=False, na=False, regex=True)

            port_day_mask = port_mask & dia_op_base.notna() & cad_base.ne("")
            if port_day_mask.any():
                port_cad_arr = cad_base.loc[port_day_mask].astype(str).str.strip().to_numpy(dtype=object)
                port_day_arr = pd.to_datetime(dia_op_base.loc[port_day_mask], errors="coerce").to_numpy(dtype="datetime64[D]").astype(np.int64)
                portaria_day_set = set(zip(port_cad_arr.tolist(), port_day_arr.tolist()))
            else:
                portaria_day_set = set()

            base_mask = point_mask & dt_base.notna() & dia_op_base.notna() & cad_base.ne("")

            if base_mask.any():
                ev = pd.DataFrame({
                    "cad": cad_base.loc[base_mask].astype(str).str.strip().to_numpy(),
                    "dt": pd.to_datetime(dt_base.loc[base_mask], errors="coerce").to_numpy(dtype="datetime64[ns]"),
                    "dia_op": pd.to_datetime(dia_op_base.loc[base_mask], errors="coerce").to_numpy(dtype="datetime64[ns]"),
                    "planta": pd.to_numeric(pl_base.loc[base_mask], errors="coerce").to_numpy(dtype=float),
                    "coletor_num": coletor_base.loc[base_mask].fillna("").astype(str).str.strip().to_numpy(dtype=object),
                })

                ev = ev.dropna(subset=["dt", "dia_op"]).copy()
                if not ev.empty:
                    ev["sec"] = (ev["dt"].astype("int64") // 1_000_000_000).astype(np.int64)
                    ev["dia_i"] = pd.to_datetime(ev["dia_op"], errors="coerce").to_numpy(dtype="datetime64[D]").astype(np.int64)
                    ev["planta_i"] = pd.to_numeric(ev["planta"], errors="coerce")
                    ev["coletor_num"] = ev["coletor_num"].fillna("").astype(str).str.strip()

                    day_groups: dict[int, dict[str, np.ndarray]] = {}
                    for day_i, g in ev.groupby("dia_i", sort=False):
                        g = g.sort_values("sec", kind="mergesort")
                        day_groups[int(day_i)] = {
                            "sec": g["sec"].to_numpy(dtype=np.int64, copy=False),
                            "cad": g["cad"].to_numpy(dtype=object, copy=False),
                            "planta": pd.to_numeric(g["planta_i"], errors="coerce").to_numpy(dtype=float, copy=False),
                            "coletor": g["coletor_num"].to_numpy(dtype=object, copy=False),
                        }

                    counts: dict[str, dict[str, int]] = {str(c): {} for c in proxy_candidate_cads}
                    counts_days: dict[str, dict[str, set[int]]] = {str(c): {} for c in proxy_candidate_cads}
                    cad_proxy_days: dict[str, set[int]] = {str(c): set() for c in proxy_candidate_cads}

                    for cad0 in proxy_candidate_cads:
                        ev0 = ev.loc[ev["cad"].eq(str(cad0)), ["dia_i", "sec", "planta_i", "coletor_num"]]
                        if ev0.empty:
                            continue

                        proxy_diag["candidatos_com_eventos_elegiveis"] += 1
                        proxy_diag["eventos_elegiveis_candidatos"] += int(len(ev0))

                        if len(ev0) > MAX_EVENTOS_POR_CANDIDATO:
                            ev0 = ev0.sort_values("sec", kind="mergesort").head(MAX_EVENTOS_POR_CANDIDATO)

                        dct = counts.setdefault(str(cad0), {})
                        dct_days = counts_days.setdefault(str(cad0), {})
                        cad_days = cad_proxy_days.setdefault(str(cad0), set())

                        for day_i, g0 in ev0.groupby("dia_i", sort=False):
                            day_data = day_groups.get(int(day_i))
                            if day_data is None:
                                continue

                            sec_day = day_data["sec"]
                            cad_day = day_data["cad"]
                            pl_day = day_data["planta"]
                            coletor_day = day_data["coletor"]

                            for row in g0.itertuples(index=False):
                                sec0 = int(row.sec)
                                l = int(np.searchsorted(sec_day, sec0 - JANELA_SEGUNDOS, side="left"))
                                r = int(np.searchsorted(sec_day, sec0 + JANELA_SEGUNDOS, side="right"))
                                if r <= l:
                                    continue

                                sec_sub = sec_day[l:r]
                                cad_sub = cad_day[l:r]
                                pl_sub = pl_day[l:r]
                                coletor_sub = coletor_day[l:r]

                                mask_other = np.array([str(c).strip() != str(cad0) for c in cad_sub], dtype=bool)
                                if not mask_other.any():
                                    continue

                                sec_other = sec_sub[mask_other]
                                cad_other = cad_sub[mask_other]
                                pl_other = pl_sub[mask_other]
                                coletor_other = coletor_sub[mask_other]

                                dif = np.abs(sec_other - sec0)
                                proxy_diag["pares_proximos_tempo"] += int(len(dif))

                                pl0 = pd.to_numeric(row.planta_i, errors="coerce")
                                coletor0 = str(row.coletor_num).strip()

                                same_planta = (~np.isnan(pl_other)) & (not pd.isna(pl0)) & (pl_other == float(pl0))
                                same_coletor = np.array(
                                    [(str(c).strip() != "") and (str(c).strip() == coletor0) and (coletor0 != "") for c in coletor_other],
                                    dtype=bool,
                                )

                                same_planta_coletor = same_planta & same_coletor
                                # Mantém chave legada por compatibilidade com o funil já existente.
                                proxy_diag["pares_com_reforco_planta_ou_familia"] += int(same_planta_coletor.sum())

                                # Aceita somente quando TODAS as condições forem verdadeiras:
                                # <= janela_proxy_minutos, mesma planta e mesmo número do coletor.
                                has_portaria_other = np.array(
                                    [((str(c).strip(), int(day_i)) in portaria_day_set) for c in cad_other],
                                    dtype=bool,
                                )
                                accepted = (dif <= JANELA_SEGUNDOS) & same_planta_coletor & has_portaria_other
                                if not accepted.any():
                                    continue

                                proxy_diag["pares_aceitos_suspeitos"] += int(accepted.sum())
                                day_key_i = int(day_i)
                                cad_days.add(day_key_i)
                                for cad_other_i in cad_other[accepted]:
                                    cad_o = str(cad_other_i).strip()
                                    if cad_o == "" or cad_o == str(cad0):
                                        continue
                                    dct[cad_o] = dct.get(cad_o, 0) + 1
                                    if cad_o not in dct_days:
                                        dct_days[cad_o] = set()
                                    dct_days[cad_o].add(day_key_i)

                    for cad0, dct in counts.items():
                        if not dct:
                            continue

                        dct_days = counts_days.get(str(cad0), {})
                        dct_rank = {k: int(len(vs)) for k, vs in dct_days.items()} if dct_days else dct
                        top_other = sorted(dct_rank.items(), key=lambda x: (-x[1], x[0]))[:10]
                        total_hits = int(len(cad_proxy_days.get(str(cad0), set())))
                        max_hits_mesmo_terceiro = int(top_other[0][1]) if top_other else 0

                        proxy_map[str(cad0)] = {
                            "top_other": [(k, int(v)) for k, v in top_other],
                            "score_total_proxy": total_hits,
                            "qtd_terceiros": int(len(dct_rank)),
                            "max_hits_mesmo_terceiro": max_hits_mesmo_terceiro,
                            "pares_total_proxy": int(sum(dct.values())),
                        }

                    proxy_diag["pessoas_com_indicio"] = int(len(proxy_map))
                    proxy_diag["pessoas_com_recorrencia"] = int(
                        sum(1 for _cad, info in proxy_map.items() if int(info.get("max_hits_mesmo_terceiro", 0)) >= 2)
                    )

                    if proxy_map:
                        def _cad_volume(cad: str) -> int:
                            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty:
                                return 0
                            try:
                                return int(top_pessoas_cad_counts.get(cad, 0))
                            except Exception:
                                return 0

                        proxy_ranked_cads = [
                            cad for cad, _info in sorted(
                                proxy_map.items(),
                                key=lambda kv: (
                                    -int(kv[1].get("score_total_proxy", 0)),        # 1) total de ocorrências suspeitas
                                    -int(kv[1].get("max_hits_mesmo_terceiro", 0)), # 2) repetição do mesmo terceiro
                                    -_cad_volume(kv[0]),                           # 3) volume total da pessoa
                                    str(kv[0]),
                                )
                            )
                        ]
                        proxy_top3_cads = proxy_ranked_cads[:3]

    # Filtro objetivo: para "Ponto sem Portaria", manter apenas pessoas com terceiro possível.
    if analisar_proxy_ponto and col_cad and (col_cad in df_occ.columns):
        valid_proxy_cads = {
            str(cad).strip()
            for cad, info in proxy_map.items()
            if int(info.get("score_total_proxy", 0)) > 0 and int(info.get("qtd_terceiros", 0)) > 0
        }
        if valid_proxy_cads:
            if isinstance(top_pessoas, pd.Series) and (not top_pessoas.empty):
                _idx_tp = top_pessoas.index.astype(str).str.strip()
                top_pessoas = top_pessoas.loc[_idx_tp.isin(valid_proxy_cads)]
            if isinstance(top_pessoas_cad_counts, pd.Series) and (not top_pessoas_cad_counts.empty):
                _idx_tpc = top_pessoas_cad_counts.index.astype(str).str.strip()
                top_pessoas_cad_counts = top_pessoas_cad_counts.loc[_idx_tpc.isin(valid_proxy_cads)]
            if isinstance(top_recent, pd.Series) and (not top_recent.empty):
                _idx_tr = top_recent.index.astype(str).str.strip()
                top_recent = top_recent.loc[_idx_tr.isin(valid_proxy_cads)]
        else:
            top_pessoas = pd.Series(dtype=int)
            top_pessoas_cad_counts = pd.Series(dtype=int)
            top_recent = pd.Series(dtype=int)
            proxy_ranked_cads = []
            proxy_top3_cads = []

    def _fmt_pessoa_rm_nome_area(cad: str) -> str:
        cad = str(cad).strip()
        nome_raw = str(nome_map.get(cad, "")).strip() if nome_map else ""
        if nome_raw:
            parts = [p for p in nome_raw.split() if p]
            if len(parts) == 1:
                nome_abv = parts[0]
            else:
                nome_abv = f"{parts[0]} {parts[-1][0]}."
        else:
            nome_abv = "-"

        area_txt = str(area_map.get(cad, "")).strip() if area_map else ""
        if area_txt == "":
            area_txt = "-"

        planta_top = _fmt_planta_nome(top_planta_cad_map.get(cad, "")) if top_planta_cad_map else ""
        planta_top = str(planta_top).strip()
        if planta_top == "":
            planta_top = "-"

        coletor_top = str(top_coletor_cad_map.get(cad, "")).strip() if top_coletor_cad_map else ""
        if coletor_top == "":
            coletor_top = "-"

        tempo_casa_txt = _format_tempo_casa(cad)
        return (
            f"RM {cad} | {nome_abv} (TC: {tempo_casa_txt}) | {area_txt} | "
            f"Planta top: {planta_top} | {coletor_top_rotulo}: {coletor_top}"
        )

    # =========================
    # PDF
    # =========================
    with PdfPages(pdf_path) as pdf:
        # CAPA (padrão único em tópicos)
        top3_piores_capa = []
        for k, v in top_pessoas.head(3).items():
            pct = (int(v) / total) if total else 0.0
            top3_piores_capa.append(f"{_wrap(_fmt_pessoa_rm_nome_area(str(k)), 60)} — {int(v)} ({pct:.1%})")

        top3_atuais_capa = []
        for k, v in top_recent.head(3).items():
            pct = (int(v) / total_recent) if total_recent else 0.0
            top3_atuais_capa.append(f"{_wrap(_fmt_pessoa_rm_nome_area(str(k)), 60)} — {int(v)} ({pct:.1%} da janela)")

        metricas_capa = [
            ("Ocorrências (pessoa-dia)", f"{total:,}".replace(",", ".")),
            ("Pessoas (cadastro)", f"{n_pessoas:,}".replace(",", ".")),
            ("Dias com ocorrência", f"{n_dias:,}".replace(",", ".")),
        ]
        periodo_capa = (
            f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"
            if pd.notna(dt_min) and pd.notna(dt_max) else None
        )
        concentracoes_capa = [
            ("Concentração Top 1", f"{top1_n:,} ({top1_pct:.1%})".replace(",", ".")),
            ("Concentração Top 3", f"{top3_n:,} ({top3_pct:.1%})".replace(",", ".")),
            ("Concentração Top 10", f"{top10_n:,} ({top10_pct:.1%})".replace(",", ".")),
        ]
        janela_info = (
            f"{total_recent:,} ({(total_recent / total):.1%} do total)".replace(",", ".")
            if total > 0 else "0 (0.0% do total)"
        )

        _render_capa_padrao_pdf(
            pdf=pdf,
            titulo=titulo,
            gen_date=gen_date,
            metricas=metricas_capa,
            periodo=periodo_capa,
            concentracoes=concentracoes_capa,
            top3_piores=top3_piores_capa,
            top3_atuais=top3_atuais_capa,
            janela_info=janela_info,
        )
        _render_metodologia_resumo_pdf(
            pdf=pdf,
            titulo_metodologia=metodologia_titulo,
            passos=metodologia_passos,
            observacoes=metodologia_observacoes,
        )

        # GERAIS
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [2.1, 1.3]})
        if mensal.empty:
            ax1.axis("off")
            ax1.text(0.02, 0.95, "Tendência mensal", fontsize=16, fontweight="bold", va="top")
            ax1.text(0.02, 0.80, "Sem datas válidas para agregação mensal.", fontsize=12, va="top")
        else:
            mensal_pct = (mensal.astype(float) / float(total) * 100.0) if total > 0 else pd.Series(0.0, index=mensal.index)
            ax1.plot(mensal.index, mensal.values, marker="o", linewidth=2.1)
            ax1.fill_between(mensal.index, mensal.values, alpha=0.15)
            ax1.set_title("Tendência mensal (ocorrências pessoa-dia + % do total)")
            ax1.set_ylabel("Ocorrências")
            ax1.grid(alpha=0.25)

            ax1_pct = ax1.twinx()
            ax1_pct.plot(
                mensal.index,
                mensal_pct.values,
                linestyle="--",
                marker=".",
                linewidth=1.4,
                color="#0B5FA5",
                alpha=0.80,
            )
            ax1_pct.set_ylabel("% do total")
            ymax_pct = float(mensal_pct.max()) if len(mensal_pct) else 0.0
            ax1_pct.set_ylim(0.0, max(ymax_pct * 1.20, 5.0))
            ax1_pct.grid(False)

        ax2.set_title("Distribuição por dia da semana (heatmap)")
        dow_total = float(dow.values.sum()) if dow.values.size else 0.0
        if dow_total > 0:
            dow_pct = (dow.astype(float) / dow_total * 100.0)
        else:
            dow_pct = dow.astype(float)

        dow_mat = np.expand_dims(dow_pct.values.astype(float), axis=0)
        im_dow = ax2.imshow(
            dow_mat,
            aspect="auto",
            cmap="Blues",
            vmin=0.0,
            vmax=max(float(dow_pct.max()), 1.0),
        )
        ax2.set_yticks([0])
        ax2.set_yticklabels(["%"])
        ax2.set_xticks(range(len(dow_pct.index)))
        ax2.set_xticklabels(list(dow_pct.index))
        ax2.set_xlabel("Dia da semana")

        for xi, vv in enumerate(dow_pct.values.astype(float)):
            if vv <= 0:
                continue
            ax2.text(xi, 0, f"{vv:.1f}%", ha="center", va="center", fontsize=9)

        cbar_dow = fig.colorbar(im_dow, ax=ax2, fraction=0.03, pad=0.03)
        cbar_dow.set_label("% de ocorrências")
        _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.94), pad=1.05)
        _save_report_page(pdf, fig)

        def _page_top_series(title: str, ser: pd.Series, max_items: int = 12):
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            if ser is None or ser.empty:
                ax.axis("off")
                ax.text(0.02, 0.95, title, fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                _save_report_page(pdf, fig)
                return

            s = ser.head(max_items)
            den = float(ser.sum())
            if den > 0:
                s_pct = (s.astype(float) / den * 100.0)
            else:
                s_pct = s.astype(float)

            y_pos = np.arange(len(s_pct))
            labels = [_wrap(x, 60) for x in s_pct.index.astype(str)]
            vals = s_pct.values.astype(float)

            ax.barh(y_pos, vals, height=0.8)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(labels)
            ax.invert_yaxis()
            ax.set_title(title)
            ax.set_xlabel("% do total (pessoa-dia)")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.22, 1.0)
            ax.set_xlim(0, xmax)
            for i, v in enumerate(vals):
                vv = float(v)
                if vv <= 0:
                    continue
                x = min(vv + xmax * 0.012, xmax * 0.985)
                ha = "left" if x < xmax * 0.96 else "right"
                ax.text(x, i, f"{vv:.1f}%", va="center", ha=ha, fontsize=9)

            _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.94), pad=1.05)
            _save_report_page(pdf, fig)

        def _nome_abreviado(nome: str) -> str:
            nome = " ".join(str(nome).strip().split())
            if not nome:
                return "-"
            parts = [p for p in nome.split(" ") if p]
            if len(parts) == 1:
                return parts[0]
            return f"{parts[0]} {parts[-1][0]}."

        def _cad_label_basico(cad: str) -> str:
            cad = str(cad).strip()
            nome_txt = _nome_abreviado(str(nome_map.get(cad, "")).strip() if nome_map else "")
            area_txt = str(area_map.get(cad, "")).strip() if area_map else ""
            if area_txt == "":
                area_txt = "-"
            tempo_casa_txt = _format_tempo_casa(cad)
            return f"RM {cad} | {nome_txt} (TC: {tempo_casa_txt}) | {area_txt}"

        def _cad_label(cad: str) -> str:
            return _fmt_pessoa_rm_nome_area(cad)

        def _cad_proxy_label(cad: str) -> str:
            return _cad_label_basico(cad)

        def _page_top3_volume_pessoas():
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty:
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — volume de ocorrências", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                _save_report_page(pdf, fig)
                return

            s = top_pessoas_cad_counts.head(3).copy()
            s.index = [_cad_label(c) for c in s.index.astype(str)]

            y_pos = np.arange(len(s))
            vals = s.values.astype(float)

            ax.barh(y_pos, vals, height=0.7)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([_wrap(x, 60) for x in s.index.astype(str)])
            ax.invert_yaxis()
            ax.set_title("Top 3 pessoas — volume de ocorrências")
            ax.set_xlabel("Ocorrências (pessoa-dia)")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.20, 1.0)
            ax.set_xlim(0, xmax)

            for i, v in enumerate(vals):
                pct = (float(v) / total * 100.0) if total else 0.0
                ax.text(min(v + xmax * 0.015, xmax * 0.98), i, f"{int(v)} ({pct:.1f}%)", va="center", fontsize=9)

            _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.94), pad=1.05)
            _save_report_page(pdf, fig)

        def _page_top3_percentual_pessoas():
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)

            if (top_pessoas_cad_counts is None) or top_pessoas_cad_counts.empty or total <= 0:
                ax.axis("off")
                ax.text(0.02, 0.95, "Top 3 pessoas — percentual sobre o total", fontsize=16, fontweight="bold", va="top")
                ax.text(0.02, 0.80, "Sem dados suficientes.", fontsize=12, va="top")
                _save_report_page(pdf, fig)
                return

            s_counts = top_pessoas_cad_counts.head(3).copy()
            s_pct = (s_counts / total * 100.0).round(2)
            s_pct.index = [_cad_label(c) for c in s_pct.index.astype(str)]

            y_pos = np.arange(len(s_pct))
            vals = s_pct.values.astype(float)

            ax.barh(y_pos, vals, height=0.7)
            ax.set_yticks(y_pos)
            ax.set_yticklabels([_wrap(x, 60) for x in s_pct.index.astype(str)])
            ax.invert_yaxis()
            ax.set_title("Top 3 pessoas — percentual sobre o total")
            ax.set_xlabel("% do total de ocorrências")
            ax.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.20, 1.0)
            ax.set_xlim(0, xmax)

            for i, (pct_v, count_v) in enumerate(zip(vals, s_counts.values.astype(int))):
                ax.text(min(pct_v + xmax * 0.015, xmax * 0.98), i, f"{pct_v:.1f}% ({int(count_v)} ocorr.)", va="center", fontsize=9)

            _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.94), pad=1.05)
            _save_report_page(pdf, fig)

        def _page_top3_proxy_pessoas():
            cor_principal = "#0B5FA5"
            cor_contorno = "#001E50"
            proxy_rank_limit = int(globals().get("proxy_rank_limit", 10))
            rank_cads = proxy_ranked_cads[: max(proxy_rank_limit, 1)] if proxy_ranked_cads else proxy_top3_cads[:3]
            rank_size = int(len(rank_cads))

            diag_lines = [
                f"Candidatos avaliados: {int(proxy_diag.get('candidatos_avaliados', 0))}",
                f"Candidatos com eventos elegíveis: {int(proxy_diag.get('candidatos_com_eventos_elegiveis', 0))}",
                f"Eventos elegíveis (candidatos): {int(proxy_diag.get('eventos_elegiveis_candidatos', 0))}",
                f"Pares próximos no tempo (<= {int(janela_proxy_minutos)} min): {int(proxy_diag.get('pares_proximos_tempo', 0))}",
                f"Pares com mesma planta + coletor: {int(proxy_diag.get('pares_com_reforco_planta_ou_familia', 0))}",
                f"Pares aceitos como suspeitos: {int(proxy_diag.get('pares_aceitos_suspeitos', 0))}",
                f"Pessoas com indício: {int(proxy_diag.get('pessoas_com_indicio', 0))}",
                f"Pessoas com recorrência (mesmo terceiro >=2): {int(proxy_diag.get('pessoas_com_recorrencia', 0))}",
            ]

            fig = plt.figure(figsize=(11.69, 8.27))
            gs = fig.add_gridspec(1, 2, width_ratios=[1.10, 1.35], wspace=0.28)
            ax_info = fig.add_subplot(gs[0, 0])
            ax_rank = fig.add_subplot(gs[0, 1])

            ax_info.axis("off")
            fig.suptitle(
                f"Ponto sem Portaria - Ranking de indício de terceiros (Top {rank_size if rank_size > 0 else 3})",
                fontsize=16,
                fontweight="bold",
                y=0.985,
            )

            y = 0.96
            ax_info.text(0.00, y, "Como ler este bloco", fontsize=12.5, fontweight="bold", va="top"); y -= 0.07
            ax_info.text(0.00, y, f"Regra aplicada: <= {int(janela_proxy_minutos)} min, mesma planta, mesmo coletor e terceiro com portaria no dia.", fontsize=10.2, va="top", wrap=True); y -= 0.12
            ax_info.text(0.00, y, "Funil da detecção", fontsize=12.0, fontweight="bold", va="top"); y -= 0.05
            ax_info.text(0.00, y, "\n".join([f"- {x}" for x in diag_lines]), fontsize=9.8, va="top")

            if (not analisar_proxy_ponto) or (not rank_cads):
                ax_rank.axis("off")
                ax_rank.text(0.02, 0.90, "Sem casos suficientes para formar o ranking.", fontsize=12, va="top")
                _save_report_page(pdf, fig)
                return

            data_proxy = []
            for cad in rank_cads:
                info = proxy_map.get(str(cad), {})
                data_proxy.append({
                    "cad": str(cad),
                    "label": _cad_label(str(cad)),
                    "score": int(info.get("score_total_proxy", 0)),
                    "qtd_terceiros": int(info.get("qtd_terceiros", 0)),
                    "max_mesmo_terceiro": int(info.get("max_hits_mesmo_terceiro", 0)),
                })

            df_proxy = pd.DataFrame(data_proxy)
            if df_proxy.empty:
                ax_rank.axis("off")
                ax_rank.text(0.02, 0.90, "Sem dados suficientes para ranking.", fontsize=12, va="top")
                _save_report_page(pdf, fig)
                return

            y_pos = np.arange(len(df_proxy))
            vals = df_proxy["score"].to_numpy(dtype=float)

            ax_rank.barh(y_pos, vals, height=0.62, color=cor_principal, edgecolor=cor_contorno, linewidth=1.0)
            ax_rank.set_yticks(y_pos)
            ax_rank.set_yticklabels([_wrap(x, 52) for x in df_proxy["label"].astype(str)])
            ax_rank.invert_yaxis()
            ax_rank.set_title(f"Top {int(len(df_proxy))} por dias suspeitos")
            ax_rank.set_xlabel("Dias com indício de terceiro (pessoa-dia)")
            ax_rank.grid(axis="x", alpha=0.25)

            xmax = max(float(vals.max()) * 1.22, 1.0)
            ax_rank.set_xlim(0, xmax)

            for i, r in enumerate(df_proxy.itertuples(index=False)):
                txt = f"{int(r.score)} dias suspeitos | {int(r.qtd_terceiros)} terceiros | máx. dias mesmo terceiro {int(r.max_mesmo_terceiro)}"
                ax_rank.text(min(float(r.score) + xmax * 0.015, xmax * 0.98), i, txt, va="center", fontsize=9.0)

            _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.95), pad=1.05)
            _save_report_page(pdf, fig)

        def _page_proxy_individual_top3():
            if (not analisar_proxy_ponto) or (not proxy_top3_cads):
                return

            cor_principal = "#0B5FA5"
            cor_contorno = "#001E50"
            total_proxy_days = int(
                sum(int(info.get("score_total_proxy", 0)) for info in proxy_map.values())
            )
            periodo_txt = (
                f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"
                if pd.notna(dt_min) and pd.notna(dt_max) else "-"
            )

            cnt_pessoas = top_pessoas_cad_counts.copy() if isinstance(top_pessoas_cad_counts, pd.Series) else pd.Series(dtype=int)
            if not cnt_pessoas.empty:
                cnt_pessoas.index = cnt_pessoas.index.astype(str)

            for pos, cad in enumerate(proxy_top3_cads[:3], start=1):
                cad_s = str(cad).strip()
                info = proxy_map.get(cad_s, {})
                score_total = int(info.get("score_total_proxy", 0))
                qtd_terceiros = int(info.get("qtd_terceiros", 0))
                max_mesmo = int(info.get("max_hits_mesmo_terceiro", 0))
                total_occ_pessoa = int(cnt_pessoas.get(cad_s, 0)) if not cnt_pessoas.empty else 0
                pct_proxy = (score_total / total_proxy_days) if total_proxy_days else 0.0

                top_other = list(info.get("top_other", []))[:6]

                fig = plt.figure(figsize=(11.69, 8.27))
                gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.35], wspace=0.30)
                ax_info = fig.add_subplot(gs[0, 0])
                ax_bar = fig.add_subplot(gs[0, 1])
                ax_info.axis("off")

                fig.suptitle(
                    f"Ponto sem Portaria - Caso individual {pos}º | {_wrap(_cad_label(cad_s), 66)}",
                    fontsize=14.4,
                    fontweight="bold",
                    y=0.975,
                )

                y = 0.95
                y = _draw_wrapped_text(ax_info, 0.00, y, "Resumo rápido", width=34, fontsize=12.8, fontweight="bold", line_step=0.060, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Período analisado: {periodo_txt}", width=38, fontsize=10.4, line_step=0.050, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Volume suspeito (dias com ocorrência): {score_total}", width=40, fontsize=10.8, line_step=0.048, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Participação no funil: {pct_proxy:.1%}", width=38, fontsize=10.8, line_step=0.048, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Terceiros distintos: {qtd_terceiros}", width=38, fontsize=10.8, line_step=0.048, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Máx. dias com o mesmo terceiro: {max_mesmo}", width=38, fontsize=10.8, line_step=0.048, max_lines=2)
                y = _draw_wrapped_text(ax_info, 0.00, y, f"Ocorrências totais da pessoa (pessoa-dia): {total_occ_pessoa}", width=38, fontsize=10.4, line_step=0.050, max_lines=2)
                y -= 0.015
                y = _draw_wrapped_text(ax_info, 0.00, y, "Critério de detecção", width=34, fontsize=11.6, fontweight="bold", line_step=0.050, max_lines=2)
                _draw_wrapped_text(
                    ax_info,
                    0.00,
                    y,
                    f"<= {int(janela_proxy_minutos)} min | mesma planta | mesmo coletor | terceiro com portaria no dia",
                    width=44,
                    fontsize=9.6,
                    line_step=0.042,
                    max_lines=4,
                )

                if not top_other:
                    ax_bar.axis("off")
                    ax_bar.text(0.02, 0.92, "Terceiros relacionados", fontsize=12.2, fontweight="bold", va="top")
                    ax_bar.text(0.02, 0.78, "Sem correspondências suficientes para detalhar.", fontsize=10.6, va="top")
                else:
                    labels = [_wrap(_cad_proxy_label(k), 50) for k, _v in top_other]
                    vals = np.array([int(v) for _k, v in top_other], dtype=float)
                    y_pos = np.arange(len(vals))

                    ax_bar.barh(y_pos, vals, color=cor_principal, edgecolor=cor_contorno, linewidth=1.0)
                    ax_bar.set_yticks(y_pos)
                    ax_bar.set_yticklabels(labels)
                    ax_bar.invert_yaxis()
                    ax_bar.set_title("Top terceiros relacionados (Top 6)")
                    ax_bar.set_xlabel("Dias com indício de participação")
                    ax_bar.grid(axis="x", alpha=0.25)

                    xmax = max(float(vals.max()) * 1.22, 1.0)
                    ax_bar.set_xlim(0, xmax)
                    for i, v in enumerate(vals):
                        pct_v = (float(v) / score_total) if score_total else 0.0
                        ax_bar.text(min(float(v) + xmax * 0.015, xmax * 0.98), i, f"{int(v)} ({pct_v:.1%})", va="center", fontsize=9.0)

                _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.94), pad=1.05)
                _save_report_page(pdf, fig)

        def _page_proxy_resumo_final_top3():
            if not analisar_proxy_ponto:
                return

            def _render_resumo_proxy_page(
                rank_cads: list[str],
                titulo_page: str,
                subtitulo_page: str,
                janela_page: str | None = None,
                ocorr_map: dict[str, int] | None = None,
                ocorr_total: int = 0,
            ) -> None:
                if not rank_cads:
                    return

                fig = plt.figure(figsize=(11.69, 8.27))
                ax = fig.add_subplot(111)
                ax.axis("off")

                ax.text(0.02, 0.95, titulo_page, fontsize=16.0, fontweight="bold", va="top")
                _draw_wrapped_text(ax, 0.02, 0.89, subtitulo_page, width=116, fontsize=10.4, line_step=0.030, max_lines=2)
                if janela_page:
                    _draw_wrapped_text(ax, 0.02, 0.845, f"Janela considerada: {janela_page}", width=116, fontsize=10.0, line_step=0.028, max_lines=2)

                y = 0.80
                for pos, cad_s in enumerate(rank_cads[:3], start=1):
                    info = proxy_map.get(cad_s, {})
                    suspeito_label = _cad_label(cad_s)
                    score_total = int(info.get("score_total_proxy", 0))
                    qtd_terceiros = int(info.get("qtd_terceiros", 0))
                    max_mesmo = int(info.get("max_hits_mesmo_terceiro", 0))
                    top_other = list(info.get("top_other", []))[:5]

                    extra_occ = ""
                    if isinstance(ocorr_map, dict):
                        occ_i = int(ocorr_map.get(cad_s, 0))
                        pct_i = (occ_i / ocorr_total) if ocorr_total else 0.0
                        extra_occ = f" | Dias com ocorrência na janela: {occ_i} ({pct_i:.1%})"

                    y = _draw_wrapped_text(
                        ax,
                        0.02,
                        y,
                        f"{pos}. Pessoa mais suspeita: {suspeito_label}",
                        width=108,
                        fontsize=11.2,
                        fontweight="bold",
                        line_step=0.034,
                        max_lines=3,
                    )
                    y = _draw_wrapped_text(
                        ax,
                        0.04,
                        y,
                        f"Volume suspeito (dias com ocorrência): {score_total} | Terceiros distintos: {qtd_terceiros} | Máx. dias com o mesmo terceiro: {max_mesmo}{extra_occ}",
                        width=108,
                        fontsize=9.8,
                        line_step=0.031,
                        max_lines=3,
                    )

                    if top_other:
                        y = _draw_wrapped_text(
                            ax,
                            0.04,
                            y,
                            "Quem pode estar batendo para a pessoa:",
                            width=108,
                            fontsize=9.6,
                            line_step=0.029,
                            max_lines=2,
                        )
                        for cad_other, dias_other in top_other:
                            y = _draw_wrapped_text(
                                ax,
                                0.06,
                                y,
                                f"- {_cad_proxy_label(str(cad_other))} ({int(dias_other)} dia(s))",
                                width=104,
                                fontsize=9.2,
                                line_step=0.028,
                                max_lines=2,
                            )
                    else:
                        y = _draw_wrapped_text(
                            ax,
                            0.04,
                            y,
                            f"Quem pode estar batendo para a pessoa: Sem terceiro identificado com a regra de {int(janela_proxy_minutos)} minuto(s).",
                            width=108,
                            fontsize=9.6,
                            line_step=0.030,
                            max_lines=3,
                        )

                    y -= 0.022
                    if y < 0.12:
                        break

                _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.95), pad=1.03)
                _save_report_page(pdf, fig)

            rank_final_cads: list[str] = []
            if proxy_top3_cads:
                rank_final_cads = [str(c).strip() for c in proxy_top3_cads[:3]]
            elif isinstance(top_pessoas_cad_counts, pd.Series) and (not top_pessoas_cad_counts.empty):
                rank_final_cads = [str(c).strip() for c in top_pessoas_cad_counts.head(3).index.astype(str).tolist()]

            _render_resumo_proxy_page(
                rank_cads=rank_final_cads,
                titulo_page="Ponto sem Portaria - Resumo final (Top 3 suspeitos de terceiro)",
                subtitulo_page=f"Regra: <= {int(janela_proxy_minutos)} min, mesma planta, mesmo coletor e terceiro com portaria no dia.",
            )

            # Página extra: resumo final apenas dos últimos 30 dias.
            rank_30_cads: list[str] = []
            ocorr_30_map: dict[str, int] = {}
            ocorr_30_total = 0
            janela_30_txt = None

            if col_cad and (col_cad in df_occ.columns) and ("_dia_operacional" in df_occ.columns):
                occ30 = df_occ.loc[:, [col_cad, "_dia_operacional"]].copy()
                occ30[col_cad] = occ30[col_cad].astype(str).str.strip()
                occ30["_dia_operacional"] = pd.to_datetime(occ30["_dia_operacional"], errors="coerce").dt.normalize()
                occ30 = occ30.loc[occ30[col_cad].ne("") & occ30["_dia_operacional"].notna()].copy()

                if not occ30.empty:
                    dt_end_30 = pd.to_datetime(occ30["_dia_operacional"].max()).normalize()
                    dt_start_30 = dt_end_30 - pd.Timedelta(days=29)
                    occ30 = occ30.loc[
                        (occ30["_dia_operacional"] >= dt_start_30)
                        & (occ30["_dia_operacional"] <= dt_end_30)
                    ].copy()
                    if not occ30.empty:
                        cad_counts_30 = occ30[col_cad].value_counts()
                        if isinstance(top_pessoas_cad_counts, pd.Series) and (not top_pessoas_cad_counts.empty):
                            eligible_idx = top_pessoas_cad_counts.index.astype(str)
                            cad_counts_30 = cad_counts_30.loc[cad_counts_30.index.isin(eligible_idx)]

                        if not cad_counts_30.empty:
                            rank_30_cads = list(cad_counts_30.head(3).index.astype(str))
                            ocorr_30_map = {str(k): int(v) for k, v in cad_counts_30.items()}
                            ocorr_30_total = int(cad_counts_30.sum())
                            janela_30_txt = f"{pd.to_datetime(dt_start_30).date()} -> {pd.to_datetime(dt_end_30).date()}"

            _render_resumo_proxy_page(
                rank_cads=rank_30_cads,
                titulo_page="Ponto sem Portaria - Resumo final (Últimos 30 dias | Top 3 suspeitos de terceiro)",
                subtitulo_page=f"Regra: <= {int(janela_proxy_minutos)} min, mesma planta, mesmo coletor e terceiro com portaria no dia.",
                janela_page=janela_30_txt,
                ocorr_map=ocorr_30_map,
                ocorr_total=ocorr_30_total,
            )

        _page_top_series("Top áreas (representatividade por pessoa-dia)", top_areas)
        _page_top_series("Top plantas (representatividade por pessoa-dia) — exclui planta 999", top_plantas)
        _page_top_series("Top área | planta (representatividade por pessoa-dia) — exclui planta 999", top_combo)

        _page_top3_volume_pessoas()
        _page_top3_percentual_pessoas()
        if analisar_proxy_ponto:
            _page_top3_proxy_pessoas()
            _page_proxy_individual_top3()

        # TOP 3 por volume (analise individual)
        # - Demais irregularidades: período total + últimos 30 dias (até 6 páginas)
        # - 999 como trajeto: somente período total (3 casos individuais)
        if col_cad and (col_cad in df_occ.columns) and (top_pessoas_cad_counts is not None) and (not top_pessoas_cad_counts.empty):
            cad_irreg_set = set(df_occ[col_cad].astype(str).str.strip().unique())

            dt_end = pd.to_datetime(dt_max).normalize() if pd.notna(dt_max) else pd.Timestamp.today().normalize()
            dt_start_30 = dt_end - pd.Timedelta(days=29)
            idx_30 = pd.date_range(dt_start_30, dt_end, freq="D")

            df_occ_ctx_total = df_occ.loc[:, [col_cad, "_dia_operacional"]].copy()
            df_occ_ctx_total[col_cad] = df_occ_ctx_total[col_cad].astype(str).str.strip()
            df_occ_ctx_total = df_occ_ctx_total.loc[df_occ_ctx_total[col_cad].ne("")].copy()

            df_occ_ctx_30 = df_occ_ctx_total.loc[
                (df_occ_ctx_total["_dia_operacional"] >= dt_start_30) &
                (df_occ_ctx_total["_dia_operacional"] <= dt_end)
            ].copy()

            cad_counts_30 = df_occ_ctx_30[col_cad].value_counts()
            if (not sem_corte_30):
                eligible_idx = top_pessoas_cad_counts.index.astype(str)
                cad_counts_30 = cad_counts_30.loc[cad_counts_30.index.isin(eligible_idx)]

            top3_total_cads = list(top_pessoas_cad_counts.head(3).index.astype(str))
            top3_30_cads = list(cad_counts_30.head(3).index.astype(str))

            if analisar_999_trajeto:
                ctx_specs = [
                    ("Período total", top3_total_cads, df_occ_ctx_total, f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"),
                ]
            else:
                ctx_specs = [
                    ("Período total", top3_total_cads, df_occ_ctx_total, f"{pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}"),
                    ("Últimos 30 dias", top3_30_cads, df_occ_ctx_30, f"{pd.to_datetime(dt_start_30).date()} → {pd.to_datetime(dt_end).date()}"),
                ]

            for ctx_nome, cads_ctx, df_ctx, janela_ctx_txt in ctx_specs:
                if not cads_ctx:
                    continue

                cad_counts_ctx = df_ctx[col_cad].value_counts()
                top10_cads_ctx = list(cad_counts_ctx.head(10).index.astype(str))
                top10_total_ctx = int(cad_counts_ctx.loc[top10_cads_ctx].sum()) if top10_cads_ctx else 0
                total_ctx = int(len(df_ctx))

                for pos_ctx, cad in enumerate(cads_ctx, start=1):
                    occ_p = df_ctx.loc[df_ctx[col_cad].eq(str(cad)), ["_dia_operacional"]]
                    if occ_p.empty:
                        continue

                    total_p = int(len(occ_p))
                    pct_p_total = (total_p / total_ctx) if total_ctx else 0.0
                    label = _cad_label(str(cad))

                    days = occ_p["_dia_operacional"].dt.normalize()
                    week_start = (days - pd.to_timedelta(days.dt.dayofweek, unit="D")).dt.normalize()
                    weekly_days = week_start.value_counts().sort_index()
                    mensal_semana = (
                        weekly_days.groupby(weekly_days.index.to_period("M").to_timestamp(), sort=False)
                                   .mean()
                                   .sort_index()
                        if not weekly_days.empty else pd.Series(dtype=float)
                    )

                    # Contagem diária de ocorrência (pessoa-dia), nunca por número de linhas.
                    daily_30 = (
                        occ_p.groupby(occ_p["_dia_operacional"].dt.normalize(), sort=False)
                             .size()
                             .reindex(idx_30, fill_value=0)
                    )

                    pct_vs_top10 = (total_p / top10_total_ctx) if top10_total_ctx else 0.0

                    area_val = None
                    her_val = None
                    if isinstance(dim_map, pd.DataFrame) and (not dim_map.empty) and ("cadastro" in dim_map.columns):
                        row_dim = dim_map.loc[dim_map["cadastro"].eq(str(cad))]
                        if not row_dim.empty:
                            area_val = row_dim["_area_norm"].iloc[0] or None
                            her_val = row_dim["_her_norm"].iloc[0] or None

                    grp_total = grp_with = grp_without = None
                    if (area_val is not None) and (her_val is not None) and isinstance(dim_map, pd.DataFrame) and (not dim_map.empty):
                        grp_cads = dim_map.loc[
                            dim_map["_area_norm"].eq(str(area_val)) & dim_map["_her_norm"].eq(str(her_val)),
                            "cadastro"
                        ].to_numpy()
                        grp_total = int(len(grp_cads))
                        if grp_total > 0:
                            grp_with = int(np.isin(grp_cads, list(cad_irreg_set)).sum())
                            grp_without = int(grp_total - grp_with)

                    other_days = int(max(total_ctx - total_p, 0))

                    fig = plt.figure(figsize=(11.69, 8.27))
                    gs = fig.add_gridspec(
                        2,
                        3,
                        width_ratios=[1.00, 1.00, 1.32],
                        height_ratios=[1.00, 1.14],
                        wspace=0.34,
                        hspace=0.54,
                    )

                    ax1 = fig.add_subplot(gs[0, 0])
                    ax2 = fig.add_subplot(gs[0, 1])
                    ax3 = fig.add_subplot(gs[0, 2])
                    ax4 = fig.add_subplot(gs[1, 0])
                    ax5 = fig.add_subplot(gs[1, 1])
                    ax6 = fig.add_subplot(gs[1, 2])
                    ax6.axis("off")

                    if analisar_999_trajeto and (ctx_nome == "Período total"):
                        titulo_individual = (
                            f"999 como trajeto  Caso individual {pos_ctx}º (período total) | "
                            f"{_wrap(label, 70)} | Total (pessoa-dia): {total_p} ({pct_p_total:.1%} do contexto)"
                        )
                    else:
                        titulo_individual = (
                            f"{ctx_nome} | Top 3 ({pos_ctx}º) — {_wrap(label, 70)} | "
                            f"Total (pessoa-dia): {total_p} ({pct_p_total:.1%} do contexto)"
                        )

                    fig.suptitle(
                        titulo_individual,
                        fontsize=14.2,
                        fontweight="bold",
                        y=0.985,
                    )

                    if mensal_semana.empty:
                        ax1.axis("off")
                        ax1.text(0.02, 0.92, "Dias com ocorrência na semana (mensal)", fontsize=11.5, fontweight="bold", va="top")
                        ax1.text(0.02, 0.78, "Sem dados suficientes.", fontsize=10, va="top")
                    else:
                        ax1.plot(mensal_semana.index, mensal_semana.values, marker="o", linewidth=2.0)
                        ax1.fill_between(mensal_semana.index, mensal_semana.values, alpha=0.12)
                        ax1.set_title("Dias com ocorrência na semana\n(média semanal no mês)", fontsize=11.5)
                        ax1.set_xlabel("Mês")
                        ax1.set_ylabel("Dias/semana")
                        ax1.grid(alpha=0.25)
                        ax1.tick_params(axis="x", rotation=35)
                        ax1.set_ylim(0, 7)

                    if daily_30.empty:
                        ax2.axis("off")
                        ax2.text(0.02, 0.92, "Últimos 30 dias (ocorrências/dia)", fontsize=11.5, fontweight="bold", va="top")
                        ax2.text(0.02, 0.78, "Sem dados suficientes.", fontsize=10, va="top")
                    else:
                        ax2.plot(daily_30.index, daily_30.values, marker="o", linewidth=1.7)
                        ax2.fill_between(daily_30.index, daily_30.values, alpha=0.12)
                        ax2.set_title("Últimos 30 dias\n(ocorrências por dia)", fontsize=11.5)
                        ax2.set_xlabel("Dia")
                        ax2.set_ylabel("Ocorrências")
                        ax2.grid(alpha=0.25)
                        ax2.tick_params(axis="x", rotation=35)

                    ax3.set_title("% de ocorrências vs Top 10 do contexto", fontsize=11.5)
                    ax3.barh(["Pessoa"], [pct_vs_top10], height=0.55)
                    ax3.set_xlim(0, 1)
                    ax3.set_xlabel("% do Top 10 do contexto")
                    ax3.grid(axis="x", alpha=0.25)
                    ax3.text(
                        min(pct_vs_top10 + 0.02, 0.97), 0,
                        f"{pct_vs_top10:.1%}",
                        va="center",
                        ha="left" if pct_vs_top10 < 0.9 else "right",
                        fontsize=10,
                    )

                    _pie_with_legend(
                        ax4,
                        [grp_with if grp_with is not None else 0, grp_without if grp_without is not None else 0],
                        [
                            f"Com irregularidade ({0 if grp_with is None else grp_with})",
                            f"Sem irregularidade ({0 if grp_without is None else grp_without})",
                        ],
                        "Área + herança:\nmesmo grupo da pessoa",
                        f"Base: {0 if grp_total is None else grp_total} pessoas",
                    )

                    _pie_with_legend(
                        ax5,
                        [total_p, other_days],
                        [f"Pessoa ({total_p})", f"Demais ({other_days})"],
                        "Dias com ocorrência:\npessoa vs demais",
                        f"Base: {total_ctx} ocorrências pessoa-dia",
                    )

                    periodo_analisado_txt = janela_ctx_txt
                    periodo_desligado_txt = periodo_desligado_map.get(str(cad), "-") if mostrar_periodo_desligado else None

                    total_horas_acumuladas_999 = None
                    if analisar_999_trajeto and {"data", "hora", "dir", "planta"}.issubset(df_irregularidade.columns):
                        try:
                            mask_cad_raw = df_irregularidade[col_cad].astype(str).str.strip().eq(str(cad))
                            if mask_cad_raw.any():
                                raw_cols = [col_cad, "data", "hora", "dir", "planta"]
                                if "_dt_ronda" in df_irregularidade.columns:
                                    raw_cols.append("_dt_ronda")
                                if col_desc_coletor and (col_desc_coletor in df_irregularidade.columns):
                                    raw_cols.append(col_desc_coletor)

                                raw_cad = df_irregularidade.loc[mask_cad_raw, raw_cols].copy(deep=False)
                                dt_raw = pd.to_datetime(_build_dt_from_data_hora(raw_cad), errors="coerce")
                                dia_raw = (dt_raw - pd.Timedelta(hours=int(globals().get("hora_corte_operacional", 11)))).dt.normalize()

                                tmp_h = pd.DataFrame({
                                    "_dia_operacional": pd.to_datetime(dia_raw, errors="coerce"),
                                    "_dt": pd.to_datetime(dt_raw, errors="coerce"),
                                    "dir": raw_cad["dir"],
                                    "planta": raw_cad["planta"],
                                })

                                if col_desc_coletor and (col_desc_coletor in raw_cad.columns):
                                    desc_h = raw_cad[col_desc_coletor].fillna("").astype(str)
                                else:
                                    desc_h = pd.Series("", index=raw_cad.index)

                                re_port_h = globals().get("regex_portaria", r"PORTARIA|PORT")
                                tmp_h["_is_port"] = desc_h.str.contains(re_port_h, case=False, na=False, regex=True)

                                dir_norm_h = tmp_h["dir"].fillna("").astype(str).str.strip().str.upper()
                                dir_norm_h = dir_norm_h.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
                                tmp_h["_is_E"] = dir_norm_h.str.startswith("E")
                                tmp_h["_is_S"] = dir_norm_h.str.startswith("S")
                                tmp_h["_is_999"] = pd.to_numeric(tmp_h["planta"], errors="coerce").eq(999)

                                tmp_h = tmp_h.dropna(subset=["_dia_operacional", "_dt"])

                                if not tmp_h.empty:
                                    dias_ctx = pd.to_datetime(occ_p["_dia_operacional"], errors="coerce").dt.normalize().dropna().unique()
                                    if len(dias_ctx) > 0:
                                        tmp_h = tmp_h.loc[tmp_h["_dia_operacional"].isin(dias_ctx)]

                                if not tmp_h.empty:
                                    port_h = tmp_h.loc[tmp_h["_is_port"] & (tmp_h["_is_E"] | tmp_h["_is_S"])].copy()
                                    ev_999_h = tmp_h.loc[tmp_h["_is_999"]].copy()

                                    if (not port_h.empty) and (not ev_999_h.empty):
                                        ent_h = (
                                            port_h.loc[port_h["_is_E"]]
                                                  .groupby("_dia_operacional", sort=False)["_dt"].min()
                                                  .rename("t_ent")
                                                  .reset_index()
                                        )
                                        sai_h = (
                                            port_h.loc[port_h["_is_S"]]
                                                  .groupby("_dia_operacional", sort=False)["_dt"].max()
                                                  .rename("t_sai")
                                                  .reset_index()
                                        )
                                        pe_h = ent_h.merge(sai_h, on="_dia_operacional", how="inner")

                                        if not pe_h.empty:
                                            ev_h = ev_999_h.merge(pe_h, on="_dia_operacional", how="inner")
                                            if not ev_h.empty:
                                                before_h = (
                                                    ev_h.loc[ev_h["_dt"] < ev_h["t_ent"]]
                                                        .groupby("_dia_operacional", sort=False)["_dt"].min()
                                                        .rename("t_999_before")
                                                        .reset_index()
                                                )
                                                after_h = (
                                                    ev_h.loc[ev_h["_dt"] > ev_h["t_sai"]]
                                                        .groupby("_dia_operacional", sort=False)["_dt"].max()
                                                        .rename("t_999_after")
                                                        .reset_index()
                                                )

                                                traj_h = (
                                                    pe_h.merge(before_h, on="_dia_operacional", how="left")
                                                        .merge(after_h, on="_dia_operacional", how="left")
                                                )

                                                traj_h["h_before"] = (
                                                    (traj_h["t_ent"] - traj_h["t_999_before"]) / pd.Timedelta(hours=1)
                                                ).where(traj_h["t_999_before"].notna(), 0.0)
                                                traj_h["h_after"] = (
                                                    (traj_h["t_999_after"] - traj_h["t_sai"]) / pd.Timedelta(hours=1)
                                                ).where(traj_h["t_999_after"].notna(), 0.0)

                                                traj_h["horas_trajeto"] = (
                                                    traj_h["h_before"].fillna(0.0).clip(lower=0.0)
                                                    + traj_h["h_after"].fillna(0.0).clip(lower=0.0)
                                                )
                                                total_horas_acumuladas_999 = float(traj_h["horas_trajeto"].sum())
                        except Exception:
                            total_horas_acumuladas_999 = None

                    y6 = 0.965
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, "Resumo", width=52, fontsize=13.0, fontweight="bold", line_step=0.056, max_lines=0)
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Pessoa (RM | Nome abreviado | Área): {_cad_label(str(cad))}", width=56, fontsize=9.9, line_step=0.044, max_lines=0)
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Participação no contexto: {pct_p_total:.1%}", width=56, fontsize=10.0, line_step=0.042, max_lines=0)
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Participação no Top 10 (contexto): {pct_vs_top10:.1%}", width=56, fontsize=10.0, line_step=0.042, max_lines=0)
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Herança: {her_val if her_val else '-'}", width=56, fontsize=10.0, line_step=0.042, max_lines=0)
                    y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Período do contexto: {periodo_analisado_txt}", width=56, fontsize=9.9, line_step=0.042, max_lines=0)

                    if analisar_999_trajeto:
                        if (total_horas_acumuladas_999 is not None) and np.isfinite(total_horas_acumuladas_999):
                            horas_fmt = f"{total_horas_acumuladas_999:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")
                            y6 = _draw_wrapped_text(
                                ax6,
                                0.00,
                                y6,
                                f"Horas acumuladas no período de trajeto aproveitado: {horas_fmt} h",
                                width=56,
                                fontsize=10.0,
                                line_step=0.042,
                                max_lines=0,
                            )
                        else:
                            y6 = _draw_wrapped_text(
                                ax6,
                                0.00,
                                y6,
                                "Horas acumuladas no período de trajeto aproveitado: indisponível",
                                width=56,
                                fontsize=9.9,
                                line_step=0.042,
                                max_lines=0,
                            )

                    if mostrar_periodo_desligado:
                        y6 = _draw_wrapped_text(ax6, 0.00, y6, f"Período já desligado: {periodo_desligado_txt}", width=56, fontsize=9.7, line_step=0.041, max_lines=0)

                    if analisar_proxy_ponto:
                        info = proxy_map.get(str(cad))
                        y6 -= 0.01
                        y6 = _draw_wrapped_text(
                            ax6,
                            0.00,
                            y6,
                            f"Possíveis terceiros com portaria no dia (<= {int(janela_proxy_minutos)} min; mesma planta; mesmo nº coletor; chapa | nome abreviado | área)",
                            width=54,
                            fontsize=9.1,
                            fontweight="bold",
                            line_step=0.037,
                            max_lines=0,
                        )
                        if (not info) or (not info.get("top_other")):
                            _draw_wrapped_text(ax6, 0.00, y6, "Sem correspondências suficientes.", width=54, fontsize=8.8, line_step=0.034, max_lines=0)
                        else:
                            for k, v in list(info["top_other"])[:5]:
                                y6 = _draw_wrapped_text(
                                    ax6,
                                    0.00,
                                    y6,
                                    f"- {_cad_proxy_label(k)} ({int(v)} dia(s))",
                                    width=56,
                                    fontsize=8.6,
                                    line_step=0.032,
                                    max_lines=0,
                                )

                    _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.955), pad=1.20)
                    _save_report_page(pdf, fig)

        if analisar_proxy_ponto:
            _page_proxy_resumo_final_top3()
        # Ponto sem Portaria — heatmaps finais das análises individuais (ranking)
        # Desativado por padrão para não gerar essas páginas no PDF.
        gerar_heatmaps_ponto_sem_portaria = bool(globals().get("gerar_heatmaps_ponto_sem_portaria", False))
        if gerar_heatmaps_ponto_sem_portaria and analisar_proxy_ponto and col_cad and (col_cad in df_occ.columns) and ("data" in df_irregularidade.columns) and ("hora" in df_irregularidade.columns):
            try:
                from matplotlib.colors import LinearSegmentedColormap as _LSC

                desc_col_hm = _pick_col_local(df_irregularidade, ["descricao coletor", "descricao_coletor", "desc_coletor", "coletor", "descricao"])
                cols_hm = [col_cad, "data", "hora"] + ([desc_col_hm] if desc_col_hm else [])
                dph = df_irregularidade.loc[:, [c for c in cols_hm if c in df_irregularidade.columns]].copy(deep=False)
                dph[col_cad] = dph[col_cad].astype(str).str.strip()
                dph = dph.loc[dph[col_cad].ne("")].copy()

                data_dt_hm = pd.to_datetime(dph["data"], dayfirst=True, errors="coerce")
                hora_s_hm = dph["hora"].fillna("00:00").astype(str).str.strip()
                hhmm_mask_hm = hora_s_hm.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s_hm = hora_s_hm.where(~hhmm_mask_hm, hora_s_hm + ":00")
                hora_td_hm = pd.to_timedelta(hora_s_hm.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"}), errors="coerce")
                dph["_dt"] = data_dt_hm.dt.normalize() + hora_td_hm
                hora_corte_hm = int(globals().get("hora_corte_operacional", 11))
                dph["_dia_key"] = (dph["_dt"] - pd.Timedelta(hours=hora_corte_hm)).dt.normalize()

                if desc_col_hm:
                    desc_hm = dph[desc_col_hm].fillna("").astype(str)
                else:
                    desc_hm = pd.Series("", index=dph.index)

                re_ponto_hm = globals().get("regex_ponto", r"PONTO")
                re_port_hm = globals().get("regex_portaria", r"PORTARIA|PORT")
                mask_ponto_hm = (
                    desc_hm.str.contains(re_ponto_hm, case=False, na=False, regex=True)
                    & (~desc_hm.str.contains(re_port_hm, case=False, na=False, regex=True))
                )

                dph = dph.loc[mask_ponto_hm & dph["_dt"].notna()].copy()
                if not dph.empty:
                    dph = (
                        dph.sort_values("_dt", kind="mergesort")
                           .drop_duplicates(subset=[col_cad, "_dia_key"], keep="first")
                           .copy()
                    )
                    dph["_dow"] = dph["_dia_key"].dt.dayofweek.astype(int)
                    dph["_hour"] = dph["_dt"].dt.hour.astype(int)

                    rank_occ = df_occ.loc[:, [col_cad, "_dia_operacional"]].copy()
                    rank_occ[col_cad] = rank_occ[col_cad].astype(str).str.strip()
                    rank_occ = rank_occ.loc[rank_occ[col_cad].ne("")].copy()
                    if valid_proxy_cads:
                        rank_occ = rank_occ.loc[rank_occ[col_cad].isin(valid_proxy_cads)].copy()

                    dt_end_hm = pd.to_datetime(dt_max).normalize() if pd.notna(dt_max) else pd.Timestamp.today().normalize()
                    dt_start_hm = dt_end_hm - pd.Timedelta(days=29)

                    top3_total_hm = list(rank_occ[col_cad].value_counts().head(3).index.astype(str))
                    rank_occ_30 = rank_occ.loc[
                        (rank_occ["_dia_operacional"] >= dt_start_hm) &
                        (rank_occ["_dia_operacional"] <= dt_end_hm)
                    ].copy()
                    top3_30_hm = list(rank_occ_30[col_cad].value_counts().head(3).index.astype(str))

                    cmap_hm = _LSC.from_list("vw_blue_hm", ["#F6F8FB", "#CFE2F3", "#0B5FA5", "#001E50"])
                    dow_labels_hm = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

                    def _render_heatmap_ponto_individual(cad_id: str, pos_txt: str, scope_txt: str, dt_ini_scope: pd.Timestamp, dt_fim_scope: pd.Timestamp):
                        dpc = dph.loc[
                            dph[col_cad].eq(str(cad_id))
                            & (dph["_dia_key"] >= dt_ini_scope)
                            & (dph["_dia_key"] <= dt_fim_scope)
                        ].copy()

                        fig = plt.figure(figsize=(11.69, 8.27))
                        gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.35], wspace=0.30)
                        ax_info = fig.add_subplot(gs[0, 0])
                        ax_hm = fig.add_subplot(gs[0, 1])
                        ax_info.axis("off")

                        label_p = _cad_label_basico(str(cad_id))
                        fig.suptitle(
                            f"Ponto sem Portaria — Heatmap individual ({scope_txt} | {pos_txt}) | {_wrap(label_p, 74)}",
                            fontsize=13.8,
                            fontweight="bold",
                            y=0.985,
                        )

                        if dpc.empty:
                            ax_hm.axis("off")
                            ax_hm.text(0.02, 0.90, "Sem dados para o recorte deste ranking.", fontsize=11.2, va="top")
                            y_info = 0.95
                            y_info = _draw_wrapped_text(ax_info, 0.00, y_info, "Resumo", width=36, fontsize=12.5, fontweight="bold", line_step=0.060, max_lines=2)
                            y_info = _draw_wrapped_text(
                                ax_info,
                                0.00,
                                y_info,
                                f"Período: {pd.to_datetime(dt_ini_scope).date()} → {pd.to_datetime(dt_fim_scope).date()}",
                                width=42,
                                fontsize=10.2,
                                line_step=0.050,
                                max_lines=2,
                            )
                            _draw_wrapped_text(
                                ax_info,
                                0.00,
                                y_info,
                                "Critério: ocorrência-dia por pessoa",
                                width=42,
                                fontsize=10.2,
                                line_step=0.050,
                                max_lines=2,
                            )
                            _save_report_page(pdf, fig)
                            return

                        mat = (
                            dpc.pivot_table(index="_dow", columns="_hour", values="_dt", aggfunc="count")
                               .reindex(index=range(7), columns=range(24), fill_value=0)
                               .fillna(0)
                               .astype(int)
                        )

                        vmax = int(mat.to_numpy().max()) if mat.size else 0
                        im = ax_hm.imshow(mat.values, aspect="auto", cmap=cmap_hm, vmin=0, vmax=max(vmax, 1))
                        ax_hm.set_xticks(range(0, 24, 2))
                        ax_hm.set_xticklabels([f"{h:02d}h" for h in range(0, 24, 2)])
                        ax_hm.set_yticks(range(7))
                        ax_hm.set_yticklabels(dow_labels_hm)
                        ax_hm.set_xlabel("Hora")
                        ax_hm.set_ylabel("Dia da semana")
                        ax_hm.set_title("Distribuição por dia da semana x hora")

                        for yi in range(mat.shape[0]):
                            for xi in range(mat.shape[1]):
                                v = int(mat.iat[yi, xi])
                                if v > 0:
                                    ax_hm.text(xi, yi, str(v), ha="center", va="center", fontsize=7.6)

                        cbar = fig.colorbar(im, ax=ax_hm, fraction=0.046, pad=0.04)
                        cbar.set_label("Ocorrências (pessoa-dia)")

                        total_scope = int(
                            rank_occ.loc[
                                rank_occ[col_cad].eq(str(cad_id))
                                & (rank_occ["_dia_operacional"] >= dt_ini_scope)
                                & (rank_occ["_dia_operacional"] <= dt_fim_scope)
                            ].shape[0]
                        )

                        y_info = 0.95
                        y_info = _draw_wrapped_text(ax_info, 0.00, y_info, "Resumo", width=36, fontsize=12.5, fontweight="bold", line_step=0.060, max_lines=2)
                        y_info = _draw_wrapped_text(
                            ax_info,
                            0.00,
                            y_info,
                            f"Período: {pd.to_datetime(dt_ini_scope).date()} → {pd.to_datetime(dt_fim_scope).date()}",
                            width=42,
                            fontsize=10.2,
                            line_step=0.050,
                            max_lines=2,
                        )
                        y_info = _draw_wrapped_text(
                            ax_info,
                            0.00,
                            y_info,
                            f"Volume da pessoa no ranking: {total_scope}",
                            width=42,
                            fontsize=10.6,
                            line_step=0.052,
                            max_lines=2,
                        )
                        y_info = _draw_wrapped_text(
                            ax_info,
                            0.00,
                            y_info,
                            "Critério: ocorrência-dia por pessoa",
                            width=42,
                            fontsize=10.2,
                            line_step=0.050,
                            max_lines=2,
                        )
                        _draw_wrapped_text(
                            ax_info,
                            0.00,
                            y_info,
                            f"Filtro proxy: <= {int(janela_proxy_minutos)} min, mesma planta, mesmo coletor e terceiro com portaria no dia.",
                            width=42,
                            fontsize=9.6,
                            line_step=0.046,
                            max_lines=4,
                        )

                        _safe_tight_layout(fig, rect=(0.02, 0.03, 0.98, 0.95), pad=1.0)
                        _save_report_page(pdf, fig)

                    # Foco objetivo: 3 heatmaps do ranking por volume-dia nos últimos 30 dias.
                    heatmap_cads = top3_30_hm[:3] if top3_30_hm else top3_total_hm[:3]
                    if heatmap_cads:
                        for pos, cad_id in enumerate(heatmap_cads, start=1):
                            _render_heatmap_ponto_individual(
                                str(cad_id),
                                f"{pos}º do Top 3",
                                "Últimos 30 dias",
                                dt_start_hm,
                                dt_end_hm,
                            )
            except Exception as _e_hm:
                fig = plt.figure(figsize=(11.69, 8.27))
                ax = fig.add_subplot(111)
                ax.axis("off")
                ax.text(0.02, 0.92, "Ponto sem Portaria — heatmaps individuais", fontsize=18, fontweight="bold", va="top")
                ax.text(0.02, 0.80, f"Falha ao gerar heatmaps finais: {_e_hm}", fontsize=11.5, va="top")
                _save_report_page(pdf, fig)

        # 999 como trajeto — Top pessoas (volume ocorrência-dia) + casos individuais
        titulo_l = str(titulo).lower()
        if ("999" in titulo_l) and ("trajeto" in titulo_l or "999" in str(pdf_path).lower()):
            required_cols = ["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]
            if all(c in df_irregularidade.columns for c in required_cols):
                from matplotlib.colors import LinearSegmentedColormap as _LSC

                cols_999 = required_cols + ([col_cargo] if (somente_executivos_atuais and col_cargo) else [])
                dfx = df_irregularidade.loc[:, [c for c in cols_999 if c in df_irregularidade.columns]].copy(deep=False)
                dfx["cadastro"] = dfx["cadastro"].astype(str).str.strip()
                dfx = dfx.loc[dfx["cadastro"].ne("")].copy()

                if somente_executivos_atuais and col_cargo and (col_cargo in dfx.columns):
                    cargo_norm_999 = dfx[col_cargo].fillna("").astype(str).str.strip().str.upper()
                    dfx = dfx.loc[cargo_norm_999.eq("E")]

                data_dt = pd.to_datetime(dfx["data"], dayfirst=True, errors="coerce")
                hora_s = dfx["hora"].fillna("00:00").astype(str).str.strip()
                hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
                hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
                hora_td = pd.to_timedelta(hora_s, errors="coerce")
                dfx["_dt"] = data_dt.dt.normalize() + hora_td
                hora_corte = int(globals().get("hora_corte_operacional", 11))
                dfx["_dia_key"] = (dfx["_dt"] - pd.Timedelta(hours=hora_corte)).dt.normalize()

                re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
                desc_999 = dfx["descricao coletor"].fillna("").astype(str)
                dfx["_is_port"] = desc_999.str.contains(re_port, case=False, na=False, regex=True)

                dirn = dfx["dir"].fillna("").astype(str).str.strip().str.upper()
                dirn = dirn.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
                dfx["_is_E"] = dirn.str.startswith("E")
                dfx["_is_S"] = dirn.str.startswith("S")

                planta_num = pd.to_numeric(dfx["planta"], errors="coerce")
                dfx["_is_999"] = planta_num.eq(999)

                port = dfx.loc[dfx["_is_port"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt", "_is_E", "_is_S"]]
                ev_999 = dfx.loc[dfx["_is_999"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt", "planta", "descricao coletor"]]

                if port.empty or ev_999.empty:
                    fig = plt.figure(figsize=(11.69, 8.27))
                    ax = fig.add_subplot(111)
                    ax.axis("off")
                    ax.text(0.02, 0.92, "999 como trajeto — Top pessoas", fontsize=18, fontweight="bold", va="top")
                    ax.text(0.02, 0.80, "Sem eventos suficientes para classificar o cenário 999 como trajeto.", fontsize=13, va="top")
                    _save_report_page(pdf, fig)
                else:
                    ent = (
                        port.loc[port["_is_E"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].min()
                            .rename("t_ent")
                            .reset_index()
                    )
                    sai = (
                        port.loc[port["_is_S"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].max()
                            .rename("t_sai")
                            .reset_index()
                    )
                    pe = ent.merge(sai, on=["cadastro", "_dia_key"], how="inner")

                    ev = ev_999.merge(pe, on=["cadastro", "_dia_key"], how="inner")
                    if ev.empty:
                        fig = plt.figure(figsize=(11.69, 8.27))
                        ax = fig.add_subplot(111)
                        ax.axis("off")
                        ax.text(0.02, 0.92, "999 como trajeto — Top pessoas", fontsize=18, fontweight="bold", va="top")
                        ax.text(0.02, 0.80, "Sem interseção entre eventos 999 e pares válidos de portaria.", fontsize=13, va="top")
                        _save_report_page(pdf, fig)
                    else:
                        ev["_before"] = ev["_dt"] < ev["t_ent"]
                        ev["_after"] = ev["_dt"] > ev["t_sai"]

                        day_flags = (
                            ev.groupby(["cadastro", "_dia_key"], sort=False)[["_before", "_after"]]
                              .any()
                              .reset_index()
                        )
                        day_flags["cat"] = np.select(
                            [
                                day_flags["_before"] & day_flags["_after"],
                                day_flags["_before"] & (~day_flags["_after"]),
                                (~day_flags["_before"]) & day_flags["_after"],
                            ],
                            ["Ambos", "Antes", "Depois"],
                            default="(sem class.)",
                        )
                        day_flags = day_flags.loc[day_flags["cat"].isin(["Antes", "Depois", "Ambos"])].copy()

                        if day_flags.empty:
                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)
                            ax.axis("off")
                            ax.text(0.02, 0.92, "999 como trajeto — Top pessoas", fontsize=18, fontweight="bold", va="top")
                            ax.text(0.02, 0.80, "Sem dias classificados em Antes/Depois/Ambos.", fontsize=13, va="top")
                            _save_report_page(pdf, fig)
                        else:
                            dt_min_999 = pd.to_datetime(day_flags["_dia_key"].min()).normalize()
                            dt_end_999 = pd.to_datetime(day_flags["_dia_key"].max()).normalize()
                            dt_start_999_30 = dt_end_999 - pd.Timedelta(days=29)

                            day_flags_30 = day_flags.loc[
                                (day_flags["_dia_key"] >= dt_start_999_30)
                                & (day_flags["_dia_key"] <= dt_end_999)
                            ].copy()

                            rank_total = day_flags.groupby("cadastro", sort=False).size().sort_values(ascending=False)
                            rank_30 = day_flags_30.groupby("cadastro", sort=False).size().sort_values(ascending=False)

                            cmap_blue = _LSC.from_list("vw_blue_rank", ["#F6F8FB", "#CFE2F3", "#0B5FA5", "#001E50"])

                            def _render_999_concentracao_top10(
                                df_flags_scope: pd.DataFrame,
                                scope_txt: str,
                                janela_txt: str,
                            ) -> None:
                                fig = plt.figure(figsize=(11.69, 8.27))
                                ax = fig.add_subplot(111)
                                titulo_top10 = (
                                    "999 como trajeto — Top 10 pessoas piores (últimos 30 dias)"
                                    if "30" in str(scope_txt)
                                    else "999 como trajeto — Top 10 pessoas piores (período total)"
                                )

                                if (df_flags_scope is None) or df_flags_scope.empty:
                                    ax.axis("off")
                                    ax.text(0.02, 0.92, titulo_top10, fontsize=18, fontweight="bold", va="top")
                                    ax.text(0.02, 0.85, f"Concentração por categoria ({scope_txt})", fontsize=11.8, va="top")
                                    ax.text(0.02, 0.80, "Sem dados suficientes para este recorte.", fontsize=13, va="top")
                                    ax.text(0.02, 0.72, f"Janela: {janela_txt}", fontsize=11.2, va="top")
                                    _save_report_page(pdf, fig)
                                    return

                                agg = (
                                    df_flags_scope.groupby(["cadastro", "cat"], sort=False)
                                                 .size()
                                                 .rename("n_dias")
                                                 .reset_index()
                                )
                                tot2 = agg.groupby("cadastro", sort=False)["n_dias"].sum().rename("n_total").reset_index()
                                if tot2.empty:
                                    ax.axis("off")
                                    ax.text(0.02, 0.92, titulo_top10, fontsize=18, fontweight="bold", va="top")
                                    ax.text(0.02, 0.85, f"Concentração por categoria ({scope_txt})", fontsize=11.8, va="top")
                                    ax.text(0.02, 0.80, "Sem dados para consolidar Top 10.", fontsize=13, va="top")
                                    _save_report_page(pdf, fig)
                                    return

                                agg = agg.merge(tot2, on="cadastro", how="left")
                                agg["pct"] = agg["n_dias"] / agg["n_total"].where(agg["n_total"] > 0, np.nan)

                                top_cads = (
                                    tot2.sort_values(["n_total", "cadastro"], ascending=[False, True])
                                        .head(10)["cadastro"].astype(str).tolist()
                                )

                                cats = ["Antes", "Depois", "Ambos"]
                                mat = (
                                    agg.loc[agg["cadastro"].isin(top_cads)]
                                       .pivot_table(index="cadastro", columns="cat", values="pct", aggfunc="first")
                                       .reindex(index=top_cads, columns=cats)
                                       .fillna(0.0)
                                )

                                tot_map = tot2.set_index("cadastro")["n_total"].to_dict()
                                y_labels = [f"{_cad_label_basico(str(c))} | Vol: {int(tot_map.get(str(c), 0))}" for c in mat.index]
                                fig.suptitle(titulo_top10, fontsize=14.8, fontweight="bold", y=0.972)
                                fig.text(0.015, 0.935, f"Concentração por categoria ({scope_txt})", fontsize=10.2, ha="left", va="top")
                                fig.text(0.015, 0.912, f"Janela: {janela_txt}", fontsize=10.2, ha="left", va="top")
                                ax.set_xlabel("Categoria")
                                ax.set_xticks([0, 1, 2])
                                ax.set_xticklabels(["Antes de entrar", "Depois de sair", "Ambos"])
                                ax.set_yticks(range(len(y_labels)))
                                ax.set_yticklabels([_wrap(y, 56) for y in y_labels])

                                im = ax.imshow(mat.values, aspect="auto", cmap=cmap_blue, vmin=0.0, vmax=1.0)
                                for yi in range(mat.shape[0]):
                                    for xi in range(mat.shape[1]):
                                        p = float(mat.iloc[yi, xi])
                                        ax.text(xi, yi, f"{p * 100:.0f}%", ha="center", va="center", fontsize=9.2)

                                cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                                cbar.set_label("% por pessoa (linha = 100%)")
                                fig.text(
                                    0.015,
                                    0.03,
                                    "Critério de Top 10: maior volume de ocorrência-dia por pessoa.",
                                    fontsize=9.2,
                                    ha="left",
                                    va="bottom",
                                )

                                fig.subplots_adjust(left=0.35, right=0.92, top=0.86, bottom=0.12)

                                _save_report_page(pdf, fig)

                            _render_999_concentracao_top10(
                                day_flags,
                                scope_txt="Período total",
                                janela_txt=f"{pd.to_datetime(dt_min_999).date()} → {pd.to_datetime(dt_end_999).date()}",
                            )
                            _render_999_concentracao_top10(
                                day_flags_30,
                                scope_txt="Últimos 30 dias",
                                janela_txt=f"{pd.to_datetime(dt_start_999_30).date()} → {pd.to_datetime(dt_end_999).date()}",
                            )

                            # Fechamento final em padrão heatmap (mesma linguagem visual dos heatmaps de concentração).
                            cat_tot = day_flags["cat"].value_counts().reindex(["Antes", "Depois", "Ambos"], fill_value=0)
                            total_reg = int(cat_tot.sum())
                            pessoas_unicas = int(day_flags["cadastro"].astype(str).nunique())
                            dias_unicos = int(pd.to_datetime(day_flags["_dia_key"], errors="coerce").nunique())

                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)

                            if total_reg <= 0:
                                ax.axis("off")
                                ax.text(0.02, 0.92, "999 como trajeto — fechamento geral", fontsize=18, fontweight="bold", va="top")
                                ax.text(0.02, 0.80, "Sem dados suficientes para consolidar o fechamento.", fontsize=13, va="top")
                            else:
                                cats = ["Antes", "Depois", "Ambos"]
                                pct_tot = (cat_tot / total_reg).reindex(cats).fillna(0.0)
                                mat_final = pd.DataFrame([pct_tot.values], index=["Todos os registros"], columns=cats)

                                ax.set_title("999 como trajeto — concentração por categoria (fechamento geral)", fontsize=15.5)
                                ax.text(
                                    0.01,
                                    1.02,
                                    f"Janela: {pd.to_datetime(dt_min_999).date()} → {pd.to_datetime(dt_end_999).date()}",
                                    transform=ax.transAxes,
                                    fontsize=10.2,
                                    va="bottom",
                                    ha="left",
                                )

                                ax.set_xlabel("Categoria")
                                ax.set_xticks([0, 1, 2])
                                ax.set_xticklabels(["Antes de entrar", "Depois de sair", "Ambos"])
                                ax.set_yticks([0])
                                ax.set_yticklabels(["Todos os registros"])

                                im = ax.imshow(mat_final.values, aspect="auto", cmap=cmap_blue, vmin=0.0, vmax=1.0)
                                for xi in range(mat_final.shape[1]):
                                    p = float(mat_final.iloc[0, xi])
                                    n = int(cat_tot.reindex(cats).iloc[xi])
                                    ax.text(xi, 0, f"{p * 100:.0f}%\n({n})", ha="center", va="center", fontsize=10.2)

                                cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                                cbar.set_label("% do total (todos os registros)")

                                fig.text(
                                    0.015,
                                    0.03,
                                    f"Total ocorrência-dia: {total_reg} | Pessoas únicas: {pessoas_unicas} | Dias operacionais: {dias_unicos}",
                                    fontsize=9.4,
                                    ha="left",
                                    va="bottom",
                                )

                            fig.subplots_adjust(left=0.12, right=0.92, top=0.90, bottom=0.12)
                            _save_report_page(pdf, fig)


        # Grill (heatmap) — horário x dia da semana (Top 3 período total + Top 3 últimos 30 dias)
        if analisar_heatmap_grill:
            required_cols_grill = ["cadastro", "data", "hora", "descricao coletor"]
            if all(c in df_irregularidade.columns for c in required_cols_grill):
                cols_grill = ["cadastro", "nome", "data", "hora", "descricao coletor", "planta"]
                dfg = df_irregularidade.loc[:, [c for c in cols_grill if c in df_irregularidade.columns]].copy(deep=False)

                if "planta" in dfg.columns:
                    dfg = dfg.loc[_mask_planta_eq_1(dfg["planta"])].copy()

                data_dt = pd.to_datetime(dfg["data"], dayfirst=True, errors="coerce")
                hora_s = dfg["hora"].fillna("00:00").astype(str).str.strip()
                hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
                hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
                hora_td = pd.to_timedelta(hora_s, errors="coerce")
                dfg["_dt"] = data_dt.dt.normalize() + hora_td
                hora_corte = int(globals().get("hora_corte_operacional", 11))
                dfg["_dia_key"] = (dfg["_dt"] - pd.Timedelta(hours=hora_corte)).dt.normalize()

                _re_grill = globals().get("regex_grill", r"(?i)\bGRILL\b")
                descg = dfg["descricao coletor"].fillna("").astype(str)
                dfg = dfg.loc[dfg["_dt"].notna() & descg.str.contains(_re_grill, case=False, na=False, regex=True)].copy()

                if dfg.empty:
                    fig = plt.figure(figsize=(11.69, 8.27))
                    ax = fig.add_subplot(111)
                    ax.axis("off")
                    ax.text(0.02, 0.92, "Grill — heatmaps Top 3 por pessoa", fontsize=18, fontweight="bold", va="top")
                    ax.text(0.02, 0.80, "Sem eventos de grill válidos para classificar.", fontsize=13, va="top")
                    _save_report_page(pdf, fig)
                else:
                    dfg["cadastro"] = dfg["cadastro"].astype(str).str.strip()
                    dfg = dfg.loc[dfg["cadastro"].ne("")].copy()

                    if dfg.empty:
                        fig = plt.figure(figsize=(11.69, 8.27))
                        ax = fig.add_subplot(111)
                        ax.axis("off")
                        ax.text(0.02, 0.92, "Grill — heatmaps Top 3 por pessoa", fontsize=18, fontweight="bold", va="top")
                        ax.text(0.02, 0.80, "Sem cadastro válido para classificar.", fontsize=13, va="top")
                        _save_report_page(pdf, fig)
                    else:
                        # Ocorrência diária: conta no máximo 1 ocorrência grill por pessoa-dia
                        dfg_pd = (
                            dfg.sort_values("_dt", kind="mergesort")
                               .drop_duplicates(subset=["cadastro", "_dia_key"], keep="first")
                               .copy()
                        )
                        dfg_pd["_dow"] = dfg_pd["_dia_key"].dt.dayofweek.astype(int)
                        dfg_pd["_hora"] = dfg_pd["_dt"].dt.hour.astype(int)

                        cont_total = dfg_pd.groupby("cadastro").size().sort_values(ascending=False)

                        dt_end_grill = pd.to_datetime(dfg_pd["_dia_key"].max()).normalize()
                        dt_start_30_grill = dt_end_grill - pd.Timedelta(days=29)
                        mask_30 = (dfg_pd["_dia_key"] >= dt_start_30_grill) & (dfg_pd["_dia_key"] <= dt_end_grill)
                        cont_30 = dfg_pd.loc[mask_30].groupby("cadastro").size().sort_values(ascending=False)

                        top3_total = cont_total.head(3)
                        top3_30 = cont_30.head(3)

                        dow_labels = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

                        def _render_grill_top3_heatmap(
                            cadastro_id: str,
                            total_pessoa_dia: int,
                            titulo_bloco: str,
                            ordem_txt: str,
                            janela_txt: str,
                            df_source: pd.DataFrame | None = None,
                        ) -> None:
                            base_src = dfg_pd if df_source is None else df_source
                            dfg_cad = base_src.loc[base_src["cadastro"].eq(str(cadastro_id))].copy()
                            if dfg_cad.empty:
                                return

                            mat = (
                                dfg_cad.pivot_table(index="_dow", columns="_hora", values="_dt", aggfunc="count")
                                       .reindex(index=range(7), columns=range(24), fill_value=0)
                                       .fillna(0)
                                       .astype(int)
                            )

                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)

                            im = ax.imshow(mat.values, aspect="auto", cmap="Blues")
                            xticks = list(range(0, 24, 2))
                            ax.set_xticks(xticks)
                            ax.set_xticklabels([f"{h:02d}h" for h in xticks], rotation=0)
                            ax.set_yticks(range(7))
                            ax.set_yticklabels(dow_labels)

                            pessoa_txt = _cad_label(str(cadastro_id))
                            fig.suptitle(f"{titulo_bloco} | {ordem_txt}", fontsize=14.2, fontweight="bold", y=0.972)
                            fig.text(
                                0.015,
                                0.936,
                                _wrap(f"{pessoa_txt} | Total pessoa-dia: {int(total_pessoa_dia)}", 120),
                                fontsize=10.0,
                                ha="left",
                                va="top",
                            )
                            fig.text(0.015, 0.914, f"Janela: {janela_txt}", fontsize=10.0, ha="left", va="top")
                            ax.set_xlabel("Hora do dia")
                            ax.set_ylabel("Dia da semana")

                            vmax = int(mat.to_numpy().max())
                            if vmax > 0:
                                pos = np.argwhere(mat.to_numpy() == vmax)[0]
                                yi, xi = int(pos[0]), int(pos[1])

                                periodos = ["Madrugada (00h-05h)", "Manhã (06h-11h)", "Tarde (12h-17h)", "Noite (18h-23h)"]
                                periodo_idx = int(np.digitize([xi], bins=[6, 12, 18], right=False)[0])
                                periodo_idx = max(0, min(periodo_idx, 3))
                                periodo_txt = periodos[periodo_idx]

                                hora_ini = f"{xi:02d}:00"
                                hora_fim = f"{(xi + 1) % 24:02d}:00"
                                fig.text(
                                    0.015,
                                    0.03,
                                    f"Pico: {dow_labels[yi]} | {hora_ini}-{hora_fim} | {vmax} ocorrência(s) pessoa-dia | Período: {periodo_txt}",
                                    fontsize=9.6,
                                    ha="left",
                                    va="bottom",
                                )

                            for yi in range(mat.shape[0]):
                                for xi in range(mat.shape[1]):
                                    v = int(mat.iat[yi, xi])
                                    if v > 0:
                                        ax.text(xi, yi, str(v), ha="center", va="center", fontsize=7.5)

                            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
                            cbar.set_label("Qtd. de ocorrências (pessoa-dia)")

                            fig.subplots_adjust(left=0.08, right=0.92, top=0.84, bottom=0.12)
                            _save_report_page(pdf, fig)

                        if top3_total.empty and top3_30.empty:
                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)
                            ax.axis("off")
                            ax.text(0.02, 0.92, "Grill — heatmaps Top 3 por pessoa", fontsize=18, fontweight="bold", va="top")
                            ax.text(0.02, 0.80, "Sem dados suficientes para gerar Top 3 no período total ou nos últimos 30 dias.", fontsize=13, va="top")
                            _save_report_page(pdf, fig)
                        else:
                            janela_total_txt = f"{pd.to_datetime(dfg_pd['_dia_key'].min()).date()} → {pd.to_datetime(dfg_pd['_dia_key'].max()).date()}"
                            janela_30_txt = f"{pd.to_datetime(dt_start_30_grill).date()} → {pd.to_datetime(dt_end_grill).date()}"

                            for pos, (cad, qtd) in enumerate(top3_total.items(), start=1):
                                _render_grill_top3_heatmap(
                                    cadastro_id=str(cad),
                                    total_pessoa_dia=int(qtd),
                                    titulo_bloco="Grill — Top 3 por volume (período total)",
                                    ordem_txt=f"{pos}º do Top 3",
                                    janela_txt=janela_total_txt,
                                )

                            for pos, (cad, qtd) in enumerate(top3_30.items(), start=1):
                                _render_grill_top3_heatmap(
                                    cadastro_id=str(cad),
                                    total_pessoa_dia=int(qtd),
                                    titulo_bloco="Grill — Top 3 por volume (últimos 30 dias)",
                                    ordem_txt=f"{pos}º do Top 3",
                                    janela_txt=janela_30_txt,
                                )

                            # Novo recorte: Top 3 com ocorrência no horário das 11h às 13h (inclusive)
                            mask_11_13 = dfg_pd["_hora"].between(11, 13, inclusive="both")
                            dfg_pd_11_13 = dfg_pd.loc[mask_11_13].copy()
                            top3_11_13 = dfg_pd_11_13.groupby("cadastro").size().sort_values(ascending=False).head(3)

                            if top3_11_13.empty:
                                fig = plt.figure(figsize=(11.69, 8.27))
                                ax = fig.add_subplot(111)
                                ax.axis("off")
                                ax.text(0.02, 0.92, "Grill — Top 3 (11h às 13h)", fontsize=18, fontweight="bold", va="top")
                                ax.text(0.02, 0.80, "Sem dados suficientes no recorte de 11h a 13h.", fontsize=13, va="top")
                                _save_report_page(pdf, fig)
                            else:
                                janela_11_13_txt = (
                                    f"{pd.to_datetime(dfg_pd_11_13['_dia_key'].min()).date()} → "
                                    f"{pd.to_datetime(dfg_pd_11_13['_dia_key'].max()).date()}"
                                )
                                for pos, (cad, qtd) in enumerate(top3_11_13.items(), start=1):
                                    _render_grill_top3_heatmap(
                                        cadastro_id=str(cad),
                                        total_pessoa_dia=int(qtd),
                                        titulo_bloco="Grill — Top 3 por volume (11h às 13h)",
                                        ordem_txt=f"{pos}º do Top 3",
                                        janela_txt=janela_11_13_txt,
                                        df_source=dfg_pd_11_13,
                                    )

        # ---------------- ANEXO FINAL: Top 2 piores por volume + base crua (3-4 dias)
        if col_cad and (col_cad in df_irregularidade.columns):
            top2_cad_counts = pd.Series(dtype=int)
            if isinstance(top_pessoas_cad_counts, pd.Series) and (not top_pessoas_cad_counts.empty):
                top2_cad_counts = top_pessoas_cad_counts.head(2).copy()
            elif isinstance(top_pessoas, pd.Series) and (not top_pessoas.empty):
                top2_cad_counts = top_pessoas.head(2).copy()

            if not top2_cad_counts.empty:
                cad_all = df_irregularidade[col_cad].astype(str).str.strip()
                dia_all = _ensure_numpy_datetime_series(dia_op, name="_dia_operacional")
                base_mask = cad_all.ne("") & dia_all.notna()
                if analisar_heatmap_grill and col_planta and (col_planta in df_irregularidade.columns):
                    base_mask = base_mask & _mask_planta_eq_1(df_irregularidade[col_planta])

                col_desc_raw = _pick_col_local(
                    df_irregularidade,
                    ["descricao coletor", "descricao_coletor", "desc_coletor", "coletor", "descricao"],
                )
                col_area_raw = col_areas if col_areas else col_area_fallback

                cols_raw_base = [
                    c
                    for c in [col_cad, col_nome, col_area_raw, "data", "hora", "dir", "planta", col_desc_raw, "_dt_ronda"]
                    if c and (c in df_irregularidade.columns)
                ]

                def _fmt_date_only(v) -> str:
                    if v is None:
                        return ""
                    txt = str(v).strip()
                    if txt.lower() in {"", "nan", "nat", "none"}:
                        return ""
                    dt = pd.to_datetime(v, dayfirst=True, errors="coerce")
                    if pd.isna(dt):
                        dt = pd.to_datetime(txt, errors="coerce")
                    if pd.notna(dt):
                        return pd.to_datetime(dt).strftime("%Y-%m-%d")
                    return re.sub(r"\s+", " ", txt)

                def _fmt_hora_only(v) -> str:
                    if v is None:
                        return ""
                    txt = str(v).strip()
                    txt_low = txt.lower()
                    if txt_low in {"", "nan", "nat", "none"}:
                        return ""

                    m = re.search(r"\b(\d{1,2}:\d{2}(?::\d{2})?)\b", txt)
                    if m:
                        hms = m.group(1)
                        if hms.count(":") == 1:
                            hms = hms + ":00"
                        if re.search(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}", txt) and hms == "00:00:00":
                            return ""
                        return hms

                    td = pd.to_timedelta(txt, errors="coerce")
                    if pd.notna(td):
                        total_sec = int(td.total_seconds()) % (24 * 3600)
                        hh = total_sec // 3600
                        mm = (total_sec % 3600) // 60
                        ss = total_sec % 60
                        if (hh, mm, ss) == (0, 0, 0):
                            return ""
                        return f"{hh:02d}:{mm:02d}:{ss:02d}"

                    dt = pd.to_datetime(txt, dayfirst=True, errors="coerce")
                    if pd.notna(dt):
                        hms = pd.to_datetime(dt).strftime("%H:%M:%S")
                        return "" if hms == "00:00:00" else hms

                    return ""

                def _txt_table(v, max_len: int = 42) -> str:
                    if v is None:
                        return ""
                    txt = re.sub(r"\s+", " ", str(v).strip())
                    if txt.lower() in {"", "nan", "nat", "none"}:
                        return ""
                    if len(txt) <= max_len:
                        return txt
                    return txt[: max_len - 3].rstrip() + "..."

                for pos, (cad_ref, qtd_pd) in enumerate(top2_cad_counts.items(), start=1):
                    cad_ref = str(cad_ref).strip()
                    if cad_ref == "":
                        continue

                    mask_cad = base_mask & cad_all.eq(cad_ref)
                    if not mask_cad.any():
                        continue

                    dias_cad = pd.to_datetime(dia_all.loc[mask_cad], errors="coerce").dropna()
                    if dias_cad.empty:
                        continue

                    vol_por_dia = dias_cad.value_counts().sort_values(ascending=False)
                    n_dias_alvo = 4 if len(vol_por_dia) >= 4 else min(3, len(vol_por_dia))
                    if n_dias_alvo <= 0:
                        continue

                    dias_sel = pd.Index(vol_por_dia.head(n_dias_alvo).index)
                    mask_sel = mask_cad & dia_all.isin(dias_sel)
                    if not mask_sel.any():
                        continue

                    base_crua = df_irregularidade.loc[mask_sel, cols_raw_base].copy(deep=False)
                    base_crua["_dia_operacional"] = pd.to_datetime(dia_all.loc[mask_sel], errors="coerce")
                    base_crua["_dt_sort"] = pd.to_datetime(_build_dt_from_data_hora(base_crua), errors="coerce")
                    base_crua = base_crua.sort_values(["_dia_operacional", "_dt_sort"], kind="mergesort")

                    dias_unicos_pessoa = int(pd.to_datetime(dias_cad, errors="coerce").nunique())
                    pct_total = (int(qtd_pd) / total) if total else 0.0

                    analitico_cols = []
                    if "_dia_operacional" in base_crua.columns:
                        analitico_cols.append("_dia_operacional")
                    for c in ["data", "hora", "dir", "planta", col_desc_raw, col_nome, col_area_raw]:
                        if c and (c in base_crua.columns):
                            analitico_cols.append(c)

                    analitico = base_crua.loc[:, analitico_cols].copy(deep=False)

                    tabela_cols: list[tuple[str, pd.Series]] = []
                    if "_dia_operacional" in analitico.columns:
                        tabela_cols.append(
                            (
                                "Dia operacional",
                                pd.to_datetime(analitico["_dia_operacional"], errors="coerce").dt.strftime("%Y-%m-%d").fillna(""),
                            )
                        )
                    if "data" in analitico.columns:
                        tabela_cols.append(("Data", analitico["data"].map(_fmt_date_only)))
                    if "hora" in analitico.columns:
                        tabela_cols.append(("Hora", analitico["hora"].map(_fmt_hora_only)))
                    if "dir" in analitico.columns:
                        tabela_cols.append(("Dir", analitico["dir"].map(lambda v: _txt_table(v, 6))))
                    if "planta" in analitico.columns:
                        tabela_cols.append(("Planta", analitico["planta"].map(_fmt_planta_nome).map(lambda v: _txt_table(v, 30))))
                    if col_desc_raw and (col_desc_raw in analitico.columns):
                        tabela_cols.append(("Descricao coletor", analitico[col_desc_raw].map(lambda v: _txt_table(v, 40))))
                    if col_nome and (col_nome in analitico.columns):
                        tabela_cols.append(("Nome", analitico[col_nome].map(lambda v: _txt_table(v, 32))))
                    if col_area_raw and (col_area_raw in analitico.columns):
                        tabela_cols.append(("Area", analitico[col_area_raw].map(lambda v: _txt_table(v, 24))))

                    if tabela_cols:
                        tabela_analitico = pd.concat([s.rename(nm) for nm, s in tabela_cols], axis=1)
                    else:
                        tabela_analitico = pd.DataFrame({"Detalhe": ["Sem linhas detalhadas para exibir na base crua selecionada."]})

                    if tabela_analitico.empty:
                        tabela_analitico = pd.DataFrame({"Detalhe": ["Sem linhas detalhadas para exibir na base crua selecionada."]})

                    tabela_analitico = tabela_analitico.fillna("")
                    for cc in list(tabela_analitico.columns):
                        tabela_analitico[cc] = tabela_analitico[cc].map(lambda v: _txt_table(v, 48 if cc == "Descricao coletor" else 36))

                    colw_map = {
                        "Dia operacional": 0.11,
                        "Data": 0.10,
                        "Hora": 0.08,
                        "Dir": 0.05,
                        "Planta": 0.17,
                        "Descricao coletor": 0.20,
                        "Nome": 0.18,
                        "Area": 0.11,
                        "Detalhe": 1.00,
                    }
                    col_widths = [float(colw_map.get(c, 0.12)) for c in list(tabela_analitico.columns)]
                    wsum = float(np.sum(col_widths)) if len(col_widths) > 0 else 1.0
                    if wsum <= 0:
                        col_widths = [1.0 / max(1, len(tabela_analitico.columns))] * max(1, len(tabela_analitico.columns))
                    else:
                        col_widths = [w / wsum for w in col_widths]

                    rows_per_page = 16 if "Descricao coletor" in tabela_analitico.columns else 20
                    n_rows = int(len(tabela_analitico))
                    n_pages = max(1, int(np.ceil(n_rows / max(1, rows_per_page))))

                    for pagina in range(1, n_pages + 1):
                        ini = (pagina - 1) * rows_per_page
                        fim = min(ini + rows_per_page, n_rows)
                        tabela_pg = tabela_analitico.iloc[ini:fim].copy()
                        linhas_pad = max(0, rows_per_page - len(tabela_pg))
                        if linhas_pad > 0:
                            tabela_pg = pd.concat(
                                [
                                    tabela_pg,
                                    pd.DataFrame(
                                        [{cc: "" for cc in tabela_pg.columns} for _ in range(linhas_pad)]
                                    ),
                                ],
                                ignore_index=True,
                            )

                        fig = plt.figure(figsize=(11.69, 8.27))
                        ax = fig.add_subplot(111)
                        ax.axis("off")

                        if pagina == 1:
                            y = 0.96
                            titulo_pg = f"Anexo final analitico - Top {pos} por volume"
                            y = _draw_wrapped_text(ax, 0.02, y, titulo_pg, width=118, fontsize=13.4, fontweight="bold", line_step=0.058, max_lines=2)
                            y = _draw_wrapped_text(ax, 0.02, y, _fmt_pessoa_rm_nome_area(cad_ref), width=118, fontsize=10.4, line_step=0.048, max_lines=2)
                            y = _draw_wrapped_text(
                                ax,
                                0.02,
                                y,
                                f"Ocorrencias pessoa-dia no caso: {int(qtd_pd)} ({pct_total:.1%} do total) | Dias com ocorrencia: {dias_unicos_pessoa}",
                                width=118,
                                fontsize=10.0,
                                line_step=0.046,
                                max_lines=2,
                            )

                            dias_txt = ", ".join([pd.to_datetime(d).strftime("%Y-%m-%d") for d in dias_sel])
                            y = _draw_wrapped_text(
                                ax,
                                0.02,
                                y,
                                f"Dias criticos usados na base crua ({len(dias_sel)}): {dias_txt}",
                                width=118,
                                fontsize=9.8,
                                line_step=0.044,
                                max_lines=2,
                            )
                            y_bottom_table = 0.05
                            table_top = max(0.46, min(0.92, y - 0.020))
                            if table_top <= (y_bottom_table + 0.30):
                                table_top = y_bottom_table + 0.30
                            table_height = table_top - y_bottom_table
                        else:
                            # Da 2a pagina em diante: somente continuidade da tabela.
                            y_bottom_table = 0.04
                            table_height = 0.90
                        tabela = ax.table(
                            cellText=tabela_pg.values.tolist(),
                            colLabels=list(tabela_pg.columns),
                            cellLoc="left",
                            colLoc="left",
                            bbox=[0.02, y_bottom_table, 0.96, table_height],
                            colWidths=col_widths,
                        )
                        tabela.auto_set_font_size(False)
                        tabela.set_fontsize(8.2)
                        tabela.scale(1.0, 1.0)

                        row_h = table_height / float(rows_per_page + 1)

                        for (r, c), cell in tabela.get_celld().items():
                            cell.set_edgecolor("#D7DEE8")
                            cell.set_linewidth(0.55)
                            cell.set_height(row_h)
                            txt_obj = cell.get_text()
                            txt_obj.set_wrap(False)
                            txt_obj.set_va("center")
                            if r == 0:
                                cell.set_facecolor("#EAF2FB")
                                txt_obj.set_fontweight("bold")
                                txt_obj.set_color("#0F172A")
                            else:
                                cell.set_facecolor("#FFFFFF" if (r % 2 == 1) else "#F8FAFC")
                                txt_obj.set_color("#1F2937")

                        _save_report_page(pdf, fig)

def row_ids_from_keys(df_min: pd.DataFrame, keys: pd.DataFrame) -> np.ndarray:
    if keys is None or keys.empty:
        return np.array([], dtype=np.int64)

    keys2 = keys.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates().copy()
    keys2["_dia_operacional"] = pd.to_datetime(keys2["_dia_operacional"], errors="coerce")

    if ("_ROWID_CACHE_HASH" not in globals()) or (len(globals()["_ROWID_CACHE_HASH"]) != len(df_min)):
        globals()["_ROWID_CACHE_ROWID"] = df_min["_row_id"].to_numpy(dtype=np.int64, copy=False)
        globals()["_ROWID_CACHE_HASH"] = pd.util.hash_pandas_object(
            df_min[["cadastro", "_dia_operacional"]], index=False
        ).to_numpy(dtype=np.uint64, copy=False)
        globals()["_ROWID_CACHE_HASH_INDEX"] = pd.Index(globals()["_ROWID_CACHE_HASH"])

    key_hash = pd.util.hash_pandas_object(
        keys2[["cadastro", "_dia_operacional"]], index=False
    ).to_numpy(dtype=np.uint64, copy=False)

    if key_hash.size == 0:
        return np.array([], dtype=np.int64)

    mask = globals()["_ROWID_CACHE_HASH_INDEX"].isin(key_hash)
    return globals()["_ROWID_CACHE_ROWID"][mask].astype(np.int64, copy=True)

# Mapa dia_operacional por row_id (para colocar na saída)
n_total = len(df_min_raw)
dia_op_arr = np.full(n_total, np.datetime64("NaT"), dtype="datetime64[ns]")
valid_map = df.loc[df["_dt_ronda"].notna(), ["_row_id", "_dia_operacional"]]
dia_op_arr[valid_map["_row_id"].to_numpy()] = valid_map["_dia_operacional"].to_numpy()

def materializar_e_exportar_por_rowid(
    row_ids: np.ndarray,
    irregularidade_col: str,
    titulo: str,
    df_relatorio: pd.DataFrame | None = None,
) -> pd.DataFrame:
    slug = slugify(irregularidade_col)

    parquet_path = DFS_DIR / f"{slug}.parquet"
    pdf_path = PDF_DIR / f"{slug}.pdf"

    excluir_planta_999_nesta_irregularidade = _deve_excluir_planta_999(
        irregularidade_col=irregularidade_col,
        titulo=titulo,
        pdf_path=pdf_path,
    )

    params_export = {
        "param_janela_portaria_min": globals().get("janela_portaria_min"),
        "param_tolerancia_entrada_min": globals().get("tolerancia_entrada_min"),
        "param_tolerancia_saida_min": globals().get("tolerancia_saida_min"),
        "param_janela_999_horas": globals().get("janela_999_horas"),
        "param_hora_corte_operacional": globals().get("hora_corte_operacional"),
        "param_descanso_min_horas": globals().get("descanso_min_horas"),
        "param_jornada_min_horas": globals().get("jornada_min_horas"),
        "param_jornada_min_minutos": globals().get("jornada_min_minutos"),
        "param_usar_jornada_por_portaria": globals().get("usar_jornada_por_portaria"),
        "param_max_jornada_portaria_horas": globals().get("max_jornada_portaria_horas"),
    }

    base_relatorio = df_relatorio if isinstance(df_relatorio, pd.DataFrame) else None

    if excluir_planta_999_nesta_irregularidade:
        row_ids = _filtrar_row_ids_sem_planta_999(row_ids)
        base_relatorio = _filtrar_df_sem_planta_999(base_relatorio)

    if row_ids.size == 0:
        df_out = df_min_raw.head(0).copy()
        df_out["dia_operacional"] = pd.to_datetime([], errors="coerce")
        df_out[irregularidade_col] = False
        for _k, _v in params_export.items():
            df_out[_k] = _v

        df_out.to_parquet(parquet_path, index=False)
        gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_out, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_out

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]

    MAX_ROWS_IN_MEMORY = 2_000_000
    REPORT_SAMPLE_ROWS = 2_000

    if row_ids.size > MAX_ROWS_IN_MEMORY:
        import pyarrow as pa
        import pyarrow.parquet as pq

        row_ids = np.sort(row_ids.astype(np.int64, copy=False))
        chunk_size = BATCH_SIZE

        if parquet_path.exists():
            parquet_path.unlink()

        writer = None
        report_parts = []
        report_count = 0

        for i in range(0, row_ids.size, chunk_size):
            ids = row_ids[i:i + chunk_size]
            part = df_min_raw.iloc[ids].copy()
            part["dia_operacional"] = pd.to_datetime(dia_op_arr[ids], errors="coerce")
            part[irregularidade_col] = True
            for _k, _v in params_export.items():
                part[_k] = _v
            part.drop(columns=["_row_id"], inplace=True, errors="ignore")

            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(parquet_path), table.schema, compression="snappy")
            writer.write_table(table)

            if report_count < REPORT_SAMPLE_ROWS:
                take = min(REPORT_SAMPLE_ROWS - report_count, len(part))
                report_parts.append(part.head(take).copy())
                report_count += take

            del ids, part, table
            gc.collect()

        if writer is not None:
            writer.close()

        if report_parts:
            df_report_fallback = pd.concat(report_parts, ignore_index=True)
        else:
            df_report_fallback = df_min_raw.head(0).copy()
            df_report_fallback["dia_operacional"] = pd.to_datetime([], errors="coerce")
            df_report_fallback[irregularidade_col] = False
            for _k, _v in params_export.items():
                df_report_fallback[_k] = _v

        if excluir_planta_999_nesta_irregularidade:
            df_report_fallback = _filtrar_df_sem_planta_999(df_report_fallback)

        gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_report_fallback, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_report_fallback

    df_out = df_min_raw.iloc[row_ids].copy()
    df_out["dia_operacional"] = pd.to_datetime(dia_op_arr[row_ids], errors="coerce")
    df_out[irregularidade_col] = True
    for _k, _v in params_export.items():
        df_out[_k] = _v
    df_out.drop(columns=["_row_id"], inplace=True, errors="ignore")

    if excluir_planta_999_nesta_irregularidade:
        df_out = _filtrar_df_sem_planta_999(df_out)

    df_out.to_parquet(parquet_path, index=False)
    gerar_relatorio_pdf(base_relatorio if base_relatorio is not None else df_out, titulo=titulo, pdf_path=pdf_path)

    gc.collect()
    print("Parquet:", parquet_path)
    print("PDF   :", pdf_path)
    return df_out

# Mantém referência dos helpers modernos para evitar regressão por redefinições posteriores.
_gerar_relatorio_pdf_moderno = gerar_relatorio_pdf
_row_ids_from_keys_moderno = row_ids_from_keys
_materializar_e_exportar_por_rowid_moderno = materializar_e_exportar_por_rowid



In [ ]:
# =========================
# IRREGULARIDADE: 999 como Trajeto
# =========================
irregularidade_col = "Irregularidade - 999 como Trajeto"
titulo = "Irregularidade – 999 como Trajeto"

# Base: precisa ter presencial e 999 no mesmo dia
day999_counts = (
    df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional"]]
      .groupby(["cadastro", "_dia_operacional"])
      .size()
      .rename("n_999")
      .reset_index()
)

day_ = day.merge(day999_counts, on=["cadastro", "_dia_operacional"], how="left")
day_["n_999"] = day_["n_999"].fillna(0).astype(int)

# Anti-truncamento específico da janela 999
day_["ANTI_TRUNC_999"] = False
mask_mm = day_["min_dt_cad"].notna() & day_["max_dt_cad"].notna()
day_.loc[mask_mm, "ANTI_TRUNC_999"] = (
    (day_.loc[mask_mm, "first_any_pres"] - t_jan_999) < day_.loc[mask_mm, "min_dt_cad"]
) | (
    (day_.loc[mask_mm, "last_any_pres"] + t_jan_999) > day_.loc[mask_mm, "max_dt_cad"]
)

# Conta 999 antes/depois do presencial
mask_999 = df_valid["999"].fillna(False).to_numpy()
if mask_999.any():
    idx_999 = np.flatnonzero(mask_999)
    ev_999 = df_valid.iloc[idx_999][["cadastro", "_dia_operacional", "_dt_ronda"]].copy()
else:
    ev_999 = pd.DataFrame(columns=["cadastro", "_dia_operacional", "_dt_ronda"])
del mask_999
ev_999 = ev_999.merge(
    day_.loc[:, ["cadastro", "_dia_operacional", "first_any_pres", "last_any_pres"]],
    on=["cadastro", "_dia_operacional"],
    how="inner",
)

ev_999["in_before"] = (ev_999["_dt_ronda"] >= (ev_999["first_any_pres"] - t_jan_999)) & (ev_999["_dt_ronda"] < ev_999["first_any_pres"])
ev_999["in_after"]  = (ev_999["_dt_ronda"] >  ev_999["last_any_pres"]) & (ev_999["_dt_ronda"] <= (ev_999["last_any_pres"] + t_jan_999))

cnt = (
    ev_999.groupby(["cadastro", "_dia_operacional"], as_index=False)
          .agg(cnt_before=("in_before", "sum"), cnt_after=("in_after", "sum"))
)

day_ = day_.merge(cnt, on=["cadastro", "_dia_operacional"], how="left")
day_[["cnt_before", "cnt_after"]] = day_[["cnt_before", "cnt_after"]].fillna(0).astype(int)

odd_before = (day_["cnt_before"] % 2 == 1) & (day_["cnt_before"] > 0)
odd_after  = (day_["cnt_after"] % 2 == 1) & (day_["cnt_after"] > 0)

day_[irregularidade_col] = (
    (day_["n_999"] > 0)
    & day_["HAS_PORTARIA_ANY"]
    & (odd_before | odd_after)
    & (~day_["ANTI_TRUNC_999"])
)

# ==========================================================
# 4.1) CORREÇÃO "FALSO VERDADEIRO" (ponto fechado) -> zera o dia
#     (INSERIDO AQUI — altera apenas o necessário)
# ==========================================================
def _norm_dir_series(s: pd.Series) -> pd.Series:
    x = s.fillna("").astype(str).str.strip().str.upper()
    x = x.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
    x = x.where(~x.str.startswith("E"), "E")
    x = x.where(~x.str.startswith("S"), "S")
    return x

def _dia_e_falso_verdadeiro(g: pd.DataFrame) -> bool:
    # g deve conter: _dt_ronda, _is_portaria, _dir_norm, _is_ponto_marca
    g = g.sort_values("_dt_ronda", kind="mergesort")
    port = g[g["_is_portaria"]]
    if port.empty:
        return False

    first_port = port.iloc[0]
    last_port  = port.iloc[-1]

    if first_port["_dir_norm"] != "E":
        return False
    if last_port["_dir_norm"] != "S":
        return False

    t0 = first_port["_dt_ronda"]
    t1 = last_port["_dt_ronda"]
    if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
        return False

    # Se houver 999 valido na janela antes/depois da portaria, nao descartar como falso verdadeiro.
    is_999 = g.get("999", pd.Series(False, index=g.index)).fillna(False)
    if is_999.any():
        t999 = g.loc[is_999, "_dt_ronda"]
        has_999_before = ((t999 >= (t0 - t_jan_999)) & (t999 < t0)).any()
        has_999_after = ((t999 > t1) & (t999 <= (t1 + t_jan_999))).any()
        if has_999_before or has_999_after:
            return False

    pontos_entre = g[
        g["_is_ponto_marca"]
        & (g["_dt_ronda"] >= t0)
        & (g["_dt_ronda"] <= t1)
    ]
    return len(pontos_entre) >= 2

keys_true = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].drop_duplicates()

grupos_descartar = set()
MAX_ROWS_VALIDACAO_FALSO_999 = int(globals().get("max_rows_validacao_falso_999", 20_000_000))
MAX_CHAVES_VALIDACAO_FALSO_999 = int(globals().get("max_chaves_validacao_falso_999", 350_000))
if not keys_true.empty:
    if (len(df_valid) > MAX_ROWS_VALIDACAO_FALSO_999) or (len(keys_true) > MAX_CHAVES_VALIDACAO_FALSO_999):
        print(
            "WARN: validação 'falso verdadeiro' do 999 foi pulada por volume alto "
            f"(linhas_df_valid={len(df_valid):,}; chaves_positivas={len(keys_true):,})."
        )
    else:
    # traz só linhas necessárias do df_valid para validar falsos positivos
        cols_need = ["cadastro", "_dia_operacional", "_dt_ronda", "999"]
        if "Portaria" in df_valid.columns: cols_need.append("Portaria")
        if "dir" in df_valid.columns: cols_need.append("dir")
        if "_dir_norm" in df_valid.columns: cols_need.append("_dir_norm")
        if "descricao coletor" in df_valid.columns: cols_need.append("descricao coletor")

        cols_base = [c for c in cols_need if c in df_valid.columns]

        # reduz o universo por cadastro antes do merge por dia
        cads_true = keys_true["cadastro"].drop_duplicates()
        mask_cad = df_valid["cadastro"].isin(cads_true)
        df_check = df_valid.loc[mask_cad, cols_base]
        del mask_cad

        df_check = df_check.merge(keys_true, on=["cadastro", "_dia_operacional"], how="inner")

        # portaria flag
        if "Portaria" in df_check.columns:
            df_check["_is_portaria"] = df_check["Portaria"].fillna(False).astype(bool)
        else:
            if "descricao coletor" in df_check.columns:
                descu = df_check["descricao coletor"].fillna("").astype(str).str.upper()
            else:
                descu = pd.Series("", index=df_check.index, dtype="string")
            df_check["_is_portaria"] = descu.str.contains(r"PORT", regex=True, na=False)

        # direção normalizada
        if "_dir_norm" in df_check.columns:
            df_check["_dir_norm"] = _norm_dir_series(df_check["_dir_norm"])
        elif "dir" in df_check.columns:
            df_check["_dir_norm"] = _norm_dir_series(df_check["dir"])
        else:
            df_check["_dir_norm"] = ""

        # ponto/marca (?:MARCA|PONTO|PONT) e NÃO portaria
        if "descricao coletor" in df_check.columns:
            descu = df_check["descricao coletor"].fillna("").astype(str).str.upper()
        else:
            descu = pd.Series("", index=df_check.index, dtype="string")
        df_check["_is_ponto_marca"] = (
            descu.str.contains(r"(?:MARCA|PONTO|PONT)", regex=True, na=False)
            & (~df_check["_is_portaria"])
        )

        grp_flag = (
            df_check.groupby(["cadastro", "_dia_operacional"], sort=False)
                    .apply(_dia_e_falso_verdadeiro)
        )
        grupos_descartar = set(grp_flag[grp_flag].index.tolist())

if grupos_descartar:
    day_["_k"] = list(zip(day_["cadastro"], day_["_dia_operacional"]))
    day_.loc[day_["_k"].isin(grupos_descartar), irregularidade_col] = False
    day_.drop(columns=["_k"], inplace=True, errors="ignore")

# (continua o fluxo normal após a validação)
keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_999_trajeto = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

# (opcional) manter em memória
if MANTER_DF_EM_MEMORIA:
    df_irregularidade_999_como_trajeto = df_999_trajeto

df_999_trajeto.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Ponto sem Portaria
# Regra de negócio:
# - suspeito: horista com PONTO presencial no dia e sem nenhuma portaria no mesmo dia;
# - não usa janela entre portaria e ponto para validar a portaria do suspeito;
# - terceiro: qualquer pessoa com PONTO em até 3 minutos do 1º OU do último ponto do suspeito;
# - o PONTO do terceiro deve ocorrer na MESMA PLANTA e no MESMO COLETOR do ponto do suspeito;
# - o terceiro precisa ter pelo menos 1 portaria no mesmo dia operacional.
# Objetivo:
# - caçar casos em que a pessoa não veio à fábrica, mas outra pessoa pode ter batido o ponto para ela.
# =========================
irregularidade_col = "Irregularidade - Ponto sem Portaria"
 
JANELA_TERCEIRO_MIN = float(
    globals().get("janela_terceiro_ponto_min", globals().get("janela_ponto_sem_portaria_min", 3))
)
JANELA_TERCEIRO_MIN = max(0.0, min(3.0, JANELA_TERCEIRO_MIN))
 
# Pela regra de negócio atual: 1º OU último ponto
EXIGIR_TERCEIRO_DUAS_PONTAS = False
 
# Pela metodologia atual: mesma planta e mesmo coletor
EXIGIR_MESMA_PLANTA = bool(globals().get("exigir_mesma_planta_proxy_ponto", True))
 
_terceiro_txt = "1º OU no último ponto"
_planta_txt = "mesma planta e "
titulo = (
    "Irregularidade  Ponto sem Portaria "
    "(somente horistas; suspeito sem nenhuma portaria no dia; "
    "sem filtrar horistas e terceiros pela janela entre portaria e ponto; "
    f"com terceiro (que tenha pelo menos 1 portaria no dia) em até {JANELA_TERCEIRO_MIN:g} min do {_terceiro_txt}; "
    f"{_planta_txt}mesmo coletor de ponto) "
    f"Gerado em: {pd.Timestamp.now().strftime('%Y-%m-%d')}"
)
 
def _proxy_num_coletor_series(s: pd.Series) -> pd.Series:
    txt = s.fillna("").astype(str).str.strip().str.upper()
    txt = txt.apply(lambda v: unicodedata.normalize("NFKD", v).encode("ascii", "ignore").decode("ascii"))
    num = txt.str.extract(r"(?:COLETOR|CATRACA|RELOGIO(?:\s*PONTO)?|PONTO|MARCA)\D*([0-9]{1,6})", expand=False)
    num_fallback = txt.str.extract(r"\b([0-9]{1,6})\b", expand=False)
    num = num.fillna(num_fallback)
    num = pd.to_numeric(num, errors="coerce")
    return num.map(lambda x: "" if pd.isna(x) else str(int(x)))
 
day_ = day.copy()
 
day_["IS_HORISTA_DAY"] = False
day_["DIA_CARGO_MISTO"] = False
day_["tem_entrada_portaria_dia"] = False
day_["tem_saida_portaria_dia"] = False
day_["PORTARIA_SEQUENCIA_IRREGULAR"] = False
day_["PORTARIA_IRREGULAR_DIA"] = False
day_["SUSPEITO_SEM_PORTARIA_DIA"] = False
day_["HAS_TERCEIRO_PROXIMO_ENTRADA"] = False
day_["HAS_TERCEIRO_PROXIMO_SAIDA"] = False
 
# --------------------------------------------------
# 1) Classificação de horista por cadastro
# --------------------------------------------------
if "cargo" in df.columns:
    cargo_map = (
        df.loc[df["cargo"].notna(), ["cadastro", "cargo"]]
          .drop_duplicates()
          .copy(deep=False)
    )
    cargo_map["cadastro"] = cargo_map["cadastro"].astype(str).str.strip()
    cargo_map["cargo_norm"] = cargo_map["cargo"].astype(str).str.strip().str.upper()
    cargo_map = cargo_map.loc[cargo_map["cadastro"].ne("") & cargo_map["cargo_norm"].ne("")]
 
    if not cargo_map.empty:
        cargo_map["_is_h"] = cargo_map["cargo_norm"].eq("H")
        cargo_map["_is_non_h"] = ~cargo_map["cargo_norm"].eq("H")
 
        agg_cargo = (
            cargo_map.groupby("cadastro", sort=False)
                .agg(
                    has_h=("_is_h", "any"),
                    has_non_h=("_is_non_h", "any"),
                    n_cargos=("cargo_norm", "nunique"),
                )
                .reset_index()
        )
 
        agg_cargo["IS_HORISTA_DAY"] = agg_cargo["has_h"] & (~agg_cargo["has_non_h"])
        agg_cargo["DIA_CARGO_MISTO"] = agg_cargo["n_cargos"].gt(1)
 
        day_ = day_.merge(
            agg_cargo[["cadastro", "IS_HORISTA_DAY", "DIA_CARGO_MISTO"]],
            on="cadastro",
            how="left",
            suffixes=("", "_C"),
        )
 
        if "IS_HORISTA_DAY_C" in day_.columns:
            day_["IS_HORISTA_DAY"] = day_["IS_HORISTA_DAY_C"].fillna(False).astype(bool)
            day_.drop(columns=["IS_HORISTA_DAY_C"], inplace=True, errors="ignore")
        else:
            day_["IS_HORISTA_DAY"] = day_["IS_HORISTA_DAY"].fillna(False).astype(bool)
 
        if "DIA_CARGO_MISTO_C" in day_.columns:
            day_["DIA_CARGO_MISTO"] = day_["DIA_CARGO_MISTO_C"].fillna(False).astype(bool)
            day_.drop(columns=["DIA_CARGO_MISTO_C"], inplace=True, errors="ignore")
        else:
            day_["DIA_CARGO_MISTO"] = day_["DIA_CARGO_MISTO"].fillna(False).astype(bool)
 
# --------------------------------------------------
# 2) Portaria do dia: flags + sequência inválida
# --------------------------------------------------
port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)
 
if not port.empty:
    port["cadastro"] = port["cadastro"].astype(str).str.strip()
    port = port.loc[port["cadastro"].ne("")].copy()
    port["_dt_ronda"] = pd.to_datetime(port["_dt_ronda"], errors="coerce")
    port["_dia_operacional"] = pd.to_datetime(port["_dia_operacional"], errors="coerce")
 
    dnorm = port["_dir_norm"].fillna("").astype(str).str.strip().str.upper()
    dnorm = dnorm.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
    dnorm = dnorm.where(~dnorm.str.startswith("E"), "E")
    dnorm = dnorm.where(~dnorm.str.startswith("S"), "S")
    port["_dir_norm"] = dnorm
 
port = port.loc[port["_dt_ronda"].notna() & port["_dia_operacional"].notna()].copy()
 
if not port.empty:
    port_flags = (
        port.assign(
            _is_entry=port["_dir_norm"].eq("E"),
            _is_saida=port["_dir_norm"].eq("S"),
        )
        .groupby(["cadastro", "_dia_operacional"], sort=False)
        .agg(
            tem_entrada_portaria_dia=("_is_entry", "any"),
            tem_saida_portaria_dia=("_is_saida", "any"),
        )
        .reset_index()
    )
 
    pf_dia = pd.to_datetime(port_flags["_dia_operacional"], errors="coerce")
    port_flags = port_flags.loc[pf_dia.notna()].copy()
    port_flags["_dia_i"] = pf_dia.loc[port_flags.index].to_numpy(dtype="datetime64[D]").astype(np.int64)
 
    port_seq = port.loc[
        port["_dir_norm"].isin(["E", "S"]),
        ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"],
    ].copy(deep=False)
 
    if port_seq.empty:
        seq_bad = port_flags.loc[:, ["cadastro", "_dia_i"]].copy(deep=False)
        seq_bad["PORTARIA_SEQUENCIA_IRREGULAR"] = False
    else:
        grp_cols = ["cadastro", "_dia_operacional"]
        port_seq = port_seq.sort_values(grp_cols + ["_dt_ronda"], kind="mergesort")
 
        cad_arr = port_seq["cadastro"].to_numpy(copy=False)
        dia_arr = pd.to_datetime(port_seq["_dia_operacional"], errors="coerce").to_numpy(copy=False)
        dir_arr = port_seq["_dir_norm"].fillna("").astype(str).to_numpy(copy=False)
        dia_i_arr = dia_arr.astype("datetime64[D]").astype(np.int64, copy=False)
 
        n = len(port_seq)
        grp_start = np.empty(n, dtype=bool)
        grp_start[0] = True
        grp_start[1:] = (cad_arr[1:] != cad_arr[:-1]) | (dia_arr[1:] != dia_arr[:-1])
        start_idx = np.flatnonzero(grp_start)
        end_idx = np.r_[start_idx[1:], n] - 1
 
        size_arr = end_idx - start_idx + 1
        first_dir = dir_arr[start_idx]
        last_dir = dir_arr[end_idx]
        first_last_bad = (first_dir != "E") | (last_dir != "S")
 
        repeated_row = np.zeros(n, dtype=bool)
        if n > 1:
            same_group_adj = ~grp_start[1:]
            repeated_row[1:] = (dir_arr[1:] == dir_arr[:-1]) & same_group_adj
        repeated_group = np.maximum.reduceat(repeated_row.astype(np.int8), start_idx).astype(bool)
 
        seq_arr = (size_arr > 1) & (first_last_bad | repeated_group)
 
        seq_bad = pd.DataFrame(
            {
                "cadastro": cad_arr[start_idx],
                "_dia_i": dia_i_arr[start_idx],
                "PORTARIA_SEQUENCIA_IRREGULAR": seq_arr,
            }
        )
else:
    port_flags = pd.DataFrame(columns=["cadastro", "_dia_operacional", "tem_entrada_portaria_dia", "tem_saida_portaria_dia", "_dia_i"])
    seq_bad = pd.DataFrame(columns=["cadastro", "_dia_i", "PORTARIA_SEQUENCIA_IRREGULAR"])
 
# --------------------------------------------------
# 3) Candidatos suspeitos: horista com ponto presencial
# --------------------------------------------------
mask_base = (
    (~day_["SEM_COBERTURA_CONTEXTO"].fillna(False).astype(bool))
    & day_["IS_HORISTA_DAY"].fillna(False).astype(bool)
    & day_["first_ponto_pres"].notna()
)
 
cand = day_.loc[
    mask_base,
    ["cadastro", "_dia_operacional", "first_ponto_pres", "last_ponto_pres"]
].copy(deep=False)
 
if not cand.empty:
    cand["_idx_day"] = cand.index.to_numpy(dtype=np.int64, copy=False)
    cand["cadastro"] = cand["cadastro"].astype(str).str.strip()
 
    cand_dia = pd.to_datetime(cand["_dia_operacional"], errors="coerce")
    cand = cand.loc[cand_dia.notna()].copy()
    cand["_dia_i"] = cand_dia.loc[cand.index].to_numpy(dtype="datetime64[D]").astype(np.int64)
    cand["_dia_operacional"] = cand_dia.loc[cand.index].dt.normalize()
    cand = cand.loc[cand["cadastro"].ne("")].copy()
 
    # flags de portaria do suspeito
    port_flags["cadastro"] = port_flags["cadastro"].astype(str).str.strip()
    seq_bad["cadastro"] = seq_bad["cadastro"].astype(str).str.strip()
 
    cand = cand.merge(
        port_flags[["cadastro", "_dia_i", "tem_entrada_portaria_dia", "tem_saida_portaria_dia"]],
        on=["cadastro", "_dia_i"],
        how="left",
    )
    cand = cand.merge(
        seq_bad[["cadastro", "_dia_i", "PORTARIA_SEQUENCIA_IRREGULAR"]],
        on=["cadastro", "_dia_i"],
        how="left",
    )
 
    cand["tem_entrada_portaria_dia"] = cand["tem_entrada_portaria_dia"].fillna(False).astype(bool)
    cand["tem_saida_portaria_dia"] = cand["tem_saida_portaria_dia"].fillna(False).astype(bool)
    cand["PORTARIA_SEQUENCIA_IRREGULAR"] = cand["PORTARIA_SEQUENCIA_IRREGULAR"].fillna(False).astype(bool)
 
    cand["SUSPEITO_SEM_PORTARIA_DIA"] = (
        (~cand["tem_entrada_portaria_dia"])
        & (~cand["tem_saida_portaria_dia"])
    )
 
    if cand["SUSPEITO_SEM_PORTARIA_DIA"].any():
        idx_sem_port = cand.loc[cand["SUSPEITO_SEM_PORTARIA_DIA"], "_idx_day"].to_numpy(dtype=np.int64, copy=False)
        day_.loc[idx_sem_port, "SUSPEITO_SEM_PORTARIA_DIA"] = True
 
        cand = cand.loc[cand["SUSPEITO_SEM_PORTARIA_DIA"]].copy()
 
        # --------------------------------------------------
        # 4) Preparar timestamps do 1º e último ponto do suspeito
        # --------------------------------------------------
        cand_first = pd.to_datetime(cand["first_ponto_pres"], errors="coerce")
        cand_last = pd.to_datetime(cand["last_ponto_pres"], errors="coerce")
        cand_last = cand_last.where(cand_last.notna(), cand_first)
 
        cand = cand.loc[cand_first.notna()].copy()
        cand["_sec_first"] = (cand_first.loc[cand.index].astype("int64") // 1_000_000_000).astype(np.int64)
        cand["_sec_last"] = (cand_last.loc[cand.index].astype("int64") // 1_000_000_000).astype(np.int64)
 
        cand["HAS_TERCEIRO_PROXIMO_ENTRADA"] = False
        cand["HAS_TERCEIRO_PROXIMO_SAIDA"] = False
 
        cand_day_set = set(cand["_dia_i"].astype(np.int64).dropna().unique().tolist())
 
        # --------------------------------------------------
        # 5) Enriquecer suspeito com planta/coletor do 1º e último ponto
        # Usa df (não df_valid), porque precisamos de planta e descrição do coletor
        # --------------------------------------------------
        ponto_src = df.loc[
            df["Ponto"].eq(True) & df["Presencial"].eq(True),
            ["cadastro", "_dia_operacional", "_dt_ronda", "planta", "descricao coletor"]
        ].copy(deep=False)
 
        if not ponto_src.empty:
            ponto_src["cadastro"] = ponto_src["cadastro"].astype(str).str.strip()
            ponto_src["_dia_operacional"] = pd.to_datetime(ponto_src["_dia_operacional"], errors="coerce")
            ponto_src["_dt_ronda"] = pd.to_datetime(ponto_src["_dt_ronda"], errors="coerce")
            ponto_src["planta_num"] = pd.to_numeric(ponto_src["planta"], errors="coerce")
            ponto_src["coletor_num"] = _proxy_num_coletor_series(ponto_src["descricao coletor"])
 
            ponto_src = ponto_src.loc[
                ponto_src["cadastro"].ne("")
                & ponto_src["_dia_operacional"].notna()
                & ponto_src["_dt_ronda"].notna()
            ].copy()
 
            first_meta = (
                cand[["cadastro", "_dia_operacional", "first_ponto_pres"]]
                .merge(
                    ponto_src.rename(columns={
                        "_dt_ronda": "first_ponto_pres",
                        "planta_num": "planta_first",
                        "coletor_num": "coletor_first",
                    })[["cadastro", "_dia_operacional", "first_ponto_pres", "planta_first", "coletor_first"]],
                    on=["cadastro", "_dia_operacional", "first_ponto_pres"],
                    how="left",
                )
                .drop_duplicates(subset=["cadastro", "_dia_operacional", "first_ponto_pres"], keep="first")
            )
 
            last_meta = (
                cand[["cadastro", "_dia_operacional", "last_ponto_pres"]]
                .merge(
                    ponto_src.rename(columns={
                        "_dt_ronda": "last_ponto_pres",
                        "planta_num": "planta_last",
                        "coletor_num": "coletor_last",
                    })[["cadastro", "_dia_operacional", "last_ponto_pres", "planta_last", "coletor_last"]],
                    on=["cadastro", "_dia_operacional", "last_ponto_pres"],
                    how="left",
                )
                .drop_duplicates(subset=["cadastro", "_dia_operacional", "last_ponto_pres"], keep="first")
            )
 
            cand = cand.merge(
                first_meta,
                on=["cadastro", "_dia_operacional", "first_ponto_pres"],
                how="left",
            )
            cand = cand.merge(
                last_meta,
                on=["cadastro", "_dia_operacional", "last_ponto_pres"],
                how="left",
            )
        else:
            cand["planta_first"] = np.nan
            cand["coletor_first"] = ""
            cand["planta_last"] = np.nan
            cand["coletor_last"] = ""
 
        # --------------------------------------------------
        # 6) Construir base de PONTO dos terceiros, com planta/coletor
        # --------------------------------------------------
        ev_parts = []
        if cand_day_set:
            cand_days_arr = np.array(list(cand_day_set), dtype=np.int64)
            nrows = len(df)
            step = int(min(BATCH_SIZE, 100_000))
 
            for i0 in range(0, nrows, step):
                part = df.iloc[i0:i0 + step]
 
                mask = part["Ponto"].eq(True) & part["Presencial"].eq(True)
                if not mask.any():
                    continue
 
                part2 = part.loc[
                    mask,
                    ["cadastro", "_dia_operacional", "_dt_ronda", "planta", "descricao coletor"]
                ].copy(deep=False)
 
                dia = pd.to_datetime(part2["_dia_operacional"], errors="coerce")
                dt = pd.to_datetime(part2["_dt_ronda"], errors="coerce")
                cad = part2["cadastro"].astype(str).str.strip()
                pl = pd.to_numeric(part2["planta"], errors="coerce")
                col = _proxy_num_coletor_series(part2["descricao coletor"])
 
                valid = dia.notna() & dt.notna() & cad.ne("")
                if not valid.any():
                    continue
 
                dia_i = dia.to_numpy(dtype="datetime64[D]")[valid].astype(np.int64, copy=False)
                keep = np.isin(dia_i, cand_days_arr)
                if not keep.any():
                    continue
 
                cad_arr = cad.to_numpy()[valid][keep]
                sec_arr = (dt.to_numpy(dtype="datetime64[ns]")[valid].astype(np.int64, copy=False) // 1_000_000_000)[keep]
                pl_arr = pl.to_numpy(dtype=float, copy=False)[valid][keep]
                col_arr = col.to_numpy(dtype=object, copy=False)[valid][keep]
 
                ev_parts.append((cad_arr, dia_i[keep], sec_arr, pl_arr, col_arr))
 
                if (i0 // step) % 10 == 0:
                    gc.collect()
 
        if ev_parts:
            ev_cad = np.concatenate([p[0] for p in ev_parts])
            ev_day = np.concatenate([p[1] for p in ev_parts]).astype(np.int64, copy=False)
            ev_sec = np.concatenate([p[2] for p in ev_parts]).astype(np.int64, copy=False)
            ev_pl = np.concatenate([p[3] for p in ev_parts]).astype(float, copy=False)
            ev_col = np.concatenate([p[4] for p in ev_parts]).astype(object, copy=False)
 
            ev = pd.DataFrame({
                "cadastro": ev_cad,
                "_dia_i": ev_day,
                "_sec": ev_sec,
                "planta": ev_pl,
                "coletor": ev_col,
            })
        else:
            ev = pd.DataFrame(columns=["cadastro", "_dia_i", "_sec", "planta", "coletor"])
 
        # --------------------------------------------------
        # 7) Terceiro precisa ter pelo menos 1 portaria no dia
        # --------------------------------------------------
        port_any = port_flags.copy()
 
        port_any["HAS_PORTARIA_ANY"] = (
            port_any["tem_entrada_portaria_dia"].fillna(False).astype(bool)
            | port_any["tem_saida_portaria_dia"].fillna(False).astype(bool)
        )
 
        if cand_day_set:
            port_any = port_any.loc[port_any["_dia_i"].isin(cand_day_set)].copy()
        else:
            port_any = port_any.head(0).copy()
 
        if not port_any.empty:
            port_any["cadastro"] = port_any["cadastro"].astype(str).str.strip()
            port_any = port_any.loc[port_any["cadastro"].ne("")].copy()
 
        any_port_day_map = {
            int(d_i): set(
                g.loc[g["HAS_PORTARIA_ANY"], "cadastro"].astype(str).str.strip().tolist()
            )
            for d_i, g in port_any.groupby("_dia_i", sort=False)
        }
 
        # --------------------------------------------------
        # 8) Índice por dia operacional para busca rápida
        # --------------------------------------------------
        if not ev.empty and not cand.empty:
            day_groups = {}
            ev_sorted = ev.sort_values(["_dia_i", "_sec"], kind="mergesort")
 
            ev_day = ev_sorted["_dia_i"].to_numpy(dtype=np.int64, copy=False)
            ev_sec = ev_sorted["_sec"].to_numpy(dtype=np.int64, copy=False)
            ev_cad = ev_sorted["cadastro"].to_numpy(dtype=object, copy=False)
            ev_pl = ev_sorted["planta"].to_numpy(dtype=float, copy=False)
            ev_col = ev_sorted["coletor"].to_numpy(dtype=object, copy=False)
 
            if ev_day.size > 0:
                grp_start = np.empty(ev_day.size, dtype=bool)
                grp_start[0] = True
                if ev_day.size > 1:
                    grp_start[1:] = ev_day[1:] != ev_day[:-1]
                start_idx = np.flatnonzero(grp_start)
                end_idx = np.r_[start_idx[1:], ev_day.size]
 
                for s, e in zip(start_idx.tolist(), end_idx.tolist()):
                    d_i = int(ev_day[s])
                    day_groups[d_i] = {
                        "sec": ev_sec[s:e],
                        "cad": ev_cad[s:e],
                        "planta": ev_pl[s:e],
                        "coletor": ev_col[s:e],
                    }
 
            janela_seg = int(round(JANELA_TERCEIRO_MIN * 60))
 
            cand_eval = cand[
                [
                    "_idx_day",
                    "_dia_i",
                    "_sec_first",
                    "_sec_last",
                    "cadastro",
                    "planta_first",
                    "coletor_first",
                    "planta_last",
                    "coletor_last",
                ]
            ].to_numpy(copy=False)
 
            hit_first = np.zeros(len(cand_eval), dtype=bool)
            hit_last = np.zeros(len(cand_eval), dtype=bool)
 
            def _has_terceiro(
                cad0: str,
                sec0: int,
                planta0,
                coletor0: str,
                sec_arr: np.ndarray,
                cad_arr: np.ndarray,
                planta_arr: np.ndarray,
                coletor_arr: np.ndarray,
                valid_cads_port_day: set[str],
                exigir_mesma_planta: bool = False,
            ) -> bool:
                l = int(np.searchsorted(sec_arr, sec0 - janela_seg, side="left"))
                r = int(np.searchsorted(sec_arr, sec0 + janela_seg, side="right"))
                if r <= l:
                    return False
 
                coletor0 = str(coletor0).strip()
                if coletor0 == "":
                    return False
 
                for c, pl, col in zip(cad_arr[l:r], planta_arr[l:r], coletor_arr[l:r]):
                    c_str = str(c).strip()
                    col_str = str(col).strip()
 
                    if c_str == cad0:
                        continue
                    if c_str not in valid_cads_port_day:
                        continue
                    if col_str == "":
                        continue
                    if col_str != coletor0:
                        continue
 
                    if exigir_mesma_planta:
                        if pd.isna(planta0) or pd.isna(pl):
                            continue
                        if float(pl) != float(planta0):
                            continue
 
                    return True
 
                return False
 
            for pos, (
                idx_day_i,
                dia_i_val,
                sec_first_val,
                sec_last_val,
                cad_val,
                planta_first,
                coletor_first,
                planta_last,
                coletor_last,
            ) in enumerate(cand_eval):
                d_i = int(dia_i_val)
                dd = day_groups.get(d_i)
                if dd is None:
                    continue
 
                valid_cads_port_day = any_port_day_map.get(d_i, set())
                if not valid_cads_port_day:
                    continue
 
                sec = dd["sec"]
                cads = dd["cad"]
                plantas = dd["planta"]
                coletores = dd["coletor"]
                cad0 = str(cad_val).strip()
 
                has_first = _has_terceiro(
                    cad0,
                    int(sec_first_val),
                    planta_first,
                    coletor_first,
                    sec,
                    cads,
                    plantas,
                    coletores,
                    valid_cads_port_day,
                    exigir_mesma_planta=EXIGIR_MESMA_PLANTA,
                )
 
                has_last = _has_terceiro(
                    cad0,
                    int(sec_last_val),
                    planta_last,
                    coletor_last,
                    sec,
                    cads,
                    plantas,
                    coletores,
                    valid_cads_port_day,
                    exigir_mesma_planta=EXIGIR_MESMA_PLANTA,
                )
 
                if has_first:
                    hit_first[pos] = True
                if has_last:
                    hit_last[pos] = True
 
            if hit_first.any():
                idx_first = np.unique(cand_eval[hit_first, 0].astype(np.int64, copy=False))
                if idx_first.size:
                    day_.loc[idx_first, "HAS_TERCEIRO_PROXIMO_ENTRADA"] = True
 
            if hit_last.any():
                idx_last = np.unique(cand_eval[hit_last, 0].astype(np.int64, copy=False))
                if idx_last.size:
                    day_.loc[idx_last, "HAS_TERCEIRO_PROXIMO_SAIDA"] = True
else:
    day_["PORTARIA_IRREGULAR_DIA"] = False
 
# --------------------------------------------------
# 9) Consolidação da irregularidade
# --------------------------------------------------
if EXIGIR_TERCEIRO_DUAS_PONTAS:
    cond_terceiro = (
        day_["HAS_TERCEIRO_PROXIMO_ENTRADA"].fillna(False).astype(bool)
        & day_["HAS_TERCEIRO_PROXIMO_SAIDA"].fillna(False).astype(bool)
    )
else:
    cond_terceiro = (
        day_["HAS_TERCEIRO_PROXIMO_ENTRADA"].fillna(False).astype(bool)
        | day_["HAS_TERCEIRO_PROXIMO_SAIDA"].fillna(False).astype(bool)
    )
 
day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"].fillna(False).astype(bool))
    & day_["IS_HORISTA_DAY"].fillna(False).astype(bool)
    & day_["first_ponto_pres"].notna()
    & day_["SUSPEITO_SEM_PORTARIA_DIA"].fillna(False).astype(bool)
    & cond_terceiro
)
 
# --------------------------------------------------
# 10) Diagnóstico do funil
# --------------------------------------------------
_diag_base = ((~day_["SEM_COBERTURA_CONTEXTO"]) & day_["IS_HORISTA_DAY"] & day_["first_ponto_pres"].notna()).fillna(False)
_diag_n_base = int(_diag_base.sum())
_diag_n_sem_port = int((_diag_base & day_["SUSPEITO_SEM_PORTARIA_DIA"].fillna(False)).sum())
_diag_n_seq_irreg = int(day_["PORTARIA_SEQUENCIA_IRREGULAR"].fillna(False).sum()) if "PORTARIA_SEQUENCIA_IRREGULAR" in day_.columns else 0
_diag_n_terc_first = int((_diag_base & day_["HAS_TERCEIRO_PROXIMO_ENTRADA"].fillna(False)).sum())
_diag_n_terc_last = int((_diag_base & day_["HAS_TERCEIRO_PROXIMO_SAIDA"].fillna(False)).sum())
_diag_n_terc_cond = int((_diag_base & cond_terceiro.fillna(False)).sum())
_diag_n_final = int(day_[irregularidade_col].fillna(False).sum())
_diag_n_misto = int(day_["DIA_CARGO_MISTO"].fillna(False).sum()) if "DIA_CARGO_MISTO" in day_.columns else 0
 
print(
    f"[DIAG ponto sem portaria] base_horista_com_ponto={_diag_n_base:,} | "
    f"suspeito_sem_portaria_dia={_diag_n_sem_port:,} ({(_diag_n_sem_port / _diag_n_base * 100) if _diag_n_base else 0:.2f}%) | "
    f"seq_portaria_irregular={_diag_n_seq_irreg:,} | "
    f"terceiro_ate_{JANELA_TERCEIRO_MIN:g}min_entrada={_diag_n_terc_first:,} | "
    f"terceiro_ate_{JANELA_TERCEIRO_MIN:g}min_saida={_diag_n_terc_last:,} | "
    f"terceiro_condicao={_diag_n_terc_cond:,} | "
    f"final={_diag_n_final:,} | dias_cargo_misto={_diag_n_misto:,} | "
    f"exigir_mesma_planta={EXIGIR_MESMA_PLANTA} | "
    f"duas_pontas={EXIGIR_TERCEIRO_DUAS_PONTAS}"
)
 
# --------------------------------------------------
# 11) Exportação
# --------------------------------------------------
FORCAR_RESUMO_DIA = True
 
if FORCAR_RESUMO_DIA:
    cols_resumo = [
        "cadastro",
        "_dia_operacional",
        "first_ponto_pres",
        "last_ponto_pres",
        "SUSPEITO_SEM_PORTARIA_DIA",
        "HAS_TERCEIRO_PROXIMO_ENTRADA",
        "HAS_TERCEIRO_PROXIMO_SAIDA",
    ]
    cols_resumo = [c for c in cols_resumo if c in day_.columns]
 
    df_ponto_sem_portaria = day_.loc[day_[irregularidade_col], cols_resumo].copy(deep=False)
 
    if "_dia_operacional" in df_ponto_sem_portaria.columns:
        df_ponto_sem_portaria["dia_operacional"] = pd.to_datetime(
            df_ponto_sem_portaria["_dia_operacional"], errors="coerce"
        )
 
    df_ponto_sem_portaria[irregularidade_col] = True
 
    slug = slugify(irregularidade_col)
    parquet_path = DFS_DIR / f"{slug}.parquet"
    pdf_path = PDF_DIR / f"{slug}.pdf"
 
    df_ponto_sem_portaria.to_parquet(parquet_path, index=False)
    gerar_relatorio_pdf(df_ponto_sem_portaria, titulo=titulo, pdf_path=pdf_path)
 
    print("Parquet:", parquet_path)
    print("PDF   :", pdf_path)
else:
    keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
    try:
        row_ids = row_ids_from_keys(df, keys)
        df_ponto_sem_portaria = materializar_e_exportar_por_rowid(
            row_ids,
            irregularidade_col=irregularidade_col,
            titulo=titulo,
        )
    except MemoryError:
        print("WARN: fallback de memoria acionado no mapeamento de row_id; gerando PDF/parquet resumidos por dia.")
        df_ponto_sem_portaria = day_.loc[day_[irregularidade_col]].copy(deep=False)
 
        if "_dia_operacional" in df_ponto_sem_portaria.columns:
            df_ponto_sem_portaria["dia_operacional"] = pd.to_datetime(
                df_ponto_sem_portaria["_dia_operacional"], errors="coerce"
            )
 
        df_ponto_sem_portaria[irregularidade_col] = True
 
        slug = slugify(irregularidade_col)
        parquet_path = DFS_DIR / f"{slug}.parquet"
        pdf_path = PDF_DIR / f"{slug}.pdf"
 
        df_ponto_sem_portaria.to_parquet(parquet_path, index=False)
        gerar_relatorio_pdf(df_ponto_sem_portaria, titulo=titulo, pdf_path=pdf_path)
 
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
 
if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ponto_sem_portaria = df_ponto_sem_portaria
 
df_ponto_sem_portaria.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Registro sem Portaria
# =========================
irregularidade_col = "Irregularidade - Registro sem Portaria"
titulo = "Irregularidade – Registro sem Portaria"

day_ = day.copy()
day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])          # FIX: não incluir executivos (cargo == 'E')
    & (day_["SEM_PORTARIA_COERENTE"])
    & (day_["first_any_pres"].notna())
)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_registro_sem_portaria = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_registro_sem_portaria = df_registro_sem_portaria

df_registro_sem_portaria.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Executivos com Histórico de 999 como trajeto (redefinida)
# =========================
# Nova definição:
#   "Executivos (cargo E) que, ANTES de virarem executivos, cometiam a irregularidade '999 como Trajeto'."
#
# A saída retorna os DIAS PRÉ-PROMOÇÃO em que a mesma lógica da irregularidade 999 como trajeto é verdadeira.

irregularidade_col = "Irregularidade - Executivos com Histórico de 999 como Trajeto"
titulo = "Irregularidade – Executivos com histórico de 999 como Trajeto (pré promoção)"

# Data de transição para cargo E (primeira ocorrência de cargo E no histórico)
df_exec = df_valid.loc[df_valid["_is_executivo"].fillna(False), ["cadastro", "_dt_ronda"]].copy()
promo = (
    df_exec.groupby("cadastro")["_dt_ronda"].min()
          .rename("dt_promocao")
          .reset_index()
)

# Dia operacional da promoção (mesma lógica do corte operacional)
promo["dia_promocao_op"] = (promo["dt_promocao"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

# Day base apenas para quem virou executivo
day_ = day.merge(promo, on="cadastro", how="inner")

# =========================
# Recalcula a irregularidade 999 (mesma lógica do bloco 999)
# =========================

# Base: precisa ter 999 no dia
day999_counts = (
    df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional"]]
      .groupby(["cadastro", "_dia_operacional"])
      .size()
      .rename("n_999")
      .reset_index()
)

day_ = day_.merge(day999_counts, on=["cadastro", "_dia_operacional"], how="left")
day_["n_999"] = day_["n_999"].fillna(0).astype(int)

# Anti-truncamento específico da janela 999
day_["ANTI_TRUNC_999"] = False
mask_mm = day_["min_dt_cad"].notna() & day_["max_dt_cad"].notna()
day_.loc[mask_mm, "ANTI_TRUNC_999"] = (
    (day_.loc[mask_mm, "first_any_pres"] - t_jan_999) < day_.loc[mask_mm, "min_dt_cad"]
) | (
    (day_.loc[mask_mm, "last_any_pres"] + t_jan_999) > day_.loc[mask_mm, "max_dt_cad"]
)

# Conta 999 antes/depois do presencial
ev_999 = df_valid.loc[df_valid["999"].fillna(False), ["cadastro", "_dia_operacional", "_dt_ronda"]].copy()
ev_999 = ev_999.merge(
    day_.loc[:, ["cadastro", "_dia_operacional", "first_any_pres", "last_any_pres"]],
    on=["cadastro", "_dia_operacional"],
    how="inner",
)

ev_999["in_before"] = (ev_999["_dt_ronda"] >= (ev_999["first_any_pres"] - t_jan_999)) & (ev_999["_dt_ronda"] < ev_999["first_any_pres"])
ev_999["in_after"]  = (ev_999["_dt_ronda"] >  ev_999["last_any_pres"]) & (ev_999["_dt_ronda"] <= (ev_999["last_any_pres"] + t_jan_999))

cnt = (
    ev_999.groupby(["cadastro", "_dia_operacional"], as_index=False)
          .agg(cnt_before=("in_before", "sum"), cnt_after=("in_after", "sum"))
)

day_ = day_.merge(cnt, on=["cadastro", "_dia_operacional"], how="left")
day_[["cnt_before", "cnt_after"]] = day_[["cnt_before", "cnt_after"]].fillna(0).astype(int)

odd_before = (day_["cnt_before"] % 2 == 1) & (day_["cnt_before"] > 0)
odd_after  = (day_["cnt_after"] % 2 == 1) & (day_["cnt_after"] > 0)

flag_999_irregularidade = (
    (day_["n_999"] > 0)
    & day_["HAS_PORTARIA_ANY"]
    & (odd_before | odd_after)
    & (~day_["ANTI_TRUNC_999"])
)

# =========================
# Irregularidade executivos: somente pré promoção
# =========================
dia_op = pd.to_datetime(day_["_dia_operacional"], errors="coerce").to_numpy(dtype="datetime64[ns]")
dia_prom_op = pd.to_datetime(day_["dia_promocao_op"], errors="coerce").to_numpy(dtype="datetime64[ns]")

day_[irregularidade_col] = (
    day_["dt_promocao"].notna().to_numpy()
    & (dia_op < dia_prom_op)
    & flag_999_irregularidade.to_numpy()
)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

# Materialização do df_ronda filtrado pelo df temporário (keys)
row_ids = row_ids_from_keys(df, keys)
df_exec_hist_999 = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ponto_de_executivos = df_exec_hist_999

df_exec_hist_999.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Ausência de Descanso entre Jornadas de Trabalho
# Regra: descanso mínimo 11h até a próxima jornada (usa Portaria E->S com fallback any_pres).
# Mantém também o dia anterior usado no cálculo.
# =========================
irregularidade_col = "Irregularidade - Ausência de Descanso entre Jornadas de Trabalho"
titulo = "Irregularidade – Ausência de Descanso entre Jornadas (mín. 11h)"

t_desc_min = pd.Timedelta(hours=11)

day_ = day.sort_values(["cadastro", "_dia_operacional"], kind="mergesort").copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

day_["j_start"] = day_["first_port_E"].where(day_["first_port_E"].notna(), day_["first_any_pres"])
day_["j_end"]   = day_["last_port_S"].where(day_["last_port_S"].notna(),  day_["last_any_pres"])

day_["prev_dia_operacional"] = day_.groupby("cadastro")["_dia_operacional"].shift(1)
day_["prev_j_end"] = day_.groupby("cadastro")["j_end"].shift(1)
day_["prev_dur"]   = day_.groupby("cadastro")["dur_any_pres"].shift(1)
day_["prev_sem_cob"] = day_.groupby("cadastro")["SEM_COBERTURA_CONTEXTO"].shift(1).fillna(False)

descanso = day_["j_start"] - day_["prev_j_end"]

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["prev_sem_cob"])
    & (day_["prev_j_end"].notna())
    & (day_["j_start"].notna())
    & (descanso < t_desc_min)
    & (day_["prev_dur"] >= t_jornada_min)
).fillna(False)

keys_main = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
keys_prev = (
    day_.loc[day_[irregularidade_col], ["cadastro", "prev_dia_operacional"]]
        .rename(columns={"prev_dia_operacional": "_dia_operacional"})
        .dropna(subset=["_dia_operacional"])
)
keys = pd.concat([keys_main, keys_prev], ignore_index=True).drop_duplicates()

row_ids = row_ids_from_keys(df, keys)
df_descanso = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ausencia_descanso = df_descanso

df_descanso.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Pessoas Trabalhando Além da Jornada de Trabalho
# Regra: presença na planta fora do intervalo Portaria E->S por >=2h.
# Não contar executivos. Não acusar em borda de contexto.
# =========================
irregularidade_col = "Irregularidade - Pessoas Trabalhando Além da Jornada de Trabalho"
titulo = "Irregularidade – Presencial fora da Portaria (>= 2h)"

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

before_gap = day_["first_port_E"] - day_["first_any_pres"]
after_gap  = day_["last_any_pres"] - day_["last_port_S"]

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])
    & (day_["first_any_pres"].notna())
    & (day_["last_any_pres"].notna())
    & (day_["first_port_E"].notna())
    & (day_["last_port_S"].notna())
    & ((before_gap >= t_gap_work) | (after_gap >= t_gap_work))
).fillna(False)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)
df_alem_jornada = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_alem_jornada = df_alem_jornada

df_alem_jornada.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Pessoas Trabalhando sem Ganhar
# Regra: ficou na fábrica >=2h (Portaria E->S com fallback any_pres) e NÃO bateu ponto.
# Não contar executivos. Não acusar em borda de contexto.
# =========================
irregularidade_col = "Irregularidade - Pessoas Trabalhando sem Ganhar"
titulo = "Irregularidade – Trabalhando sem Ganhar (>=2h na fábrica e sem ponto)"

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

if port.empty:
    day_["first_port_E"] = pd.NaT
    day_["last_port_S"]  = pd.NaT
else:
    first_E = (
        port.loc[port["_dir_norm"].eq("E")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .min()
            .rename("first_port_E")
            .reset_index()
    )
    last_S = (
        port.loc[port["_dir_norm"].eq("S")]
            .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
            .max()
            .rename("last_port_S")
            .reset_index()
    )
    day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
    day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

dur_port = day_["last_port_S"] - day_["first_port_E"]
dur_fabrica = dur_port.where(dur_port.notna() & (dur_port >= pd.Timedelta(0)), day_["dur_any_pres"])

day_[irregularidade_col] = (
    (~day_["SEM_COBERTURA_CONTEXTO"])
    & (~day_["IS_EXECUTIVO_DAY"])
    & (dur_fabrica >= t_gap_work)  # t_gap_work = 2h
    & (~day_["HAS_PONTO_ANY"])
).fillna(False)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)
df_sem_ganhar = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_sem_ganhar = df_sem_ganhar

df_sem_ganhar.head(3)

In [ ]:
# =========================
# IRREGULARIDADE: Registros de Pessoas Desligadas (se existir data de desligado)
# =========================
irregularidade_col = "Irregularidade - Registros de Pessoas Desligadas"
titulo = "Irregularidade – Registros de Pessoas Desligadas"

# Descobre coluna de desligado SEM carregar o parquet inteiro
dem_col = None
for cand in ["Data_Dem", "data_desligado", "DataDemissao", "data_demissao"]:
    try:
        tmp = pd.read_parquet(PARQUET_PATH, columns=["cadastro", cand])
        dem_col = cand
        break
    except Exception:
        continue

# limpa cache de row_id/hash para reduzir pico de memória nesta etapa
for _v in ("_ROWID_CACHE_HASH", "_ROWID_CACHE_HASH_INDEX", "_ROWID_CACHE_ROWID"):
    if _v in globals():
        del globals()[_v]
gc.collect()

day_ = day.loc[:, ["cadastro", "_dia_operacional"]]

if dem_col is None:
    print("⚠️ Nenhuma coluna de data de desligado encontrada. Irregularidade ficará vazia.")
    day_[irregularidade_col] = False
else:
    dem = tmp.copy()
    dem["cadastro"] = dem["cadastro"].astype(str).str.strip()
    dem[dem_col] = pd.to_datetime(dem[dem_col], dayfirst=True, errors="coerce")
    dem = dem.loc[dem[dem_col].notna()].copy()
    afast_map = dem.groupby("cadastro")[dem_col].max().rename("dt_dem").reset_index()

    day_ = day_.merge(afast_map, on="cadastro", how="left")
    day_[irregularidade_col] = day_["dt_dem"].notna() & (pd.to_datetime(day_["_dia_operacional"], errors="coerce") > day_["dt_dem"])

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)
df_desligados = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_desligados = df_desligados

df_desligados.head(3)


In [ ]:
# =========================
# Copiando base
# =========================
df = df_min_raw.copy()

# cadastro como string
df["cadastro"] = df["cadastro"].astype(str).str.strip()

# descrição (normalização robusta para reduzir falso negativo com acentuação)
desc = df["descricao coletor"].fillna("").astype(str)
desc_norm = (
    desc.str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
        .str.upper()
)

df["Portaria"] = desc.str.contains(regex_portaria, case=False, na=False, regex=True)
df["Ponto"] = desc.str.contains(regex_ponto, case=False, na=False, regex=True)

mask_grill = desc_norm.str.contains(r"\bGRILL\b", na=False, regex=True)
mask_ref_base = (
    desc_norm.str.contains(r"\bREFEIT(?:ORIO)?\b", na=False, regex=True)
    | desc_norm.str.contains(r"\bREF[.\s-]*ALA\b", na=False, regex=True)
)

df["_GRILL"] = mask_grill
df["_REFEITORIO"] = mask_ref_base & (~mask_grill)

# planta == 999 (robusto)
planta_num = pd.to_numeric(df["planta"], errors="coerce")
df["999"] = planta_num.eq(999)

# Presencial: planta válida e não 999
df["Presencial"] = planta_num.notna() & (~df["999"])

# Executivo: cargo == 'E'
cargo_norm = df["cargo"].fillna("").astype(str).str.strip().str.upper()
df["_is_executivo"] = cargo_norm.eq("E")

# dir normalizado
dir_norm = df["dir"].fillna("").astype(str).str.strip().str.upper()
dir_norm = dir_norm.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
dir_norm = dir_norm.where(~dir_norm.str.startswith("E"), "E")
dir_norm = dir_norm.where(~dir_norm.str.startswith("S"), "S")
df["_dir_norm"] = dir_norm

# dt_ronda robusto (data + hora)
data_dt = pd.to_datetime(df["data"], dayfirst=True, errors="coerce")
hora_s = df["hora"].fillna("00:00").astype(str).str.strip()
hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
hora_td = pd.to_timedelta(hora_s, errors="coerce")

df["_dia_key"] = data_dt.dt.normalize()
df["_dt_ronda"] = df["_dia_key"] + hora_td

# Dia operacional base (corte às 11h)
df["_dia_operacional"] = (df["_dt_ronda"] - pd.Timedelta(hours=hora_corte_operacional)).dt.normalize()

print("OK: pré-processamento mínimo aplicado.")
df.head(3)

# =========================
# Ajuste de dia operacional por jornada (Portaria E->S) - versão otimizada
# =========================
if usar_jornada_por_portaria:
    cols_min = ["cadastro", "_dt_ronda", "Portaria", "_dir_norm", "999", "_dia_operacional"]
    df_min = df[cols_min].copy()

    df_sorted = (
        df_min.sort_values(["cadastro", "_dt_ronda"], kind="mergesort")
              .reset_index(drop=False)
    )

    orig_idx = df_sorted["index"].to_numpy()
    times    = df_sorted["_dt_ronda"].to_numpy()
    port     = df_sorted["Portaria"].fillna(False).to_numpy(dtype=bool)
    dirn     = df_sorted["_dir_norm"].to_numpy()
    is999    = df_sorted["999"].fillna(False).to_numpy(dtype=bool)

    dia_op = df_sorted["_dia_operacional"].to_numpy(copy=True)
    if not dia_op.flags.writeable:
        dia_op = dia_op.copy()
        dia_op.setflags(write=True)

    tmax = np.timedelta64(int(pd.Timedelta(t_max_jornada_port).value), "ns")
    tjan = np.timedelta64(int(pd.Timedelta(t_jan_999).value), "ns")

    grp_pos = df_sorted.groupby("cadastro", sort=False).indices

    for cad, pos in grp_pos.items():
        t = times[pos]
        valid = ~np.isnat(t)
        if not valid.any():
            continue

        posv  = pos[valid]
        tv    = t[valid]
        portv = port[posv]
        dirv  = dirn[posv]
        is9v  = is999[posv]

        m_ent = portv & (dirv == "E")
        m_sai = portv & (dirv == "S")

        ent = tv[m_ent]
        sai = tv[m_sai]

        if ent.size == 0 or sai.size == 0:
            continue

        i = j = 0
        while i < ent.size and j < sai.size:
            dt_ent = ent[i]
            dt_sai = sai[j]

            if dt_sai < dt_ent:
                j += 1
                continue

            if (dt_sai - dt_ent) > tmax:
                i += 1
                continue

            dia = dt_ent.astype("datetime64[D]").astype("datetime64[ns]")

            l = np.searchsorted(tv, dt_ent, side="left")
            r = np.searchsorted(tv, dt_sai, side="right")
            if r > l:
                dia_op[posv[l:r]] = dia

            l2 = np.searchsorted(tv, dt_ent - tjan, side="left")
            r2 = np.searchsorted(tv, dt_ent, side="left")
            if r2 > l2:
                sl = slice(l2, r2)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            l3 = np.searchsorted(tv, dt_sai, side="right")
            r3 = np.searchsorted(tv, dt_sai + tjan, side="right")
            if r3 > l3:
                sl = slice(l3, r3)
                m9 = is9v[sl]
                if m9.any():
                    dia_op[posv[sl][m9]] = dia

            i += 1
            j += 1

    s = pd.Series(dia_op, index=orig_idx, name="_dia_operacional").reindex(df.index)
    df["_dia_operacional"] = s.to_numpy()

for _v in ("df_min", "df_sorted", "orig_idx", "times", "port", "dirn", "is999", "dia_op", "grp_pos"):
    if _v in globals():
        del globals()[_v]
gc.collect()

print("OK: _dia_operacional ajustado por jornada (Portaria E->S) [otimizado].")

# =========================
# Considera apenas registros com dt válido (sem cópia profunda)
# =========================
mask_dt = df["_dt_ronda"].notna()

COLS_KEEP = [
    "cadastro", "_dia_operacional", "_dt_ronda",
    "_dia_key",  # <<< ALTERADO: necessário para Semana Feriado Prolongado
    "Presencial", "Ponto", "Portaria",
    "_REFEITORIO", "_GRILL", "_is_executivo",
    "_dir_norm",
    "999",
]

for _extra_col in ["nome", "área", "area"]:
    if (_extra_col in df.columns) and (_extra_col not in COLS_KEEP):
        COLS_KEEP.append(_extra_col)

df_valid = df.loc[mask_dt, COLS_KEEP]

cad_minmax = (
    df_valid.groupby("cadastro", sort=False)["_dt_ronda"]
      .agg(min_dt_cad="min", max_dt_cad="max")
      .reset_index()
)

pres_any = df_valid.loc[df_valid["Presencial"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda"]]
day_any = (
    pres_any.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
            .agg(
                first_any_pres=("_dt_ronda", "min"),
                last_any_pres=("_dt_ronda", "max"),
                n_any_pres=("_dt_ronda", "size"),
            )
)
day_any["dur_any_pres"] = day_any["last_any_pres"] - day_any["first_any_pres"]

pres_ponto = df_valid.loc[
    df_valid["Ponto"].eq(True) & df_valid["Presencial"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda"]
]
day_ponto = (
    pres_ponto.groupby(["cadastro", "_dia_operacional"], as_index=False, sort=False)
              .agg(first_ponto_pres=("_dt_ronda", "min"), last_ponto_pres=("_dt_ronda", "max"))
)

day_has_ponto_any = (
    df_valid.loc[df_valid["Ponto"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PONTO_ANY=True)
)

day_has_port = (
    df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_PORTARIA_ANY=True)
)

day_has_ref = (
    df_valid.loc[df_valid["_REFEITORIO"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_REFEITORIO_ANY=True)
)

day_has_grill = (
    df_valid.loc[df_valid["_GRILL"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(HAS_GRILL_ANY=True)
)

day_is_exec = (
    df_valid.loc[df_valid["_is_executivo"].eq(True), ["cadastro", "_dia_operacional"]]
      .drop_duplicates()
      .assign(IS_EXECUTIVO_DAY=True)
)

day = day_any.merge(day_ponto, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ponto_any, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_port, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_ref, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_has_grill, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(day_is_exec, on=["cadastro", "_dia_operacional"], how="left")
day = day.merge(cad_minmax, on="cadastro", how="left")

for c in ["HAS_PONTO_ANY", "HAS_PORTARIA_ANY", "HAS_REFEITORIO_ANY", "HAS_GRILL_ANY", "IS_EXECUTIVO_DAY"]:
    day[c] = day[c].fillna(False).astype(bool)

day["w_start_needed"] = day["first_any_pres"] - t_jan_port
day["w_end_needed"] = day["last_any_pres"] + t_jan_port
day["SEM_COBERTURA_CONTEXTO"] = False
mask_mm = day["min_dt_cad"].notna() & day["max_dt_cad"].notna()
day.loc[mask_mm, "SEM_COBERTURA_CONTEXTO"] = (day["w_start_needed"] < day["min_dt_cad"]) | (day["w_end_needed"] > day["max_dt_cad"])

port = df_valid.loc[df_valid["Portaria"].eq(True), ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]].copy(deep=False)
if port.empty:
    day["ok_entry"] = False
    day["ok_exit"] = False
else:
    port = port.merge(
        day[["cadastro", "_dia_operacional", "w_start_needed", "w_end_needed", "first_any_pres", "last_any_pres"]],
        on=["cadastro", "_dia_operacional"],
        how="inner",
    )

    in_window = (port["_dt_ronda"] >= port["w_start_needed"]) & (port["_dt_ronda"] <= port["w_end_needed"])

    port["ok_entry"] = (
        in_window
        & port["_dir_norm"].eq("E")
        & (port["_dt_ronda"] <= (port["first_any_pres"] + t_tol_ent))
    )

    port["ok_exit"] = (
        in_window
        & port["_dir_norm"].eq("S")
        & (port["_dt_ronda"] >= (port["last_any_pres"] - t_tol_sai))
    )

    ok = port.groupby(["cadastro", "_dia_operacional"], sort=False)[["ok_entry", "ok_exit"]].any().reset_index()
    day = day.merge(ok, on=["cadastro", "_dia_operacional"], how="left")
    day[["ok_entry", "ok_exit"]] = day[["ok_entry", "ok_exit"]].fillna(False)

day["SEM_PORTARIA_COERENTE"] = np.where(
    day["IS_EXECUTIVO_DAY"],
    ~day["HAS_PORTARIA_ANY"],
    ~(day["ok_entry"] | day["ok_exit"]),
)

day.loc[day["SEM_COBERTURA_CONTEXTO"], "SEM_PORTARIA_COERENTE"] = False

# =========================
# Regra grill/refeitório — REFEITÓRIO NORMAL NO MESMO DIA
# =========================
day["HAS_REFEITORIO_NORMAL_NO_DIA"] = day["HAS_REFEITORIO_ANY"].fillna(False).astype(bool)

# Compatibilidade com blocos antigos que ainda leem a flag antiga.
day["HAS_REFEITORIO_BEFORE_GRILL"] = day["HAS_REFEITORIO_NORMAL_NO_DIA"]

day["Irregularidade - Não entrou no refeitório, mas passou no grill"] = (
    day["HAS_GRILL_ANY"].fillna(False).astype(bool)
    & (~day["HAS_REFEITORIO_NORMAL_NO_DIA"])
)

print("OK: day construído.")
day.head(3)

# =========================
# Helpers (igual ao seu)
# =========================
def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text.lower()

def _pick_col(df_: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_.columns:
            return c
    return None


def gerar_relatorio_pdf(df_irregularidade: pd.DataFrame, titulo: str, pdf_path: Path) -> None:
    """
    Relatório enxuto e analítico.
    - Capa com data de geração + métricas + percentuais (Top 1/3/10 e recência)
    - Remove tabelas/listagens
    - Gerais: tendência mensal, DOW, e 3 gráficos separados (áreas, plantas, áreas|planta)
    - Top 3: rankings (quantidade, percentual, atualidade) + 4 gráficos por pessoa
      (mensal, DOW, ocorrências diárias, vazio)
    - Caso especial (999 trajeto): gráfico de "concentração" (Antes/Depois/Ambos) por pessoa com 3 bolinhas
    - Otimizações: trabalha com subconjuntos de colunas e evita cópias profundas grandes
    """
    import datetime as _dt
    import numpy as np

    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 150,
        "font.size": 12,
        "axes.titlesize": 18,
        "axes.labelsize": 13,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    })

    def _wrap(s: str, width: int = 44) -> str:
        s = "" if s is None else str(s)
        s = s.strip()
        if len(s) <= width:
            return s
        words = s.split()
        out, line = [], ""
        for w in words:
            if len(line) + len(w) + 1 <= width:
                line = (line + " " + w).strip()
            else:
                if line:
                    out.append(line)
                line = w
        if line:
            out.append(line)
        if len(out) <= 4:
            return "\n".join(out)
        return "\n".join(out[:4]) + "…"

    def _as_series(df_: pd.DataFrame, col: str) -> pd.Series:
        x = df_.loc[:, col]
        if isinstance(x, pd.DataFrame):
            x = x.iloc[:, 0]
        return x

    def _pick_col_local(df_: pd.DataFrame, candidates: list[str]) -> str | None:
        for c in candidates:
            if c in df_.columns:
                return c
        return None

    def _annotate_barh(ax, values, total_base: int | None = None, fs: int = 9):
        # values: array aligned com barras (na ordem do ax)
        for i, v in enumerate(values):
            v = float(v)
            if v <= 0:
                continue
            if total_base and total_base > 0:
                ax.text(v, i, f"  {int(v)} ({(v/total_base):.1%})", va="center", fontsize=fs)
            else:
                ax.text(v, i, f"  {int(v)}", va="center", fontsize=fs)

    def _annotate_bar(ax, values, total_base: int | None = None, fs: int = 9):
        for i, v in enumerate(values):
            v = float(v)
            if v <= 0:
                continue
            if total_base and total_base > 0:
                ax.text(i, v, f"{int(v)}\n({(v/total_base):.1%})", ha="center", va="bottom", fontsize=fs)
            else:
                ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=fs)

    gen_date = _dt.date.today().strftime("%Y-%m-%d")

    # Caso vazio
    if df_irregularidade is None or df_irregularidade.empty:
        with PdfPages(pdf_path) as pdf:
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            ax.axis("off")
            ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
            ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")
            ax.text(0.02, 0.84, "Sem linhas para gerar o relatório.", fontsize=14, va="top")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        return

    # remove colunas duplicadas sem deep copy
    if getattr(df_irregularidade, "columns", None) is not None:
        df_irregularidade = df_irregularidade.loc[:, ~df_irregularidade.columns.duplicated()]

    # Colunas
    col_cad = _pick_col_local(df_irregularidade, ["cadastro"])
    col_nome = _pick_col_local(df_irregularidade, ["nome"])
    col_areas = "área" if "área" in df_irregularidade.columns else None
    col_area_fallback = _pick_col_local(df_irregularidade, ["area"])  # fallback real
    col_planta = "planta" if "planta" in df_irregularidade.columns else None

    # Determina _dia_operacional
    if "_dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "_dia_operacional"), errors="coerce")
    elif "dia_operacional" in df_irregularidade.columns:
        dia_op = pd.to_datetime(_as_series(df_irregularidade, "dia_operacional"), errors="coerce")
    else:
        data_dt = (
            pd.to_datetime(_as_series(df_irregularidade, "data"), dayfirst=True, errors="coerce")
            if "data" in df_irregularidade.columns
            else pd.Series(pd.NaT, index=df_irregularidade.index)
        )
        if "hora" not in df_irregularidade.columns:
            dia_op = data_dt.dt.normalize()
        else:
            hora_s = _as_series(df_irregularidade, "hora").fillna("00:00").astype(str).str.strip()
            hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
            hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
            hora_td = pd.to_timedelta(hora_s, errors="coerce")
            dia_op = (data_dt.dt.normalize() + hora_td - pd.Timedelta(hours=11)).dt.normalize()

    # subset mínimo para métricas gerais (evita carregar tudo)
    cols_keep = ["_dia_operacional"]
    if col_cad: cols_keep.append(col_cad)
    if col_nome: cols_keep.append(col_nome)
    if col_areas: cols_keep.append(col_areas)
    elif col_area_fallback: cols_keep.append(col_area_fallback)
    if col_planta: cols_keep.append(col_planta)

    dfv = df_irregularidade.loc[:, [c for c in cols_keep if c != "_dia_operacional"]].copy()
    dfv["_dia_operacional"] = pd.to_datetime(dia_op, errors="coerce")

    if col_cad and (col_cad in dfv.columns):
        dfv[col_cad] = dfv[col_cad].astype(str).str.strip()
        dfv = dfv[dfv[col_cad].ne("")]

    key_cols = [col_cad, "_dia_operacional"] if (col_cad and col_cad in dfv.columns) else ["_dia_operacional"]

    # ocorrência = pessoa-dia
    df_occ = (
        dfv.dropna(subset=["_dia_operacional"])
           .drop_duplicates(subset=key_cols, keep="first")
    )

    if df_occ.empty:
        with PdfPages(pdf_path) as pdf:
            fig = plt.figure(figsize=(11.69, 8.27))
            ax = fig.add_subplot(111)
            ax.axis("off")
            ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
            ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")
            ax.text(0.02, 0.84, "Sem ocorrências após normalização de _dia_operacional.", fontsize=14, va="top")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        return

    total = int(len(df_occ))
    n_pessoas = int(df_occ[col_cad].nunique()) if (col_cad and col_cad in df_occ.columns) else 0
    n_dias = int(df_occ["_dia_operacional"].nunique())
    dt_min = df_occ["_dia_operacional"].min()
    dt_max = df_occ["_dia_operacional"].max()

    # nome_map por cadastro (mais frequente)
    nome_map = {}
    if col_cad and col_nome and (col_cad in dfv.columns) and (col_nome in dfv.columns):
        tmp = dfv[[col_cad, col_nome]].copy()
        tmp[col_nome] = tmp[col_nome].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[col_nome].ne("")]
        if not tmp.empty:
            cnt = (
                tmp.groupby([col_cad, col_nome], sort=False)
                   .size()
                   .reset_index(name="n")
                   .sort_values([col_cad, "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map = cnt.drop_duplicates(subset=[col_cad], keep="first").set_index(col_cad)[col_nome].to_dict()

    # labels de pessoa (cadastro + nome)
    if col_cad and (col_cad in df_occ.columns):
        cad_series = df_occ[col_cad].astype(str).str.strip()
        pessoa_label = cad_series.map(lambda c: f"{c} - {nome_map.get(c,'')}".strip(" -")) if nome_map else cad_series
        top_pessoas = pessoa_label.value_counts()
    else:
        top_pessoas = pd.Series(dtype=int)

    # Top 1/3/10
    top1_n = int(top_pessoas.head(1).sum()) if (not top_pessoas.empty) else 0
    top3_n = int(top_pessoas.head(3).sum()) if (not top_pessoas.empty) else 0
    top10_n = int(top_pessoas.head(10).sum()) if (not top_pessoas.empty) else 0
    top1_pct = (top1_n / total) if total else 0.0
    top3_pct = (top3_n / total) if total else 0.0
    top10_pct = (top10_n / total) if total else 0.0

    # Mensal / DOW (gerais)
    dia_op_np = pd.Series(
        pd.to_datetime(df_occ["_dia_operacional"], errors="coerce").to_numpy(dtype="datetime64[ns]"),
        index=df_occ.index,
        name="_dia_operacional",
    )

    mes = dia_op_np.dt.to_period("M").dt.to_timestamp()
    mensal = mes.value_counts().sort_index()

    dow_map = {0: "Seg", 1: "Ter", 2: "Qua", 3: "Qui", 4: "Sex", 5: "Sáb", 6: "Dom"}
    dow = (
        df_occ["_dia_operacional"].dt.dayofweek.map(dow_map)
            .value_counts()
            .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
            .fillna(0).astype(int)
    )

    # Recência (atuais): últimos 30 dias (ou menor se período curto)
    dt_max_norm = pd.to_datetime(dt_max).normalize()
    window_days = 30
    if pd.notna(dt_min) and pd.notna(dt_max):
        span = (pd.to_datetime(dt_max) - pd.to_datetime(dt_min)).days
        if span < 45:
            window_days = 14
        if span < 21:
            window_days = 7

    dt_cut = dt_max_norm - pd.Timedelta(days=window_days)
    recent = df_occ.loc[df_occ["_dia_operacional"] >= dt_cut]
    total_recent = int(len(recent))
    if col_cad and (col_cad in df_occ.columns) and total_recent > 0:
        cad_recent = recent[col_cad].astype(str).str.strip()
        label_recent = cad_recent.map(lambda c: f"{c} - {nome_map.get(c,'')}".strip(" -")) if nome_map else cad_recent
        top_recent = label_recent.value_counts()
    else:
        top_recent = pd.Series(dtype=int)

    # Top 3 listas
    top3_qty = top_pessoas.head(3) if not top_pessoas.empty else pd.Series(dtype=int)
    top3_pct_ser = (top3_qty / total) if (total and not top3_qty.empty) else pd.Series(dtype=float)

    top3_recent = top_recent.head(3) if not top_recent.empty else pd.Series(dtype=int)
    top3_recent_pct = (top3_recent / total_recent) if (total_recent and not top3_recent.empty) else pd.Series(dtype=float)

    # ---------- Funções de representatividade (áreas / plantas / combo) por pessoa-dia
    def _rep_value_per_occ(df_in: pd.DataFrame, key_cols2: list[str], val_col: str) -> pd.Series:
        if (val_col is None) or (val_col not in df_in.columns):
            return pd.Series(dtype=str)
        tmp = df_in[key_cols2 + [val_col]].copy()
        tmp[val_col] = tmp[val_col].fillna("").astype(str).str.strip()
        tmp = tmp[tmp[val_col].ne("")]
        if tmp.empty:
            return pd.Series(dtype=str)
        cnt = (
            tmp.groupby(key_cols2 + [val_col], sort=False)
               .size()
               .reset_index(name="n")
               .sort_values(key_cols2 + ["n"], ascending=[True]*len(key_cols2) + [False], kind="mergesort")
        )
        rep = cnt.drop_duplicates(subset=key_cols2, keep="first").set_index(key_cols2)[val_col]
        return rep

    # área: prefer 'área', senão 'area'
    area_col_use = col_areas if col_areas else col_area_fallback
    area_rep = pd.Series(dtype=str)
    if area_col_use and col_cad and (col_cad in dfv.columns):
        area_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], area_col_use)

    # planta
    planta_rep = pd.Series(dtype=str)
    if col_planta and col_cad and (col_cad in dfv.columns):
        planta_rep = _rep_value_per_occ(dfv, [col_cad, "_dia_operacional"], col_planta)

    # combo
    combo_rep = pd.Series(dtype=str)
    if (not area_rep.empty) and (not planta_rep.empty):
        # constrói combo por ocorrência (índice MultiIndex)
        idx = area_rep.index.intersection(planta_rep.index)
        if len(idx) > 0:
            combo_rep = (area_rep.loc[idx].astype(str) + " | " + planta_rep.loc[idx].astype(str))

    top_areas = area_rep.value_counts() if not area_rep.empty else pd.Series(dtype=int)
    top_plantas = planta_rep.value_counts() if not planta_rep.empty else pd.Series(dtype=int)
    top_combo = combo_rep.value_counts() if not combo_rep.empty else pd.Series(dtype=int)

    # ---------- Top 3 pessoas (para painéis): por quantidade total
    top3_labels = list(top3_qty.index) if not top3_qty.empty else []
    top3_cads = [str(lbl).split(" - ", 1)[0].strip() for lbl in top3_labels]

    # ---------- PDF
    with PdfPages(pdf_path) as pdf:
        # ---------------- CAPA
        fig = plt.figure(figsize=(11.69, 8.27))
        ax = fig.add_subplot(111)
        ax.axis("off")

        ax.text(0.02, 0.96, titulo, fontsize=24, fontweight="bold", va="top")
        ax.text(0.02, 0.91, f"Gerado em: {gen_date}", fontsize=12, va="top")

        ax.text(0.02, 0.84, f"Ocorrências (pessoa-dia): {total:,}".replace(",", "."), fontsize=16, va="top")
        if n_pessoas:
            ax.text(0.02, 0.80, f"Pessoas (cadastro): {n_pessoas:,}".replace(",", "."), fontsize=16, va="top")
        ax.text(0.02, 0.76, f"Dias com ocorrência: {n_dias:,}".replace(",", "."), fontsize=16, va="top")

        if pd.notna(dt_min) and pd.notna(dt_max):
            ax.text(0.02, 0.72, f"Período: {pd.to_datetime(dt_min).date()} → {pd.to_datetime(dt_max).date()}",
                    fontsize=14, va="top")

        y = 0.64
        if total and not top_pessoas.empty:
            ax.text(0.02, y, f"Concentração Top 1: {top1_n:,} ({top1_pct:.1%})".replace(",", "."),
                    fontsize=14, va="top")
            y -= 0.05
            ax.text(0.02, y, f"Concentração Top 3: {top3_n:,} ({top3_pct:.1%})".replace(",", "."),
                    fontsize=14, va="top")
            y -= 0.05
            ax.text(0.02, y, f"Concentração Top 10: {top10_n:,} ({top10_pct:.1%})".replace(",", "."),
                    fontsize=13, va="top")
            y -= 0.07

            ax.text(0.02, y, "Top 3 piores (quantidade e % do total):", fontsize=14, fontweight="bold", va="top")
            y -= 0.04
            for i, (k, v) in enumerate(top3_qty.items(), start=1):
                pct = (int(v) / total) if total else 0.0
                ax.text(0.03, y, f"{i}. {_wrap(k, 60)} — {int(v)} ({pct:.1%})", fontsize=12.5, va="top")
                y -= 0.04

            y -= 0.02
            if total_recent > 0 and not top3_recent.empty:
                ax.text(0.02, y, f"Top 3 atuais (últimos {window_days} dias) — base: {total_recent} ({total_recent/total:.1%} do total)",
                        fontsize=13, fontweight="bold", va="top")
                y -= 0.04
                for i, (k, v) in enumerate(top3_recent.items(), start=1):
                    pct_r = (int(v) / total_recent) if total_recent else 0.0
                    ax.text(0.03, y, f"{i}. {_wrap(k, 60)} — {int(v)} ({pct_r:.1%} da janela)",
                            fontsize=11.8, va="top")
                    y -= 0.035

        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- GERAIS: mensal + DOW
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [2.1, 1.3]})

        if mensal.empty:
            ax1.axis("off")
            ax1.text(0.02, 0.95, "Tendência mensal", fontsize=18, fontweight="bold", va="top")
            ax1.text(0.02, 0.80, "Sem datas válidas para agregação mensal.", fontsize=13, va="top")
        else:
            ax1.plot(mensal.index, mensal.values, marker="o", linewidth=2.3)
            ax1.fill_between(mensal.index, mensal.values, alpha=0.15)
            ax1.set_title("Tendência mensal de ocorrências (pessoa-dia)")
            ax1.set_xlabel("Mês")
            ax1.set_ylabel("Ocorrências")
            ax1.grid(alpha=0.30)
            ax1.tick_params(axis="x", rotation=35)

            # anota pico com %
            peak_v = int(mensal.max())
            if peak_v > 0 and total > 0:
                peak_i = mensal.idxmax()
                ax1.annotate(f"Pico: {peak_v} ({peak_v/total:.1%})",
                             xy=(peak_i, peak_v),
                             xytext=(0, 10),
                             textcoords="offset points")

        ax2.bar(dow.index, dow.values)
        ax2.set_title("Ocorrências por dia da semana (pessoa-dia)")
        ax2.set_xlabel("Dia")
        ax2.set_ylabel("Ocorrências")
        ax2.grid(axis="y", alpha=0.30)
        _annotate_bar(ax2, dow.values, total_base=total, fs=9)

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- GERAIS: Áreas / Plantas / (Área | Planta)
        fig, axs = plt.subplots(3, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [1.2, 1.2, 1.6]})
        axa, axp, axc = axs

        # Áreas
        if top_areas.empty:
            axa.axis("off")
            axa.text(0.02, 0.92, "Top áreas (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axa.text(0.02, 0.74, "Sem coluna 'área'/'area' ou sem valores.", fontsize=12, va="top")
        else:
            s = top_areas.head(12).sort_values()
            axa.barh([_wrap(i, 44) for i in s.index], s.values)
            axa.set_title("Top áreas (por dias com ocorrência)")
            axa.set_xlabel("Ocorrências (pessoa-dia)")
            axa.grid(axis="x", alpha=0.25)
            _annotate_barh(axa, s.values, total_base=total, fs=9)

        # Plantas
        if top_plantas.empty:
            axp.axis("off")
            axp.text(0.02, 0.92, "Top plantas (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axp.text(0.02, 0.74, "Sem coluna 'planta' ou sem valores.", fontsize=12, va="top")
        else:
            s = top_plantas.head(12).sort_values()
            axp.barh([_wrap(i, 44) for i in s.index], s.values)
            axp.set_title("Top plantas (por dias com ocorrência)")
            axp.set_xlabel("Ocorrências (pessoa-dia)")
            axp.grid(axis="x", alpha=0.25)
            _annotate_barh(axp, s.values, total_base=total, fs=9)

        # Área | Planta
        if top_combo.empty:
            axc.axis("off")
            axc.text(0.02, 0.92, "Top (Área | Planta) (por dias com ocorrência)", fontsize=15, fontweight="bold", va="top")
            axc.text(0.02, 0.74, "Sem dados suficientes para combinar área e planta.", fontsize=12, va="top")
        else:
            s = top_combo.head(15).sort_values()
            axc.barh([_wrap(i, 56) for i in s.index], s.values)
            axc.set_title("Top (Área | Planta) (por dias com ocorrência)")
            axc.set_xlabel("Ocorrências (pessoa-dia)")
            axc.grid(axis="x", alpha=0.25)
            _annotate_barh(axc, s.values, total_base=total, fs=8)

        fig.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # ---------------- TOP 3: Quantidade / Percentual / Atualidade
        if not top_pessoas.empty:
            fig, axs = plt.subplots(3, 1, figsize=(11.69, 8.27), gridspec_kw={"height_ratios": [1.25, 1.1, 1.25]})
            axq, axpct, axr = axs

            # Quantidade
            s = top3_qty.iloc[::-1]  # invert para ficar maior no topo no barh
            axq.barh([_wrap(x, 60) for x in s.index], s.values)
            axq.set_title("Top 3 piores — por quantidade (pessoa-dia)")
            axq.set_xlabel("Ocorrências")
            axq.grid(axis="x", alpha=0.25)
            _annotate_barh(axq, s.values, total_base=total, fs=10)

            # Percentual (mesmos 3, mas eixo em %)
            sp = top3_pct_ser.loc[top3_qty.index].iloc[::-1]
            axpct.barh([_wrap(x, 60) for x in sp.index], sp.values)
            axpct.set_title("Top 3 piores — por percentual do total")
            axpct.set_xlabel("% do total")
            axpct.grid(axis="x", alpha=0.25)
            for i, v in enumerate(sp.values):
                axpct.text(v, i, f"  {v:.1%}", va="center", fontsize=10)

            # Atualidade
            if total_recent <= 0 or top3_recent.empty:
                axr.axis("off")
                axr.text(0.02, 0.92, f"Top 3 atuais (últimos {window_days} dias)", fontsize=15, fontweight="bold", va="top")
                axr.text(0.02, 0.74, "Sem dados suficientes na janela recente.", fontsize=12, va="top")
            else:
                sr = top3_recent.iloc[::-1]
                axr.barh([_wrap(x, 60) for x in sr.index], sr.values)
                axr.set_title(f"Top 3 atuais — últimos {window_days} dias")
                axr.set_xlabel("Ocorrências na janela")
                axr.grid(axis="x", alpha=0.25)
                # anota % na janela e % no total
                for i, v in enumerate(sr.values):
                    pr = (int(v) / total_recent) if total_recent else 0.0
                    # % do total geral também (para visão analítica)
                    # total por pessoa no geral (se existir no top_pessoas)
                    k = sr.index[i]
                    vg = int(top_pessoas.get(k, 0))
                    p_total = (vg / total) if total else 0.0
                    axr.text(v, i, f"  {int(v)} ({pr:.1%} da janela) | total: {vg} ({p_total:.1%})",
                             va="center", fontsize=9)

            fig.tight_layout()
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

        # ---------------- TOP 3: 4 gráficos por pessoa (mensal / DOW / diário / vazio)
        if top3_cads and col_cad and (col_cad in df_occ.columns):
            # range mensal completo
            mes_min = pd.to_datetime(dt_min).to_period("M").to_timestamp()
            mes_max = pd.to_datetime(dt_max).to_period("M").to_timestamp()
            idx_all = pd.date_range(mes_min, mes_max, freq="MS")

            for cad in top3_cads[:3]:
                occ_p = df_occ.loc[df_occ[col_cad].astype(str).eq(str(cad)), ["_dia_operacional"]]
                if occ_p.empty:
                    continue

                total_p = int(len(occ_p))
                pct_p = (total_p / total) if total else 0.0
                nome = nome_map.get(str(cad), "") if nome_map else ""
                label = f"{cad} - {nome}".strip(" -")

                # mensal
                ser_m = (
                    occ_p.assign(_mes=occ_p["_dia_operacional"].dt.to_period("M").dt.to_timestamp())
                         .groupby("_mes", sort=False).size()
                         .reindex(idx_all, fill_value=0)
                )

                # DOW
                dow_p = (
                    occ_p["_dia_operacional"].dt.dayofweek.map(dow_map)
                        .value_counts()
                        .reindex(["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"])
                        .fillna(0).astype(int)
                )

                # diário (últimos 60 dias para legibilidade)
                occ_p_day = (
                    occ_p.groupby(occ_p["_dia_operacional"].dt.normalize(), sort=False).size().rename("n").sort_index()
                )
                dt_cut_p = pd.to_datetime(dt_max).normalize() - pd.Timedelta(days=60)
                occ_p_day_60 = occ_p_day.loc[occ_p_day.index >= dt_cut_p]
                total_p_60 = int(occ_p_day_60.sum()) if not occ_p_day_60.empty else 0

                fig, axs = plt.subplots(2, 2, figsize=(11.69, 8.27))
                (ax1, ax2), (ax3, ax4) = axs

                fig.suptitle(
                    f"Top 3 — {_wrap(label, 70)}  |  Total: {total_p} ({pct_p:.1%} do total)",
                    fontsize=16,
                    fontweight="bold",
                )

                # (1) mensal
                ax1.plot(ser_m.index, ser_m.values, marker="o", linewidth=2.0)
                ax1.fill_between(ser_m.index, ser_m.values, alpha=0.12)
                ax1.set_title("Evolução mensal (pessoa-dia)")
                ax1.set_xlabel("Mês")
                ax1.set_ylabel("Ocorrências")
                ax1.grid(alpha=0.25)
                ax1.tick_params(axis="x", rotation=35)
                if len(ser_m) > 0:
                    peak_v = int(ser_m.max())
                    if peak_v > 0 and total_p > 0:
                        peak_i = ser_m.idxmax()
                        ax1.annotate(f"Pico: {peak_v} ({peak_v/total_p:.1%})",
                                     xy=(peak_i, peak_v),
                                     xytext=(0, 10),
                                     textcoords="offset points")

                # (2) DOW
                ax2.bar(dow_p.index, dow_p.values)
                ax2.set_title("Ocorrências por dia da semana")
                ax2.set_xlabel("Dia")
                ax2.set_ylabel("Ocorrências")
                ax2.grid(axis="y", alpha=0.25)
                _annotate_bar(ax2, dow_p.values, total_base=total_p, fs=8)

                # (3) diário
                if occ_p_day_60.empty:
                    ax3.axis("off")
                    ax3.text(0.02, 0.92, "Ocorrências diárias (últimos 60 dias)", fontsize=14, fontweight="bold", va="top")
                    ax3.text(0.02, 0.78, "Sem ocorrências na janela recente.", fontsize=11, va="top")
                else:
                    ax3.plot(occ_p_day_60.index, occ_p_day_60.values, marker="o", linewidth=1.8)
                    ax3.fill_between(occ_p_day_60.index, occ_p_day_60.values, alpha=0.12)
                    ax3.set_title("Ocorrências diárias (últimos 60 dias)")
                    ax3.set_xlabel("Dia")
                    ax3.set_ylabel("Ocorrências")
                    ax3.grid(alpha=0.25)
                    ax3.tick_params(axis="x", rotation=35)
                    # anota máximo na janela
                    mx = int(occ_p_day_60.max())
                    if mx > 0 and total_p_60 > 0:
                        mi = occ_p_day_60.idxmax()
                        ax3.annotate(f"Máx: {mx} ({mx/total_p_60:.1%})",
                                     xy=(mi, mx),
                                     xytext=(0, 10),
                                     textcoords="offset points")

                # (4) vazio*
                ax4.axis("off")
                ax4.text(0.02, 0.92, "Reservado", fontsize=14, fontweight="bold", va="top")
                ax4.text(0.02, 0.78, "—", fontsize=12, va="top")

                fig.tight_layout(rect=[0, 0, 1, 0.94])
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)

        # ---------------- Caso especial: 999 trajeto (3 bolinhas por pessoa)
        # Critério de ativação: "999" e "trajeto" no título ou nome do arquivo
        titulo_l = str(titulo).lower()
        if ("999" in titulo_l) and ("trajeto" in titulo_l or "999" in str(pdf_path).lower()):
            # precisa de colunas básicas para reconstruir dt e identificar portaria/999
            if all(c in df_irregularidade.columns for c in ["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]):
                dfx = df_irregularidade[["cadastro", "data", "hora", "dir", "planta", "descricao coletor"]].copy()
                dfx["cadastro"] = dfx["cadastro"].astype(str).str.strip()

                # dt
                data_dt = pd.to_datetime(dfx["data"], dayfirst=True, errors="coerce")
                hora_s = dfx["hora"].fillna("00:00").astype(str).str.strip()
                hora_s = hora_s.replace({"": "00:00", "nan": "00:00", "NaT": "00:00"})
                hhmm_mask = hora_s.str.match(r"^\d{1,2}:\d{2}$", na=False)
                hora_s = hora_s.where(~hhmm_mask, hora_s + ":00")
                hora_td = pd.to_timedelta(hora_s, errors="coerce")
                dfx["_dt"] = data_dt.dt.normalize() + hora_td
                dfx["_dia_key"] = data_dt.dt.normalize()

                # portaria (usa regex_portaria global se existir; senão fallback)
                _re_port = globals().get("regex_portaria", r"PORTARIA|PORT")
                desc = dfx["descricao coletor"].fillna("").astype(str)
                dfx["_is_port"] = desc.str.contains(_re_port, case=False, na=False, regex=True)

                # dir norm
                dirn = dfx["dir"].fillna("").astype(str).str.strip().str.upper()
                dirn = dirn.replace({"ENTRADA": "E", "SAIDA": "S", "SAÍDA": "S"})
                dfx["_is_E"] = dirn.str.startswith("E")
                dfx["_is_S"] = dirn.str.startswith("S")

                # 999
                planta_num = pd.to_numeric(dfx["planta"], errors="coerce")
                dfx["_is_999"] = planta_num.eq(999)

                # portaria por pessoa-dia (dia_key)
                port = dfx.loc[dfx["_is_port"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt", "_is_E", "_is_S"]]
                if not port.empty:
                    ent = (
                        port.loc[port["_is_E"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].min()
                            .rename("t_ent")
                            .reset_index()
                    )
                    sai = (
                        port.loc[port["_is_S"]]
                            .groupby(["cadastro", "_dia_key"], sort=False)["_dt"].max()
                            .rename("t_sai")
                            .reset_index()
                    )
                    pe = ent.merge(sai, on=["cadastro", "_dia_key"], how="outer")

                    # 999 eventos por pessoa-dia
                    ev = dfx.loc[dfx["_is_999"] & dfx["_dt"].notna(), ["cadastro", "_dia_key", "_dt"]]
                    if (not ev.empty) and (not pe.empty):
                        ev = ev.merge(pe, on=["cadastro", "_dia_key"], how="left")
                        # classifica por evento
                        ev["_before"] = ev["t_ent"].notna() & (ev["_dt"] < ev["t_ent"])
                        ev["_after"]  = ev["t_sai"].notna() & (ev["_dt"] > ev["t_sai"])

                        day_flags = (
                            ev.groupby(["cadastro", "_dia_key"], sort=False)[["_before", "_after"]]
                              .any()
                              .reset_index()
                        )
                        day_flags["cat"] = np.select(
                            [
                                day_flags["_before"] & day_flags["_after"],
                                day_flags["_before"] & (~day_flags["_after"]),
                                (~day_flags["_before"]) & day_flags["_after"],
                            ],
                            ["Ambos", "Antes", "Depois"],
                            default="(sem class.)",
                        )

                        # agrega por pessoa
                        # considera apenas categorias esperadas
                        day_flags = day_flags[day_flags["cat"].isin(["Antes", "Depois", "Ambos"])]
                        if not day_flags.empty:
                            agg = (
                                day_flags.groupby(["cadastro", "cat"], sort=False)
                                         .size()
                                         .rename("n_dias")
                                         .reset_index()
                            )
                            tot = agg.groupby("cadastro", sort=False)["n_dias"].sum().rename("n_total").reset_index()
                            agg = agg.merge(tot, on="cadastro", how="left")
                            agg["pct"] = agg["n_dias"] / agg["n_total"].where(agg["n_total"] > 0, np.nan)

                            # top pessoas por total de dias com 999 classificado (até 10)
                            top_cads = (
                                tot.sort_values("n_total", ascending=False)
                                   .head(10)["cadastro"].astype(str).tolist()
                            )

                            # prepara matriz (cadastro x cat)
                            cats = ["Antes", "Depois", "Ambos"]
                            mat = (
                                agg.loc[agg["cadastro"].isin(top_cads)]
                                   .pivot_table(index="cadastro", columns="cat", values="pct", aggfunc="first")
                                   .reindex(columns=cats)
                                   .fillna(0.0)
                            )
                            # labels com nome
                            if nome_map:
                                y_labels = [f"{c} - {nome_map.get(str(c),'')}".strip(" -") for c in mat.index]
                            else:
                                y_labels = list(mat.index)

                            # plot "3 bolinhas"
                            fig = plt.figure(figsize=(11.69, 8.27))
                            ax = fig.add_subplot(111)
                            ax.set_title("999 como trajeto — concentração por período (Top 10 pessoas)")
                            ax.set_xlabel("Período do dia relativo à Portaria")
                            ax.set_yticks(range(len(y_labels)))
                            ax.set_yticklabels([_wrap(y, 60) for y in y_labels])
                            ax.set_xticks([0, 1, 2])
                            ax.set_xticklabels(["Antes", "Depois", "Ambos"])
                            ax.set_xlim(-0.5, 2.5)
                            ax.grid(axis="x", alpha=0.25)

                            # tamanho proporcional ao pct (mínimo visível)
                            for yi, cad in enumerate(mat.index):
                                for xi, cat in enumerate(cats):
                                    p = float(mat.loc[cad, cat])
                                    s = 40 + 900 * p  # escala
                                    ax.scatter([xi], [yi], s=s)
                                    # anota p% (somente se relevante)
                                    if p >= 0.15:
                                        ax.text(xi + 0.06, yi, f"{p:.0%}", va="center", fontsize=9)

                            pdf.savefig(fig, bbox_inches="tight")
                            plt.close(fig)
            # se faltar coluna, simplesmente não cria página especial

def row_ids_from_keys(df_min: pd.DataFrame, keys: pd.DataFrame) -> np.ndarray:
    if keys is None or keys.empty:
        return np.array([], dtype=np.int64)

    keys2 = keys.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates().copy()
    keys2["_dia_operacional"] = pd.to_datetime(keys2["_dia_operacional"], errors="coerce")

    if ("_ROWID_CACHE_HASH" not in globals()) or (len(globals()["_ROWID_CACHE_HASH"]) != len(df_min)):
        globals()["_ROWID_CACHE_ROWID"] = df_min["_row_id"].to_numpy(dtype=np.int64, copy=False)
        globals()["_ROWID_CACHE_HASH"] = pd.util.hash_pandas_object(
            df_min[["cadastro", "_dia_operacional"]], index=False
        ).to_numpy(dtype=np.uint64, copy=False)
        globals()["_ROWID_CACHE_HASH_INDEX"] = pd.Index(globals()["_ROWID_CACHE_HASH"])

    key_hash = pd.util.hash_pandas_object(
        keys2[["cadastro", "_dia_operacional"]], index=False
    ).to_numpy(dtype=np.uint64, copy=False)

    if key_hash.size == 0:
        return np.array([], dtype=np.int64)

    mask = globals()["_ROWID_CACHE_HASH_INDEX"].isin(key_hash)
    return globals()["_ROWID_CACHE_ROWID"][mask].astype(np.int64, copy=True)


n_total = len(df_min_raw)
dia_op_arr = np.full(n_total, np.datetime64("NaT"), dtype="datetime64[ns]")
valid_map = df.loc[df["_dt_ronda"].notna(), ["_row_id", "_dia_operacional"]]
dia_op_arr[valid_map["_row_id"].to_numpy()] = valid_map["_dia_operacional"].to_numpy()


def materializar_e_exportar_por_rowid(row_ids: np.ndarray, irregularidade_col: str, titulo: str) -> pd.DataFrame:
    slug = slugify(irregularidade_col)

    parquet_path = DFS_DIR / f"{slug}.parquet"
    pdf_path = PDF_DIR / f"{slug}.pdf"

    params_export = {
        "param_janela_portaria_min": globals().get("janela_portaria_min"),
        "param_tolerancia_entrada_min": globals().get("tolerancia_entrada_min"),
        "param_tolerancia_saida_min": globals().get("tolerancia_saida_min"),
        "param_janela_999_horas": globals().get("janela_999_horas"),
        "param_hora_corte_operacional": globals().get("hora_corte_operacional"),
        "param_descanso_min_horas": globals().get("descanso_min_horas"),
        "param_jornada_min_horas": globals().get("jornada_min_horas"),
        "param_jornada_min_minutos": globals().get("jornada_min_minutos"),
        "param_usar_jornada_por_portaria": globals().get("usar_jornada_por_portaria"),
        "param_max_jornada_portaria_horas": globals().get("max_jornada_portaria_horas"),
    }

    if row_ids.size == 0:
        df_out = df_min_raw.head(0).copy()
        df_out["dia_operacional"] = pd.to_datetime([], errors="coerce")
        df_out[irregularidade_col] = False
        for _k, _v in params_export.items():
            df_out[_k] = _v
        df_out.to_parquet(parquet_path, index=False)
        gerar_relatorio_pdf(df_out, titulo=titulo, pdf_path=pdf_path)
        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_out

    row_ids = row_ids[(row_ids >= 0) & (row_ids < len(df_min_raw))]

    MAX_ROWS_IN_MEMORY = 2_000_000
    REPORT_SAMPLE_ROWS = 2_000

    if row_ids.size > MAX_ROWS_IN_MEMORY:
        import pyarrow as pa
        import pyarrow.parquet as pq

        row_ids = np.sort(row_ids.astype(np.int64, copy=False))
        chunk_size = 250_000

        if parquet_path.exists():
            parquet_path.unlink()

        writer = None
        report_parts = []
        report_count = 0

        for i in range(0, row_ids.size, chunk_size):
            ids = row_ids[i:i + chunk_size]
            part = df_min_raw.iloc[ids].copy()
            part["dia_operacional"] = pd.to_datetime(dia_op_arr[ids], errors="coerce")
            part[irregularidade_col] = True
            for _k, _v in params_export.items():
                part[_k] = _v
            part.drop(columns=["_row_id"], inplace=True, errors="ignore")

            table = pa.Table.from_pandas(part, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(str(parquet_path), table.schema, compression="snappy")
            writer.write_table(table)

            if report_count < REPORT_SAMPLE_ROWS:
                take = min(REPORT_SAMPLE_ROWS - report_count, len(part))
                report_parts.append(part.head(take).copy())
                report_count += take

            del ids, part, table
            gc.collect()

        if writer is not None:
            writer.close()

        if report_parts:
            df_report = pd.concat(report_parts, ignore_index=True)
        else:
            df_report = df_min_raw.head(0).copy()
            df_report["dia_operacional"] = pd.to_datetime([], errors="coerce")
            df_report[irregularidade_col] = False
            for _k, _v in params_export.items():
                df_report[_k] = _v

        gerar_relatorio_pdf(df_report, titulo=titulo, pdf_path=pdf_path)

        gc.collect()
        print("Parquet:", parquet_path)
        print("PDF   :", pdf_path)
        return df_report

    df_out = df_min_raw.iloc[row_ids].copy()
    df_out["dia_operacional"] = pd.to_datetime(dia_op_arr[row_ids], errors="coerce")
    df_out[irregularidade_col] = True
    for _k, _v in params_export.items():
        df_out[_k] = _v
    df_out.drop(columns=["_row_id"], inplace=True, errors="ignore")

    df_out.to_parquet(parquet_path, index=False)
    gerar_relatorio_pdf(df_out, titulo=titulo, pdf_path=pdf_path)

    gc.collect()
    print("Parquet:", parquet_path)
    print("PDF   :", pdf_path)
    return df_out


# =========================
# <<< ALTERADO: Semanas de feriado prolongado (2024, 2025, 2026) com feriado na TERÇA ou QUINTA
# =========================
from dateutil.easter import easter

FERIADOS_INFO = "Esses feriados são os de 2024, 2025 e 2026."
ANOS_FERIADOS = (2024, 2025, 2026)

def feriados_br_df(anos=ANOS_FERIADOS) -> pd.DataFrame:
    rows = []

    fixos = [
        (1, 1,  "Confraternização Universal"),
        (4, 21, "Tiradentes"),
        (5, 1,  "Dia do Trabalho"),
        (9, 7,  "Independência do Brasil"),
        (10, 12,"Nossa Senhora Aparecida"),
        (11, 2, "Finados"),
        (11, 15,"Proclamação da República"),
        (12, 25,"Natal"),
    ]

    for ano in anos:
        for m, d, nome in fixos:
            rows.append((pd.Timestamp(year=ano, month=m, day=d), nome))

        # Consciência Negra (feriado nacional a partir de 2024)
        if ano >= 2024:
            rows.append((pd.Timestamp(year=ano, month=11, day=20), "Consciência Negra"))

        pascoa = pd.Timestamp(easter(ano))
        rows.append((pascoa - pd.Timedelta(days=2), "Sexta-feira Santa"))
        rows.append((pascoa + pd.Timedelta(days=60), "Corpus Christi"))

        # Carnaval (terça) — comum em calendário corporativo; mantém terça conforme solicitado
        rows.append((pascoa - pd.Timedelta(days=47), "Carnaval (terça)"))

    fer = pd.DataFrame(rows, columns=["data", "nome"])
    fer["data"] = pd.to_datetime(fer["data"], errors="coerce").dt.normalize()
    fer = fer.dropna(subset=["data"]).drop_duplicates(subset=["data"]).sort_values("data").reset_index(drop=True)
    return fer

feriados = feriados_br_df()

# filtra feriados que caem na terça (1) ou quinta (3)
fer_ter_qui = feriados.loc[feriados["data"].dt.dayofweek.isin([1, 3])].copy()

# semana considerada: segunda..sexta (para a regra de 3 dias presenciais)
fer_ter_qui["semana_ini"] = (fer_ter_qui["data"] - pd.to_timedelta(fer_ter_qui["data"].dt.dayofweek, unit="D")).dt.normalize()
fer_ter_qui["semana_fim"] = fer_ter_qui["semana_ini"] + pd.Timedelta(days=4)

semanas_feriado_prolongado = (
    fer_ter_qui[["semana_ini", "semana_fim"]]
    .drop_duplicates()
    .sort_values("semana_ini")
    .itertuples(index=False, name=None)
)
semanas_feriado_prolongado = list(semanas_feriado_prolongado)

# Reusa os helpers modernos já definidos anteriormente no notebook.
if "_gerar_relatorio_pdf_moderno" in globals():
    gerar_relatorio_pdf = _gerar_relatorio_pdf_moderno
if "_row_ids_from_keys_moderno" in globals():
    row_ids_from_keys = _row_ids_from_keys_moderno
if "_materializar_e_exportar_por_rowid_moderno" in globals():
    materializar_e_exportar_por_rowid = _materializar_e_exportar_por_rowid_moderno


In [ ]:
# =========================
# IRREGULARIDADE: Semana Feriado Prolongado (menos de 3 dias presenciais na semana)
# =========================
irregularidade_col = "Irregularidade - Pessoas sem cumprir os 3 Dias Presenciais em Semana de Feriado Prolongado"
titulo = (
    "Irregularidade – Semana de Feriado Prolongado "
    "(menos de 3 dias presenciais na semana)\n"
    + FERIADOS_INFO
)
data_inicio_analise_feriado = pd.Timestamp("2025-05-01")

day_ = day.copy()
if "cadastro" in day_.columns:
    day_["cadastro"] = day_["cadastro"].astype(str).str.strip()

# Marca semana de feriado por data real do registro, sem copiar df inteiro
semana_ini = pd.Series(pd.NaT, index=df_valid.index)
dia_key_base = (
    pd.to_datetime(df_valid["_dia_key"], errors="coerce").dt.normalize()
    if "_dia_key" in df_valid.columns
    else pd.to_datetime(df_valid["_dt_ronda"], errors="coerce").dt.normalize()
)

for ini_w, fim_w in semanas_feriado_prolongado:
    m_w = dia_key_base.between(ini_w, fim_w) & (dia_key_base >= data_inicio_analise_feriado)
    semana_ini.loc[m_w] = ini_w

m_feriado = semana_ini.notna()
cols_base = ["cadastro", "_dia_operacional", "_dt_ronda", "Presencial", "Portaria", "_dir_norm"]
gw = df_valid.loc[m_feriado, cols_base].copy()
gw["_semana_feriado_ini"] = semana_ini.loc[m_feriado].to_numpy()
gw["_dia_key"] = dia_key_base.loc[m_feriado].to_numpy()

gw["cadastro"] = gw["cadastro"].astype(str).str.strip()

# Tenta trazer nome/área se existirem em df_valid; fallback por cadastro usando df.
if "nome" in df_valid.columns:
    gw["nome"] = df_valid.loc[m_feriado, "nome"].to_numpy()
else:
    gw["nome"] = None

if "área" in df_valid.columns:
    gw["area"] = df_valid.loc[m_feriado, "área"].to_numpy()
elif "area" in df_valid.columns:
    gw["area"] = df_valid.loc[m_feriado, "area"].to_numpy()
else:
    gw["area"] = None

if ("nome" in df.columns) and (gw["nome"].isna().all() or gw["nome"].astype(str).str.strip().eq("").all()):
    tmp_nome_src = df.loc[df["nome"].notna(), ["cadastro", "nome"]].copy()
    if not tmp_nome_src.empty:
        tmp_nome_src["cadastro"] = tmp_nome_src["cadastro"].astype(str).str.strip()
        tmp_nome_src["nome"] = tmp_nome_src["nome"].astype(str).str.strip()
        tmp_nome_src = tmp_nome_src[(tmp_nome_src["cadastro"].ne("")) & (tmp_nome_src["nome"].ne(""))]
        if not tmp_nome_src.empty:
            cnt_nome_src = (
                tmp_nome_src.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
            )
            nome_map_src = cnt_nome_src.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()
            gw["nome"] = gw["cadastro"].map(nome_map_src)

col_area_src = "área" if "área" in df.columns else ("area" if "area" in df.columns else None)
if (col_area_src is not None) and (gw["area"].isna().all() or gw["area"].astype(str).str.strip().eq("").all()):
    tmp_area_src = df.loc[df[col_area_src].notna(), ["cadastro", col_area_src]].copy()
    tmp_area_src = tmp_area_src.rename(columns={col_area_src: "area"})
    if not tmp_area_src.empty:
        tmp_area_src["cadastro"] = tmp_area_src["cadastro"].astype(str).str.strip()
        tmp_area_src["area"] = tmp_area_src["area"].astype(str).str.strip()
        tmp_area_src = tmp_area_src[(tmp_area_src["cadastro"].ne("")) & (tmp_area_src["area"].ne(""))]
        if not tmp_area_src.empty:
            cnt_area_src = (
                tmp_area_src.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
            )
            area_map_src = cnt_area_src.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()
            gw["area"] = gw["cadastro"].map(area_map_src)

base_colaboradores_area = {}
if not gw.empty and "area" in gw.columns:
    tmp_base_area = gw.loc[gw["area"].notna(), ["cadastro", "area"]].copy()
    if not tmp_base_area.empty:
        tmp_base_area["cadastro"] = tmp_base_area["cadastro"].astype(str).str.strip()
        tmp_base_area["area"] = tmp_base_area["area"].astype(str).str.strip()
        tmp_base_area = tmp_base_area[(tmp_base_area["cadastro"].ne("")) & (tmp_base_area["area"].ne(""))]
        if not tmp_base_area.empty:
            cnt_base_area = (
                tmp_base_area.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
            )
            area_base_por_cadastro = cnt_base_area.drop_duplicates(subset=["cadastro"], keep="first")
            base_colaboradores_area = area_base_por_cadastro["area"].value_counts().to_dict()

if gw.empty:
    day_[irregularidade_col] = False
    df_relatorio_feriado = pd.DataFrame(columns=[
        "cadastro", "nome", "area", "semana_inicio", "semana_fim",
        "dias_com_registro", "dias_presenciais", "horas_presenciais", "nao_cumpriu_3", "total_colaboradores_area"
    ])
else:
    # Portaria E->S por dia (dentro da semana do feriado)
    port = gw.loc[gw["Portaria"].astype(bool) & gw["_dt_ronda"].notna()].copy()

    # quantidade de dias com qualquer registro na semana
    dias_com_registro = (
        gw.groupby(["cadastro", "_semana_feriado_ini"], sort=False)["_dia_operacional"]
          .nunique()
          .rename("dias_com_registro")
          .reset_index()
    )

    # nome/área mais representativos por cadastro
    nome_map = {}
    area_map = {}
    cads_semana = gw["cadastro"].dropna().astype(str).str.strip().unique()

    if "nome" in gw.columns:
        tmp_nome = gw.loc[gw["nome"].notna(), ["cadastro", "nome"]].copy()
        if not tmp_nome.empty:
            tmp_nome["nome"] = tmp_nome["nome"].astype(str).str.strip()
            tmp_nome = tmp_nome[tmp_nome["nome"].ne("")]
            if not tmp_nome.empty:
                cnt_nome = (
                    tmp_nome.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                nome_map = cnt_nome.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()

    if "area" in gw.columns:
        tmp_area = gw.loc[gw["area"].notna(), ["cadastro", "area"]].copy()
        if not tmp_area.empty:
            tmp_area["area"] = tmp_area["area"].astype(str).str.strip()
            tmp_area = tmp_area[tmp_area["area"].ne("")]
            if not tmp_area.empty:
                cnt_area = (
                    tmp_area.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                area_map = cnt_area.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()

    # Critério principal da irregularidade: quantidade de dias presenciais na semana.
    dias_presenciais_semana = (
        gw.loc[
            gw["Presencial"].fillna(False).astype(bool),
            ["cadastro", "_semana_feriado_ini", "_dia_operacional"]
        ]
        .dropna(subset=["_dia_operacional"])
        .drop_duplicates()
        .groupby(["cadastro", "_semana_feriado_ini"], sort=False)
        .size()
        .rename("dias_presenciais")
        .reset_index()
    )

    wk = dias_com_registro.merge(
        dias_presenciais_semana,
        on=["cadastro", "_semana_feriado_ini"],
        how="left",
    )
    wk["dias_presenciais"] = pd.to_numeric(wk["dias_presenciais"], errors="coerce").fillna(0).astype(int)

    # Horas presenciais ficam como métrica auxiliar (não entram no critério principal).
    if port.empty:
        wk["horas_presenciais"] = 0.0
    else:
        first_E = (
            port.loc[port["_dir_norm"].astype(str).eq("E")]
                .groupby(["cadastro", "_semana_feriado_ini", "_dia_key"], sort=False)["_dt_ronda"]
                .min()
                .rename("first_port_E")
                .reset_index()
        )
        last_S = (
            port.loc[port["_dir_norm"].astype(str).eq("S")]
                .groupby(["cadastro", "_semana_feriado_ini", "_dia_key"], sort=False)["_dt_ronda"]
                .max()
                .rename("last_port_S")
                .reset_index()
        )

        day_dur = first_E.merge(
            last_S,
            on=["cadastro", "_semana_feriado_ini", "_dia_key"],
            how="inner",
        )
        day_dur["dur"] = day_dur["last_port_S"] - day_dur["first_port_E"]
        day_dur["horas_dia"] = (day_dur["dur"] / pd.Timedelta(hours=1)).where(
            day_dur["dur"].notna() & (day_dur["dur"] > pd.Timedelta(0)),
            0.0,
        ).fillna(0.0)

        wk_horas = (
            day_dur.groupby(["cadastro", "_semana_feriado_ini"], sort=False)["horas_dia"]
                  .sum()
                  .rename("horas_presenciais")
                  .reset_index()
        )

        wk = wk.merge(wk_horas, on=["cadastro", "_semana_feriado_ini"], how="left")
        wk["horas_presenciais"] = pd.to_numeric(wk["horas_presenciais"], errors="coerce").fillna(0.0)

    wk["nao_cumpriu_3"] = wk["dias_presenciais"] < 3

    df_relatorio_feriado = (
        wk.loc[wk["nao_cumpriu_3"]].copy()
          .rename(columns={"_semana_feriado_ini": "semana_inicio"})
    )
    df_relatorio_feriado["semana_inicio"] = pd.to_datetime(df_relatorio_feriado["semana_inicio"], errors="coerce").dt.normalize()
    df_relatorio_feriado["semana_fim"] = df_relatorio_feriado["semana_inicio"] + pd.Timedelta(days=6)
    df_relatorio_feriado["cadastro"] = df_relatorio_feriado["cadastro"].astype(str).str.strip()
    df_relatorio_feriado["nome"] = df_relatorio_feriado["cadastro"].map(nome_map).fillna("Sem Nome")
    df_relatorio_feriado["area"] = df_relatorio_feriado["cadastro"].map(area_map).fillna("Sem Área")
    df_relatorio_feriado["total_colaboradores_area"] = (
        df_relatorio_feriado["area"].map(base_colaboradores_area).fillna(0).astype(int)
    )

    df_relatorio_feriado = df_relatorio_feriado.loc[:, [
        "cadastro", "nome", "area", "semana_inicio", "semana_fim",
        "dias_com_registro", "dias_presenciais", "horas_presenciais", "nao_cumpriu_3", "total_colaboradores_area"
    ]].copy()

    # marca os dias (dia_operacional) que caem nessas semanas irregulares
    wk_irregularidade = (
        df_relatorio_feriado.loc[:, ["cadastro", "semana_inicio"]]
                            .rename(columns={"semana_inicio": "_semana_feriado_ini"})
                            .drop_duplicates()
    )

    if wk_irregularidade.empty:
        day_[irregularidade_col] = False
    else:
        dias_flag = (
            gw.merge(wk_irregularidade, on=["cadastro", "_semana_feriado_ini"], how="inner")
              .loc[:, ["cadastro", "_dia_operacional"]]
              .drop_duplicates()
        )

        day_ = day_.merge(
            dias_flag.assign(_flag_feriado=True),
            on=["cadastro", "_dia_operacional"],
            how="left"
        )
        day_[irregularidade_col] = day_["_flag_feriado"].fillna(False).astype(bool)

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()

row_ids = row_ids_from_keys(df, keys)

df_feriado = materializar_e_exportar_por_rowid(
    row_ids,
    irregularidade_col=irregularidade_col,
    titulo=titulo,
)

# Enriquecimento final de nome/área usando a base materializada da própria irregularidade.
if isinstance(df_relatorio_feriado, pd.DataFrame) and (not df_relatorio_feriado.empty) and isinstance(df_feriado, pd.DataFrame) and (not df_feriado.empty):
    if "cadastro" in df_relatorio_feriado.columns:
        df_relatorio_feriado["cadastro"] = df_relatorio_feriado["cadastro"].astype(str).str.strip()

    if "nome" in df_feriado.columns:
        tmp_nome = df_feriado.loc[df_feriado["nome"].notna(), ["cadastro", "nome"]].copy()
        if not tmp_nome.empty:
            tmp_nome["cadastro"] = tmp_nome["cadastro"].astype(str).str.strip()
            tmp_nome["nome"] = tmp_nome["nome"].astype(str).str.strip()
            tmp_nome = tmp_nome[tmp_nome["nome"].ne("")]
            if not tmp_nome.empty:
                cnt_nome = (
                    tmp_nome.groupby(["cadastro", "nome"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                nome_map_final = cnt_nome.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["nome"].to_dict()
                if nome_map_final:
                    nome_atual = df_relatorio_feriado["nome"].fillna("Sem Nome").astype(str)
                    df_relatorio_feriado["nome"] = (
                        df_relatorio_feriado["cadastro"].map(nome_map_final)
                        .where(df_relatorio_feriado["cadastro"].map(nome_map_final).notna(), nome_atual)
                        .fillna("Sem Nome")
                    )

    col_area_mat = "área" if "área" in df_feriado.columns else ("area" if "area" in df_feriado.columns else None)
    if col_area_mat is not None:
        tmp_area = df_feriado.loc[df_feriado[col_area_mat].notna(), ["cadastro", col_area_mat]].copy()
        tmp_area = tmp_area.rename(columns={col_area_mat: "area"})
        if not tmp_area.empty:
            tmp_area["cadastro"] = tmp_area["cadastro"].astype(str).str.strip()
            tmp_area["area"] = tmp_area["area"].astype(str).str.strip()
            tmp_area = tmp_area[tmp_area["area"].ne("")]
            if not tmp_area.empty:
                cnt_area = (
                    tmp_area.groupby(["cadastro", "area"], sort=False)
                            .size()
                            .reset_index(name="n")
                            .sort_values(["cadastro", "n"], ascending=[True, False], kind="mergesort")
                )
                area_map_final = cnt_area.drop_duplicates(subset=["cadastro"], keep="first").set_index("cadastro")["area"].to_dict()
                if area_map_final:
                    area_atual = df_relatorio_feriado["area"].fillna("Sem Área").astype(str)
                    df_relatorio_feriado["area"] = (
                        df_relatorio_feriado["cadastro"].map(area_map_final)
                        .where(df_relatorio_feriado["cadastro"].map(area_map_final).notna(), area_atual)
                        .fillna("Sem Área")
                    )

# Mantém o recorte temporal a partir de maio/2025 no dataframe de relatório.
if isinstance(df_relatorio_feriado, pd.DataFrame) and (not df_relatorio_feriado.empty):
    if "semana_fim" in df_relatorio_feriado.columns:
        sf = pd.to_datetime(df_relatorio_feriado["semana_fim"], errors="coerce")
        df_relatorio_feriado = df_relatorio_feriado.loc[sf >= data_inicio_analise_feriado].copy()
    if "total_colaboradores_area" not in df_relatorio_feriado.columns:
        df_relatorio_feriado["total_colaboradores_area"] = 0
    df_relatorio_feriado["total_colaboradores_area"] = (
        df_relatorio_feriado["area"].map(base_colaboradores_area).fillna(0).astype(int)
    )

# sobrescreve o PDF com o relatório semanal
pdf_path = PDF_DIR / f"{slugify(irregularidade_col)}.pdf"
gerar_relatorio_pdf(
    df_relatorio_feriado,
    titulo=titulo,
    pdf_path=pdf_path,
)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_feriado_prolongado = df_feriado

df_feriado.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Não entrou no refeitório, mas passou no grill
# Regra:
# - houve GRILL no dia
# - NÃO houve REFEITÓRIO NORMAL no mesmo dia
# - não acusar em borda de contexto
# =========================
irregularidade_col = "Irregularidade - Não entrou no refeitório, mas passou no grill"
titulo = "Irregularidade – Não entrou no refeitório, mas passou no grill"

# Garante o helper moderno de visualização neste bloco.
if "_gerar_relatorio_pdf_moderno" in globals():
    gerar_relatorio_pdf = _gerar_relatorio_pdf_moderno
if "_row_ids_from_keys_moderno" in globals():
    row_ids_from_keys = _row_ids_from_keys_moderno
if "_materializar_e_exportar_por_rowid_moderno" in globals():
    materializar_e_exportar_por_rowid = _materializar_e_exportar_por_rowid_moderno

day_ = day.copy()

# Usa o pré-cálculo já feito no day (evita normalização pesada da base completa).
ref_col = None
if "HAS_REFEITORIO_NORMAL_NO_DIA" in day_.columns:
    ref_col = "HAS_REFEITORIO_NORMAL_NO_DIA"
elif "HAS_REFEITORIO_ANY" in day_.columns:
    ref_col = "HAS_REFEITORIO_ANY"
elif "HAS_REFEITORIO_BEFORE_GRILL" in day_.columns:
    ref_col = "HAS_REFEITORIO_BEFORE_GRILL"

if ref_col and ("HAS_GRILL_ANY" in day_.columns):
    day_[irregularidade_col] = (
        (~day_["SEM_COBERTURA_CONTEXTO"].fillna(False).astype(bool))
        & day_["HAS_GRILL_ANY"].fillna(False).astype(bool)
        & (~day_[ref_col].fillna(False).astype(bool))
    )
else:
    # Fallback mínimo caso o day não tenha as flags calculadas.
    day_[irregularidade_col] = False

keys = day_.loc[day_[irregularidade_col], ["cadastro", "_dia_operacional"]].copy()
row_ids = row_ids_from_keys(df, keys)

df_ref_grill = materializar_e_exportar_por_rowid(
    row_ids,
    irregularidade_col=irregularidade_col,
    titulo=titulo,
)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_ref_grill = df_ref_grill

df_ref_grill.head(3)


In [ ]:
# =========================
# IRREGULARIDADE: Ausência das horas Presenciais Mínimas (SEMANAL)
# Regra: horas presenciais semanais = soma(Portaria S - Portaria E) por dia.
# Não contar executivos. Não acusar em borda de contexto.
#
# NECESSÁRIO: definir min_horas_semana (default 24.0)
# =========================
irregularidade_col = "Irregularidade - Ausência das horas Presenciais Mínimas"
titulo = "Irregularidade – Ausência de Horas Presenciais Mínimas (semanal; Portaria E→S)"

MIN_HORAS_SEMANA = float(globals().get("min_horas_semana", 24.0))

day_ = day.copy()

port = df_valid.loc[
    df_valid["Portaria"].eq(True),
    ["cadastro", "_dia_operacional", "_dt_ronda", "_dir_norm"]
].copy(deep=False)

first_E = (
    port.loc[port["_dir_norm"].eq("E")]
        .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
        .min()
        .rename("first_port_E")
        .reset_index()
)
last_S = (
    port.loc[port["_dir_norm"].eq("S")]
        .groupby(["cadastro", "_dia_operacional"], sort=False)["_dt_ronda"]
        .max()
        .rename("last_port_S")
        .reset_index()
)

day_ = day_.merge(first_E, on=["cadastro", "_dia_operacional"], how="left")
day_ = day_.merge(last_S,  on=["cadastro", "_dia_operacional"], how="left")

dur_port = day_["last_port_S"] - day_["first_port_E"]
day_["_horas_pres_dia"] = (dur_port / pd.Timedelta(hours=1)).where(
    dur_port.notna() & (dur_port >= pd.Timedelta(0)),
    0.0
).fillna(0.0)

dia = pd.to_datetime(day_["_dia_operacional"], errors="coerce")
day_["_sem_ini"] = (dia - pd.to_timedelta(dia.dt.dayofweek, unit="D")).dt.normalize()

base = day_.loc[
    (~day_["IS_EXECUTIVO_DAY"].astype(bool))
    & (~day_["SEM_COBERTURA_CONTEXTO"]),
    ["cadastro", "_dia_operacional", "_sem_ini", "_horas_pres_dia"]
].copy()

semanal = (
    base.groupby(["cadastro", "_sem_ini"], as_index=False)["_horas_pres_dia"]
        .sum()
        .rename(columns={"_horas_pres_dia": "horas_pres_semana"})
)

semanal["FLAG_MIN_HORAS"] = semanal["horas_pres_semana"] < MIN_HORAS_SEMANA

keys_sem = semanal.loc[semanal["FLAG_MIN_HORAS"], ["cadastro", "_sem_ini"]].drop_duplicates()

if keys_sem.empty:
    keys = day_.head(0)[["cadastro", "_dia_operacional"]].copy()
else:
    day_flag = day_.merge(keys_sem.assign(_flag=True), on=["cadastro", "_sem_ini"], how="inner")
    keys = day_flag.loc[:, ["cadastro", "_dia_operacional"]].drop_duplicates()

row_ids = row_ids_from_keys(df, keys)
df_min_horas = materializar_e_exportar_por_rowid(row_ids, irregularidade_col=irregularidade_col, titulo=titulo)

if MANTER_DF_EM_MEMORIA:
    df_irregularidade_horas_minimas = df_min_horas

df_min_horas.head(3)

## 8) Resumo de arquivos gerados


In [ ]:
# Lista PDFs e parquets gerados
pdfs = sorted(PDF_DIR.glob("*.pdf"))
pars = sorted(DFS_DIR.glob("*/*.parquet"))

print(f"PDFs gerados: {len(pdfs)}")
for p in pdfs:
    print(" -", p)

print(f"Parquets gerados: {len(pars)}")
for p in pars:
    print(" -", p)
